# DLM Extraction Pipeline

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import os, json, copy
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import types as T
from pyspark.sql.window import Window
from bisect import bisect_left as _bisect_left

RUN_DATE = datetime.today().date()

# ── Widget Parameters ────────────────────────────────────────────────────────
dbutils.widgets.text("ZONE", "B")
dbutils.widgets.text("PHASE", "2")
dbutils.widgets.text("CYCLE", "1")
dbutils.widgets.text("UPC", "N_07024_01001_GJ")

ZONE  = dbutils.widgets.get("ZONE")
PHASE = dbutils.widgets.get("PHASE")
CYCLE = dbutils.widgets.get("CYCLE")
UPC   = dbutils.widgets.get("UPC")

# ── Target Schemas (dual-write) ──────────────────────────────────────────────
CATALOG = "nhai_dhms"
DLM_POSTGRES_SCHEMA = "dlm_postgres"   # breach_flag=1 only
DLM_KPI_SCHEMA      = "dlm_kpi"        # all data (breach_flag=0 and 1)
DLM_POSTGRES_NS = f"{CATALOG}.{DLM_POSTGRES_SCHEMA}"
DLM_KPI_NS      = f"{CATALOG}.{DLM_KPI_SCHEMA}"

# ── Data Paths ───────────────────────────────────────────────────────────────
BRONZE_ROOT = "/Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output"
SCHEMA_WORKBOOK = "/Workspace/Users/vansh.choubey9201@softsensor.ai/(Clone) BRONZE_CODE_16_07_26/(Clone) Clone_NSV_Extraction_updated_bronze (1)/nsv_defect_schema_master 1.xlsx"
TRACKER_PATH = "/Workspace/Shared/NSV Extraction/NSV_Activity_Tracker_ZoneWise_Standardized_17072026.xlsx"

# ── Dynamic Path Resolution ──────────────────────────────────────────────────
def _find_latest_date_partition(base_path):
    if not os.path.exists(base_path):
        return None
    for y in sorted(os.listdir(base_path), reverse=True):
        if not y.startswith("year="): continue
        yp = os.path.join(base_path, y)
        for m in sorted(os.listdir(yp), reverse=True):
            if not m.startswith("month="): continue
            mp = os.path.join(yp, m)
            for d in sorted(os.listdir(mp), reverse=True):
                if not d.startswith("day="): continue
                return f"{y}/{m}/{d}"
    return None

def _find_deliverable_parent(base):
    if not os.path.exists(base): return base
    children = sorted(os.listdir(base))
    if any(c.startswith("Deliverable=") for c in children): return base
    for child in children:
        child_path = os.path.join(base, child)
        if os.path.isdir(child_path):
            found = _find_deliverable_parent(child_path)
            if found != child_path: return found
            if any(gc.startswith("Deliverable=") for gc in os.listdir(child_path)):
                return child_path
    return base

def _find_lane_parent(base):
    if not os.path.exists(base): return None
    children = sorted(os.listdir(base))
    if any(c.startswith("Lane=") for c in children): return base
    for child in children:
        child_path = os.path.join(base, child)
        if os.path.isdir(child_path):
            found = _find_lane_parent(child_path)
            if found: return found
    return None

# NSV deliverable path
_nsv_root = f"{BRONZE_ROOT}/Source=NSV/Zone={ZONE}/Phase={PHASE}/Cycle={CYCLE}/UPC={UPC}"
_nsv_date = _find_latest_date_partition(_nsv_root)
UPC_BASE = f"{_nsv_root}/{_nsv_date}" if _nsv_date else _nsv_root
DELIVERABLE_BASE = _find_deliverable_parent(UPC_BASE)
D1_BASE = f"{DELIVERABLE_BASE}/Deliverable=Deliverable - 1"
D2_BASE = f"{DELIVERABLE_BASE}/Deliverable=Deliverable - 2"
D3_BASE = f"{DELIVERABLE_BASE}/Deliverable=Deliverable - 3"
D4_BASE = f"{DELIVERABLE_BASE}/Deliverable=Deliverable - 4"

# Backup path
_backup_root = f"{BRONZE_ROOT}/Source=NSV_Backup/Zone={ZONE}/Phase={PHASE}/Cycle={CYCLE}/UPC={UPC}"
_backup_date = _find_latest_date_partition(_backup_root)
if _backup_date:
    _backup_date_path = f"{_backup_root}/{_backup_date}"
    BACKUP_BASE = _find_lane_parent(_backup_date_path) or _backup_date_path
else:
    BACKUP_BASE = _backup_root

# ── Clustering Constants ─────────────────────────────────────────────────────
PARENT_LENGTH_M = 1000
CHILD_100M = 100
CHILD_10M = 10
RAW_SEGMENT_M = 10
DEFECT_INTERVALS = {
    d: {'parent_m': PARENT_LENGTH_M, 'child_mid_m': CHILD_100M, 'child_10m_m': CHILD_10M}
    for d in ['roughness', 'rutting', 'cracking', 'ravelling', 'pothole']
}
LEVEL_STANDARDS = {
    'parent':     ['CA'],
    'child_100m': ['IRC', 'CA'],
    'child_10m':  ['CA', 'IRC'],
}

print(f"DLM Extraction Pipeline")
print(f"UPC: {UPC} | Zone={ZONE}, Phase={PHASE}, Cycle={CYCLE}")
print(f"Deliverable base: {DELIVERABLE_BASE}")
print(f"Backup base:      {BACKUP_BASE}")
print(f"Target schemas:   {DLM_POSTGRES_NS} (breach only), {DLM_KPI_NS} (all)")

DLM Extraction Pipeline
UPC: N_07024_01001_GJ | Zone=B, Phase=2, Cycle=1
Deliverable base: /Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output/Source=NSV/Zone=B/Phase=2/Cycle=1/UPC=N_07024_01001_GJ/year=2026/month=09/day=12
Backup base:      /Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output/Source=NSV_Backup/Zone=B/Phase=2/Cycle=1/UPC=N_07024_01001_GJ/year=2026/month=09/day=12/Pavement=Unknown/Variant=Default
Target schemas:   nhai_dhms.dlm_postgres (breach only), nhai_dhms.dlm_kpi (all)


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRACKER + D2 THRESHOLD PARSING
# ═══════════════════════════════════════════════════════════════════════════════

def _parse_std(val):
    s = str(val).strip().upper() if pd.notna(val) else ""
    if 'IRC' in s: return 'IRC'
    if 'CA' in s: return 'CA'
    return None

def _parse_interval(val):
    try: return int(float(val))
    except: return None

def load_tracker_context(zone, phase, cycle, upc_slash):
    sheet_map = {"A": "Zone A", "B": "Zone B", "C": "Zone C", "D": "Zone D", "E": "Zone E"}
    sheet_name = sheet_map.get(zone, f"Zone {zone}")

    def _default_ctx():
        defs = {d: {'standard': 'CA', 'interval_m': 100}
                for d in ['ravelling', 'rutting', 'cracking', 'roughness', 'pothole']}
        return {
            'survey_start_date': None, 'survey_end_date': None,
            'piu': None, 'ae_ie': None, 'contractor': None, 'state': None,
            'pavement_type': None, 'pci': None, 'threshold_label_override': None,
            'defect_standards': defs, 'raw_defect_standards': copy.deepcopy(defs),
        }

    try:
        df = pd.read_excel(TRACKER_PATH, sheet_name=sheet_name, header=1)
    except Exception as e:
        print(f"  WARNING: Could not load tracker: {e}")
        return _default_ctx()

    upc_norm = upc_slash.replace('/', '_').replace('_', '').lower()
    best_row = None
    for _, row in df.iterrows():
        upc_val = str(row.iloc[12]).replace('/', '_').replace('_', '').lower()
        if upc_norm not in upc_val and upc_val not in upc_norm: continue
        phase_val = str(row.iloc[2]).lower().replace('phase', '').strip()
        cycle_val = str(row.iloc[3]).lower().replace('cycle', '').strip()
        if phase_val == str(phase).strip() and cycle_val == str(cycle).strip():
            best_row = row; break
        if best_row is None: best_row = row

    if best_row is None:
        print(f"  WARNING: UPC {upc_slash} not found in tracker.")
        return _default_ctx()

    r = best_row
    survey_start = pd.to_datetime(r.iloc[21], errors='coerce')
    survey_end = pd.to_datetime(r.iloc[22], errors='coerce')
    defect_standards = {
        'ravelling': {'standard': _parse_std(r.iloc[46]), 'interval_m': _parse_interval(r.iloc[47]) or 100},
        'rutting':   {'standard': _parse_std(r.iloc[48]), 'interval_m': _parse_interval(r.iloc[49]) or 100},
        'cracking':  {'standard': _parse_std(r.iloc[50]), 'interval_m': _parse_interval(r.iloc[51]) or 100},
        'roughness': {'standard': _parse_std(r.iloc[52]), 'interval_m': _parse_interval(r.iloc[53]) or 100},
    }
    defect_standards['pothole'] = defect_standards['cracking'].copy()
    raw_defect_standards = copy.deepcopy(defect_standards)

    ctx = {
        'survey_start_date': survey_start.date() if pd.notnull(survey_start) else None,
        'survey_end_date': survey_end.date() if pd.notnull(survey_end) else None,
        'piu': str(r.iloc[7]).strip() if pd.notna(r.iloc[7]) else None,
        'ae_ie': str(r.iloc[9]).strip() if pd.notna(r.iloc[9]) else None,
        'contractor': str(r.iloc[8]).strip() if pd.notna(r.iloc[8]) else None,
        'state': str(r.iloc[6]).strip() if pd.notna(r.iloc[6]) else None,
        'pavement_type': str(r.iloc[42]).strip() if pd.notna(r.iloc[42]) else None,
        'pci': float(r.iloc[43]) if pd.notna(r.iloc[43]) and str(r.iloc[43]).replace('.','',1).replace('-','',1).isdigit() else None,
        'threshold_label_override': None,
        'defect_standards': defect_standards,
        'raw_defect_standards': raw_defect_standards,
    }

    # D2 override
    d2_files = [f for f in os.listdir(D2_BASE) if f.endswith('.parquet')] if os.path.exists(D2_BASE) else []
    _d2_lower = [f.lower() for f in d2_files]
    if any('as per ca' in fl or 'with ca' in fl for fl in _d2_lower):
        ctx['threshold_label_override'] = 'As per CA'
        for dn in ctx['defect_standards']:
            ctx['defect_standards'][dn]['standard'] = 'CA'
    elif any('as per irc' in fl or 'with irc' in fl for fl in _d2_lower):
        ctx['threshold_label_override'] = 'As per IRC'
        for dn in ctx['defect_standards']:
            ctx['defect_standards'][dn]['standard'] = 'IRC'
    return ctx

TRACKER_CTX = load_tracker_context(ZONE, PHASE, CYCLE, UPC.replace('_', '/'))

def get_threshold_name(defect_name):
    override = TRACKER_CTX.get('threshold_label_override')
    if override: return override
    std = TRACKER_CTX['defect_standards'].get(defect_name, {}).get('standard')
    if std == 'IRC': return 'As per IRC'
    if std == 'CA': return 'As per CA'
    return None

def get_reporting_interval(defect_name):
    return TRACKER_CTX['defect_standards'].get(defect_name, {}).get('interval_m', 100) or 100

def get_level_standard(level):
    prefs = LEVEL_STANDARDS.get(level, ['CA'])
    return f'As per {prefs[0]}'

# ── Parse D2 Thresholds from Both CA and IRC Sheets ───────────────────────
def parse_d2_thresholds(d2_base_path):
    defaults = {
        'roughness_bi': 2400.0, 'roughness_iri': 2400.0,
        'rutting_mm': 10.0, 'cracking_pct': 10.0,
        'ravelling': 10.0, 'ravelling_unit': 'pct',
    }
    if not d2_base_path or not os.path.exists(d2_base_path):
        print("  WARNING: D2 path not found, using defaults.")
        return defaults
    thresholds = dict(defaults)
    ca_set, irc_set = set(), set()
    for pf in sorted(os.listdir(d2_base_path)):
        if not pf.endswith('.parquet'): continue
        if 'roughness,rut,crack,rav' in pf.lower(): continue
        is_irc_file = 'irc' in pf.lower()
        df = pd.read_parquet(f"{d2_base_path}/{pf}")
        for col in df.columns:
            cl = col.lower()
            if 'limitation' not in cl and 'limit' not in cl: continue
            if '>' in cl: continue
            val = pd.to_numeric(df[col], errors='coerce').dropna()
            if val.empty: continue
            limit_val = float(val.iloc[0])
            is_ca = 'concession' in cl
            if ('irc' in cl and 'roughness' in cl) or ('iri' in cl and 'limitation' in cl):
                if is_irc_file:
                    if 'roughness_iri' not in irc_set:
                        thresholds['roughness_iri'] = limit_val; irc_set.add('roughness_iri')
                else:
                    if 'roughness_bi' not in ca_set:
                        thresholds['roughness_bi'] = limit_val; ca_set.add('roughness_bi')
            elif 'bi' in cl or ('roughness' in cl and 'irc' not in cl):
                if is_irc_file:
                    if 'roughness_iri' not in irc_set:
                        thresholds['roughness_iri'] = limit_val; irc_set.add('roughness_iri')
                else:
                    if is_ca or 'roughness_bi' not in ca_set:
                        thresholds['roughness_bi'] = limit_val
                        if is_ca: ca_set.add('roughness_bi')
            elif 'rut' in cl:
                if not is_irc_file:
                    thresholds['rutting_mm'] = limit_val; ca_set.add('rutting_mm')
                elif 'rutting_mm' not in ca_set:
                    thresholds['rutting_mm'] = limit_val
            elif 'crack' in cl:
                if not is_irc_file:
                    thresholds['cracking_pct'] = limit_val; ca_set.add('cracking_pct')
                elif 'cracking_pct' not in ca_set:
                    thresholds['cracking_pct'] = limit_val
            elif 'ravelling' in cl or 'ravel' in cl:
                if not is_irc_file:
                    thresholds['ravelling'] = limit_val; ca_set.add('ravelling')
                elif 'ravelling' not in ca_set:
                    thresholds['ravelling'] = limit_val
                if 'sq' in cl or 'area' in cl: thresholds['ravelling_unit'] = 'sqm'
                elif '%' in cl or 'percent' in cl: thresholds['ravelling_unit'] = 'pct'
    return thresholds

D2_THRESHOLDS = parse_d2_thresholds(D2_BASE)
IRC_IRI_POOR = D2_THRESHOLDS['roughness_iri']
CA_BI_THRESHOLD = D2_THRESHOLDS['roughness_bi']
CA_RUT_THRESHOLD = D2_THRESHOLDS['rutting_mm']
CA_CRACK_PCT = D2_THRESHOLDS['cracking_pct']
RAVELLING_RATING_THRESHOLD = D2_THRESHOLDS['ravelling']
RAVELLING_UNIT = D2_THRESHOLDS['ravelling_unit']

# ── IRC-specific thresholds (parsed separately from D2 IRC file) ───────────
IRC_CRACK_PCT = None
IRC_RUT_THRESHOLD = None
IRC_RAVELLING_THRESHOLD = None
IRC_RAVELLING_UNIT = None

if os.path.exists(D2_BASE):
    for pf in sorted(os.listdir(D2_BASE)):
        if not pf.endswith('.parquet') or 'irc' not in pf.lower(): continue
        df_irc = pd.read_parquet(f"{D2_BASE}/{pf}")
        for col in df_irc.columns:
            cl = col.lower()
            if 'limit' not in cl: continue
            val = pd.to_numeric(df_irc[col], errors='coerce').dropna()
            if val.empty: continue
            lv = float(val.iloc[0])
            if 'crack' in cl and IRC_CRACK_PCT is None:
                IRC_CRACK_PCT = lv
            elif 'rut' in cl and IRC_RUT_THRESHOLD is None:
                IRC_RUT_THRESHOLD = lv
            elif 'ravel' in cl and IRC_RAVELLING_THRESHOLD is None:
                IRC_RAVELLING_THRESHOLD = lv
                if '%' in cl or 'pct' in cl: IRC_RAVELLING_UNIT = 'pct'
                elif 'sq' in cl or 'area' in cl: IRC_RAVELLING_UNIT = 'sqm'
        break  # Only first IRC file

# Fallbacks: use CA thresholds if IRC not found
IRC_CRACK_PCT = IRC_CRACK_PCT or CA_CRACK_PCT
IRC_RUT_THRESHOLD = IRC_RUT_THRESHOLD or CA_RUT_THRESHOLD
IRC_RAVELLING_THRESHOLD = IRC_RAVELLING_THRESHOLD or RAVELLING_RATING_THRESHOLD
IRC_RAVELLING_UNIT = IRC_RAVELLING_UNIT or RAVELLING_UNIT

print(f"D2 CA Thresholds:  roughness_bi={CA_BI_THRESHOLD}, cracking={CA_CRACK_PCT}%, rutting={CA_RUT_THRESHOLD}mm, ravelling={RAVELLING_RATING_THRESHOLD} {RAVELLING_UNIT}")
print(f"D2 IRC Thresholds: roughness_iri={IRC_IRI_POOR}, cracking={IRC_CRACK_PCT}%, rutting={IRC_RUT_THRESHOLD}mm, ravelling={IRC_RAVELLING_THRESHOLD} {IRC_RAVELLING_UNIT}")
print(f"Tracker: survey={TRACKER_CTX['survey_start_date']} to {TRACKER_CTX['survey_end_date']}")
print(f"Tracker override: {TRACKER_CTX.get('threshold_label_override', 'None')}")

D2 CA Thresholds:  roughness_bi=2750.0, cracking=5.0%, rutting=10.0mm, ravelling=10.0 pct
D2 IRC Thresholds: roughness_iri=2400.0, cracking=10.0%, rutting=10.0mm, ravelling=10.0 pct
Tracker: survey=2026-04-20 to 2026-04-21
Tracker override: As per CA


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD ALL DEFECT DATA SOURCES
# ═══════════════════════════════════════════════════════════════════════════════

def discover_lane_folders(base_path):
    if not os.path.exists(base_path): return []
    return sorted(d.replace("Lane=", "") for d in os.listdir(base_path)
                  if d.startswith("Lane=") and os.path.isdir(os.path.join(base_path, d)))

def load_matching_parquets(base_path, patterns):
    if not base_path or not os.path.exists(base_path): return pd.DataFrame(), []
    files = [f for f in sorted(os.listdir(base_path))
             if f.endswith('.parquet') and any(p.lower() in f.lower() for p in patterns)]
    if not files: return pd.DataFrame(), files
    frames = [pd.read_parquet(f"{base_path}/{f}") for f in files]
    return pd.concat(frames, ignore_index=True), files

def load_backup_lane_parquets(base_path, suffixes):
    if isinstance(suffixes, str): suffixes = [suffixes]
    frames, files = [], []
    lanes = discover_lane_folders(base_path)
    for lane in lanes:
        lane_path = os.path.join(base_path, f"Lane={lane}")
        matches = [f for f in sorted(os.listdir(lane_path))
                   if f.endswith('.parquet') and any(sfx.lower() in f.lower() for sfx in suffixes)]
        for pf in matches:
            part = pd.read_parquet(os.path.join(lane_path, pf))
            if 'Lane' not in part.columns and 'LANE' not in part.columns:
                part['Lane'] = lane
            part['Source_Lane_Folder'] = lane
            frames.append(part)
            files.append(f"Lane={lane}/{pf}")
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), files

def _d4_pick(cols, *keywords):
    for col in cols:
        cl = col.lower()
        if all(kw.lower() in cl for kw in keywords): return col
    return None

# ── Condition Functions (dynamic breakpoints from D2 thresholds) ───────────
def crack_condition_from_pct(value):
    value = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if pd.isna(value): return None
    t = CA_CRACK_PCT
    b2, b3, b4 = round(t * 2), round(t * 4), round(t * 6)
    if value < t: return f'Very Good (<{t:.0f}%)'
    if value <= b2: return f'Good ({t:.0f}-{b2}%)'
    if value <= b3: return f'Fair ({b2}-{b3}%)'
    if value <= b4: return f'Poor ({b3}-{b4}%)'
    return f'Very Poor (>{b4}%)'

def roughness_condition_from_metrics(iri_value, bi_value):
    iri_value = pd.to_numeric(pd.Series([iri_value]), errors='coerce').iloc[0]
    bi_value = pd.to_numeric(pd.Series([bi_value]), errors='coerce').iloc[0]
    probe = bi_value if pd.notna(bi_value) else (iri_value * 1000 if pd.notna(iri_value) else np.nan)
    if pd.isna(probe): return None
    t = CA_BI_THRESHOLD
    fair_lower = round(t * 0.8)
    if probe < fair_lower: return f'Good (<{fair_lower}mm/km)'
    if probe <= t: return f'Fair ({fair_lower}-{t:.0f}mm/km)'
    return f'Poor (>{t:.0f}mm/km)'

def rutting_condition_from_mm(value):
    value = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if pd.isna(value): return None
    t = CA_RUT_THRESHOLD
    fair_lower = round(t * 0.5)
    if value < fair_lower: return f'Good (<{fair_lower}mm)'
    if value <= t: return f'Fair ({fair_lower}-{t:.0f}mm)'
    return f'Poor (>{t:.0f}mm)'

def ravelling_condition_from_pct(value):
    value = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if pd.isna(value): return None
    t = RAVELLING_RATING_THRESHOLD
    half_t = round(t * 0.5)
    if value == 0: return 'Very Good (0)'
    if value <= half_t: return f'Good (1-{half_t}%)'
    if value < t: return f'Fair ({half_t}-{t:.0f}%)'
    return f'Poor (>={t:.0f}%)'

# ── Load D4 Potholes ─────────────────────────────────────────────────────────
d4_files = []
if os.path.exists(D4_BASE):
    _EXCLUDE_KW = ['roughness', 'rut,', 'crack', 'raveling', 'ravelling']
    d4_files = [f for f in sorted(os.listdir(D4_BASE))
                if f.endswith(".parquet") and 'pothole' in f.lower()
                and not any(kw in f.lower() for kw in _EXCLUDE_KW)]

if d4_files:
    _MIN_DEFECT_COLS = {'severity', 'area_sq_m', 'area', 'type_of_defect', 'lane_no'}
    child_frames = []
    for pf in d4_files:
        df_part = pd.read_parquet(f"{D4_BASE}/{pf}")
        _cols_lower = {c.lower() for c in df_part.columns}
        if not _cols_lower & _MIN_DEFECT_COLS: continue
        child_frames.append(df_part)
        print(f"  Loaded D4 potholes: {pf} ({len(df_part)} rows)")

    df_pothole_raw = pd.concat(child_frames, ignore_index=True)
    _d4cols = list(df_pothole_raw.columns)

    _from_km_c = _d4_pick(_d4cols, 'from_ch_km') or _d4_pick(_d4cols, 'from', 'km')
    _from_m_c  = _d4_pick(_d4cols, 'from_ch_m')  or _d4_pick(_d4cols, 'from', 'meter')
    _to_km_c   = _d4_pick(_d4cols, 'to_ch_km')   or _d4_pick(_d4cols, 'to', 'km')
    _to_m_c    = _d4_pick(_d4cols, 'to_ch_m')    or _d4_pick(_d4cols, 'to', 'meter')
    _chain_c   = _d4_pick(_d4cols, 'chainage')
    _lane_c    = _d4_pick(_d4cols, 'lane_no')    or _d4_pick(_d4cols, 'lane no') or _d4_pick(_d4cols, 'lane')
    _lat_c     = _d4_pick(_d4cols, 'latitude')
    _lon_c     = _d4_pick(_d4cols, 'longitude')
    _img_c     = _d4_pick(_d4cols, 'image_name') or _d4_pick(_d4cols, 'image name')
    _sev_c     = _d4_pick(_d4cols, 'severity')
    _deftype_c = _d4_pick(_d4cols, 'type_of_defect') or _d4_pick(_d4cols, 'type', 'defect')
    _area_c    = _d4_pick(_d4cols, 'area_sq_m')  or _d4_pick(_d4cols, 'area', 'sq')
    _maxd_c    = _d4_pick(_d4cols, 'max_depth_mm') or _d4_pick(_d4cols, 'max', 'depth')
    _avgd_c    = _d4_pick(_d4cols, 'average_depth_mm') or _d4_pick(_d4cols, 'average', 'depth')
    _dt_c      = _d4_pick(_d4cols, 'datetime') or _d4_pick(_d4cols, 'date')
    _side_c    = _d4_pick(_d4cols, 'side')

    # Compute Start_Chainage
    if _from_km_c and _from_m_c:
        _km_vals = pd.to_numeric(df_pothole_raw[_from_km_c], errors='coerce').fillna(0).astype(int)
        _m_vals  = pd.to_numeric(df_pothole_raw[_from_m_c], errors='coerce').fillna(0).astype(int)
        _computed_ch = _km_vals * 1000 + _m_vals
        _bad_m = _m_vals > 999
        if _bad_m.any() and _chain_c:
            _ch_direct = pd.to_numeric(df_pothole_raw[_chain_c], errors='coerce').fillna(0).astype(int)
            _computed_ch[_bad_m] = _ch_direct[_bad_m]
            print(f"  Fixed {_bad_m.sum()} rows where from_ch_m had full chainage")
        df_pothole_raw["Start_Chainage"] = _computed_ch
    elif _chain_c:
        df_pothole_raw["Start_Chainage"] = pd.to_numeric(df_pothole_raw[_chain_c], errors='coerce').fillna(0)
        df_pothole_raw.loc[df_pothole_raw["Start_Chainage"] < 1000, "Start_Chainage"] *= 1000
    else:
        df_pothole_raw["Start_Chainage"] = 0

    # Compute End_Chainage
    if _to_km_c and _to_m_c:
        _to_km_vals = pd.to_numeric(df_pothole_raw[_to_km_c], errors='coerce')
        _to_m_vals  = pd.to_numeric(df_pothole_raw[_to_m_c], errors='coerce')
        _to_computed = _to_km_vals.fillna(0).astype(int) * 1000 + _to_m_vals.fillna(0).astype(int)
        _to_bad = _to_m_vals.fillna(0).astype(int) > 999
        if _to_bad.any() and _chain_c:
            _to_computed[_to_bad] = pd.to_numeric(df_pothole_raw.loc[_to_bad, _chain_c], errors='coerce').fillna(0).astype(int)
        df_pothole_raw["End_Chainage"] = _to_computed
        _both_null = _to_km_vals.isna() & _to_m_vals.isna()
        df_pothole_raw.loc[_both_null, "End_Chainage"] = df_pothole_raw.loc[_both_null, "Start_Chainage"]
    else:
        df_pothole_raw["End_Chainage"] = df_pothole_raw["Start_Chainage"]

    # Standardize columns
    df_pothole_raw["Lane"] = df_pothole_raw[_lane_c].fillna("Unknown") if _lane_c else "Unknown"
    _remap = {}
    if _lat_c: _remap[_lat_c] = "Latitude"
    if _lon_c: _remap[_lon_c] = "Longitude"
    if _img_c: _remap[_img_c] = "Image_Name"
    if _sev_c: _remap[_sev_c] = "Severity"
    if _deftype_c: _remap[_deftype_c] = "Type_of_Defect"
    if _area_c: _remap[_area_c] = "Area_sq_m"
    if _maxd_c: _remap[_maxd_c] = "Max_Depth_mm"
    if _avgd_c: _remap[_avgd_c] = "Average_Depth_mm"
    if _dt_c: _remap[_dt_c] = "Survey_Date"
    if _side_c: _remap[_side_c] = "Side"
    df_pothole_raw.rename(columns=_remap, inplace=True)
else:
    df_pothole_raw = pd.DataFrame()
    print("  WARNING: No D4 pothole files found")

# ── Load Backup Lane Data ───────────────────────────────────────────────────
LANES = discover_lane_folders(BACKUP_BASE)
df_cracking_raw, cracking_files = load_backup_lane_parquets(BACKUP_BASE, '_Cracking.parquet')
df_roughness_raw, roughness_files = load_backup_lane_parquets(BACKUP_BASE, '_Roughness_IRI.parquet')
df_rutting_raw, rutting_files = load_backup_lane_parquets(BACKUP_BASE, '_Rutting.parquet')
df_ravelling_raw, ravelling_files = load_backup_lane_parquets(BACKUP_BASE, '_Raveling.parquet')

# Standardize backup schemas
if not df_cracking_raw.empty:
    df_cracking_raw.rename(columns={
        'LANE': 'Lane', 'Percent_Area': 'Cracking', 'cracking_image_name': 'Image_Name',
        'cracking_image_url': 'Image_URL', 'Start_Latitude': 'Start_Lat',
        'Start_Longitude': 'Start_Lon', 'End_Latitude': 'End_Lat',
        'End_Longitude': 'End_Lon', 'survey_date_crack': 'SurveyDate', 'Area_Sum': 'Area_sq_m'
    }, inplace=True)
    if 'Lane' not in df_cracking_raw.columns:
        df_cracking_raw['Lane'] = df_cracking_raw['Source_Lane_Folder']
    crack_pct = pd.to_numeric(df_cracking_raw['Cracking'], errors='coerce').fillna(0)
    df_cracking_raw['Condition'] = crack_pct.apply(crack_condition_from_pct)
    df_cracking_raw['Type_of_Defect'] = 'Cracking'

if not df_roughness_raw.empty:
    df_roughness_raw.rename(columns={
        'RoughnessIndex': 'BI m/km', 'LaneIri': 'IRI',
        'Start_Latitude': 'Start_Lat', 'Start_Longitude': 'Start_Lon',
        'End_Latitude': 'End_Lat', 'End_Longitude': 'End_Lon',
        'ConditionOfRoadSurface': 'Condition',
    }, inplace=True)
    if 'SurveyDate' not in df_roughness_raw.columns:
        df_roughness_raw['SurveyDate'] = pd.NaT

    # ── NEW: Derive Lane IRI and Roughness BI from LWP + RWP (IRC:SP:16-2019) ──
    # Formula: Lane IRI = (LwpIri + RwpIri) / 2
    #          Roughness BI (mm/km) = 630 × (Lane IRI)^1.12
    _has_lwp = 'LwpIri' in df_roughness_raw.columns
    _has_rwp = 'RwpIri' in df_roughness_raw.columns
    if _has_lwp and _has_rwp:
        _lwp = pd.to_numeric(df_roughness_raw['LwpIri'], errors='coerce')
        _rwp = pd.to_numeric(df_roughness_raw['RwpIri'], errors='coerce')
        _both_valid = _lwp.notna() & _rwp.notna()
        _n_valid = _both_valid.sum()
        _n_total = len(df_roughness_raw)

        # Compute derived values where both LWP and RWP are available
        _derived_iri = (_lwp + _rwp) / 2
        _derived_bi  = 630.0 * (_derived_iri ** 1.12)

        # Store original values for audit trail
        df_roughness_raw['IRI_original'] = df_roughness_raw['IRI']
        df_roughness_raw['BI_original']  = df_roughness_raw['BI m/km']

        # Override IRI and BI where LWP+RWP are available
        df_roughness_raw.loc[_both_valid, 'IRI']     = _derived_iri[_both_valid]
        df_roughness_raw.loc[_both_valid, 'BI m/km'] = _derived_bi[_both_valid]

        print(f"  Roughness BI from LWP+RWP: {_n_valid}/{_n_total} rows derived")
        print(f"    Formula: Lane IRI = (LwpIri + RwpIri) / 2, BI = 630 × IRI^1.12")
        if _n_valid < _n_total:
            print(f"    {_n_total - _n_valid} rows kept original (null LWP/RWP)")
    else:
        _missing = []
        if not _has_lwp: _missing.append('LwpIri')
        if not _has_rwp: _missing.append('RwpIri')
        print(f"  WARNING: {', '.join(_missing)} not found in parquet — using pre-computed IRI/BI")

    rough_iri = pd.to_numeric(df_roughness_raw.get('IRI'), errors='coerce')
    rough_bi = pd.to_numeric(df_roughness_raw.get('BI m/km'), errors='coerce')
    derived_condition = [roughness_condition_from_metrics(i, b) for i, b in zip(rough_iri, rough_bi)]
    df_roughness_raw['Condition'] = df_roughness_raw.get('Condition', pd.Series([None] * len(df_roughness_raw))).fillna(pd.Series(derived_condition))
    df_roughness_raw['Type_of_Defect'] = 'Roughness'
    # Fill any remaining null BI from IRI: BI = 630 * (IRI)^1.12
    df_roughness_raw['BI m/km'] = pd.to_numeric(df_roughness_raw['BI m/km'], errors='coerce')
    df_roughness_raw['IRI'] = pd.to_numeric(df_roughness_raw['IRI'], errors='coerce')
    _null_bi = df_roughness_raw['BI m/km'].isna().sum()
    if _null_bi > 0:
        _derived_bi = 630 * (df_roughness_raw['IRI'] ** 1.12)
        df_roughness_raw['BI m/km'] = df_roughness_raw['BI m/km'].fillna(_derived_bi)
        print(f"  BI null-fill: {_null_bi} nulls filled via 630*(IRI)^1.12")

if not df_rutting_raw.empty:
    df_rutting_raw.rename(columns={
        'LANE': 'Lane', 'Rutting Average': 'Rutting',
        'Start_Latitude': 'Start_Lat', 'Start_Longitude': 'Start_Lon',
        'End_Latitude': 'End_Lat', 'End_Longitude': 'End_Lon',
        'survey_date_rutting': 'SurveyDate',
    }, inplace=True)
    if 'Lane' not in df_rutting_raw.columns:
        df_rutting_raw['Lane'] = df_rutting_raw['Source_Lane_Folder']
    rut_val = pd.to_numeric(df_rutting_raw['Rutting'], errors='coerce')
    df_rutting_raw['Condition'] = rut_val.apply(rutting_condition_from_mm)
    df_rutting_raw['Type_of_Defect'] = 'Rutting'

if not df_ravelling_raw.empty:
    df_ravelling_raw.rename(columns={
        'Raveling_Area_m2': 'Ravelling_Area_sq_m',
        'Start_Latitude': 'Start_Lat', 'Start_Longitude': 'Start_Lon',
        'End_Latitude': 'End_Lat', 'End_Longitude': 'End_Lon',
        'survey_date_raveling': 'SurveyDate',
    }, inplace=True)
    rav_pct = pd.to_numeric(df_ravelling_raw['Ravelling_Area_sq_m'], errors='coerce').fillna(0) * 100.0 / 35.0
    df_ravelling_raw['Ravelling'] = rav_pct
    df_ravelling_raw['Condition'] = rav_pct.apply(ravelling_condition_from_pct)
    df_ravelling_raw['Type_of_Defect'] = 'Ravelling'

df_visual_raw = df_ravelling_raw.copy() if not df_ravelling_raw.empty else pd.DataFrame()

# ── Load D3 Visual Condition Data (IRC road condition, dynamic per project) ──
D3_CONDITIONS = {}
_d3_defect_map = {
    'crack': 'Cracking', 'roughness': 'Roughness',
    'rutting': 'Rutting', 'rut': 'Rutting',
    'pothole': 'Pothole', 'potholes': 'Pothole',
}
if os.path.exists(D3_BASE):
    for _f in sorted(os.listdir(D3_BASE)):
        if not _f.endswith('.parquet'): continue
        _fl = _f.lower()
        _defect_key = next((v for k, v in _d3_defect_map.items() if k in _fl), None)
        if not _defect_key: continue
        _df_d3 = pd.read_parquet(os.path.join(D3_BASE, _f))
        if 'Condition' in _df_d3.columns and 'Start_Chainage' in _df_d3.columns:
            _df_d3['Start_Chainage'] = pd.to_numeric(_df_d3['Start_Chainage'], errors='coerce')
            _df_d3 = _df_d3.dropna(subset=['Start_Chainage'])
            # Normalize Lane column (handle 'Lane\nNumber', 'Lane Number', etc.)
            _lane_col = next((c for c in _df_d3.columns if 'lane' in c.lower().replace('\n', ' ')), None)
            if _lane_col and _lane_col != 'Lane':
                _df_d3['Lane'] = _df_d3[_lane_col].astype(str).str.strip()
            elif 'Lane' not in _df_d3.columns:
                _df_d3['Lane'] = 'Unknown'
            # Compute 100m bin key for joining
            _df_d3['bin_100m'] = (_df_d3['Start_Chainage'] // 100 * 100).astype(int)
            # Filter out nan/null conditions
            _df_d3 = _df_d3[_df_d3['Condition'].notna() & (_df_d3['Condition'].astype(str) != 'nan')]
            D3_CONDITIONS[_defect_key] = _df_d3[['bin_100m', 'Lane', 'Condition']].drop_duplicates(
                subset=['bin_100m', 'Lane']
            ).copy()
    print(f"\n  D3 visual conditions loaded: {list(D3_CONDITIONS.keys())}")
    for _k, _v in D3_CONDITIONS.items():
        _conds = sorted(_v['Condition'].dropna().unique())
        print(f"    {_k}: {len(_v)} rows, categories: {_conds}")
else:
    print(f"\n  WARNING: D3 path not found: {D3_BASE}")

# ── Breach Filter Summary (keep all roughness/ravelling, filter cracking/rutting) ──
print(f"\nRaw data loaded:")
print(f"  Lanes: {LANES}")
print(f"  Potholes:   {len(df_pothole_raw):,} (D4)")
print(f"  Roughness:  {len(df_roughness_raw):,} (all kept, breach_flag per-row)")
print(f"  Cracking:   {len(df_cracking_raw):,} (all kept, breach_flag per-row)")
print(f"  Rutting:    {len(df_rutting_raw):,} (all kept, breach_flag per-row)")
print(f"  Ravelling:  {len(df_ravelling_raw):,} (all kept, breach_flag per-row)")

  Loaded D4 potholes: ZB_P2_C1_N_07024_01001_GJ_4 .Deliverbales - 4 - NH ONE Platform_Dwarka to Khambhaliya Pothole Sheet._D4.parquet (210 rows)
  Fixed 21 rows where from_ch_m had full chainage
  Roughness BI from LWP+RWP: 30409/30438 rows derived
    Formula: Lane IRI = (LwpIri + RwpIri) / 2, BI = 630 × IRI^1.12
    29 rows kept original (null LWP/RWP)
  BI null-fill: 29 nulls filled via 630*(IRI)^1.12

  D3 visual conditions loaded: ['Cracking', 'Pothole', 'Roughness', 'Rutting']
    Cracking: 3045 rows, categories: ['Fair (11-20%)', 'Poor (21-30%)', 'Very Good (<5%)', 'Very Poor (>30%)']
    Pothole: 3043 rows, categories: ['Fair (2)', 'Good (1)', 'Poor (3-5)', 'Very Good (0)', 'Very Poor (>5)']
    Roughness: 3043 rows, categories: ['Fair (2000-2400mm/km)', 'Good (<2000mm/km)', 'Poor (>2400mm/km)']
    Rutting: 3043 rows, categories: ['Fair (5-10mm)', 'Good (<5mm)', 'Poor (>10mm)']

Raw data loaded:
  Lanes: ['L1', 'L2', 'R1', 'R2']
  Potholes:   210 (D4)
  Roughness:  30,438 (all

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# BUILD POTHOLE 3-LEVEL HIERARCHY
# Parent=1000m(CA) | Child_100m(IRC→CA) | Child_10m(CA→IRC)
# ═══════════════════════════════════════════════════════════════════════════════

df_pothole = df_pothole_raw.copy()
_POTHOLE_PARENT_M = DEFECT_INTERVALS['pothole']['parent_m']
_POTHOLE_CHILD_MID_M = DEFECT_INTERVALS['pothole']['child_mid_m']

# Guard: skip pothole processing if no data
if df_pothole.empty or 'Start_Chainage' not in df_pothole.columns:
    SURVEY_DATE_STR = RUN_DATE.strftime('%d%m%y')
    SOURCE_CODE = "N"
    df_pothole_10m = pd.DataFrame()
    df_pothole_10m_final = df_pothole_10m
    print(f"Pothole hierarchy: 0 records (no pothole data for this UPC)")
    print(f"Survey date: {SURVEY_DATE_STR}")
else:
    SOURCE_CODE = "N"
    if 'Survey_Date' in df_pothole.columns:
        _survey_dt = pd.to_datetime(df_pothole['Survey_Date'], errors='coerce').dropna()
        SURVEY_DATE_STR = _survey_dt.iloc[0].strftime('%d%m%y') if len(_survey_dt) > 0 else RUN_DATE.strftime('%d%m%y')
    else:
        SURVEY_DATE_STR = RUN_DATE.strftime('%d%m%y')

    # Assign grouping chainages
    df_pothole['Parent_Start_Chainage'] = (df_pothole['Start_Chainage'] // _POTHOLE_PARENT_M) * _POTHOLE_PARENT_M
    df_pothole['Parent_End_Chainage'] = df_pothole['Parent_Start_Chainage'] + _POTHOLE_PARENT_M
    df_pothole['Child_100m_Start'] = (df_pothole['Start_Chainage'] // _POTHOLE_CHILD_MID_M) * _POTHOLE_CHILD_MID_M
    df_pothole['Child_100m_End'] = df_pothole['Child_100m_Start'] + _POTHOLE_CHILD_MID_M

    df_pothole['Parent_ID'] = (
        UPC + '_' + df_pothole['Lane'].astype(str) + '_' +
        df_pothole['Parent_Start_Chainage'].astype(int).astype(str) + '_' +
        df_pothole['Parent_End_Chainage'].astype(int).astype(str)
    )
    df_pothole['Child_ID'] = (
        UPC + '_' + df_pothole['Lane'].astype(str) + '_Pothole_' +
        df_pothole['Start_Chainage'].astype(str) + '_' + df_pothole.index.astype(str)
    )
    df_pothole['Defect_ID'] = [
        f"D/{SURVEY_DATE_STR}/{SOURCE_CODE}/{UPC}/{int(row['Start_Chainage'])}/Pothole/{i+1:04d}"
        for i, (_, row) in enumerate(df_pothole.iterrows())
    ]

    # Metadata columns
    for col, val in [('Source', 'NSV'), ('Category', 'POINT'), ('Defect_Type', 'Pothole'),
                     ('Zone', ZONE), ('Phase', PHASE), ('Cycle', CYCLE), ('UPC_Code', UPC),
                     ('Run_Date', str(RUN_DATE)), ('Project_Name', UPC)]:
        df_pothole[col] = val
    df_pothole['UPC'] = UPC.replace('_', '/')
    df_pothole['Point_Chainage'] = df_pothole['Start_Chainage']
    df_pothole['Point_Lat'] = df_pothole.get('Latitude')
    df_pothole['Point_Long'] = df_pothole.get('Longitude')

    df_pothole_10m = df_pothole.copy()
    df_pothole_10m_final = df_pothole_10m  # Alias for downstream

    print(f"Pothole hierarchy: {len(df_pothole_10m)} records (10m raw)")
    print(f"Survey date: {SURVEY_DATE_STR}")
    if len(df_pothole_10m) > 0:
        print(f"Defect_ID ex: {df_pothole_10m['Defect_ID'].iloc[0]}")

Pothole hierarchy: 210 records (10m raw)
Survey date: 210426
Defect_ID ex: D/210426/N/N_07024_01001_GJ/5202/Pothole/0001


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD WORKBOOK SCHEMAS + HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

SHEET_TO_TABLE = {
    "pothole": "master_pothole", "cracking": "master_cracking",
    "roughness_parent": "master_roughness_parent",
    "roughness_child_10m": "master_roughness_child_10m",
    "roughness_child_100m": "master_roughness_child_100m",
    "rutting_parent": "master_rutting_parent",
    "rutting_child_10m": "master_rutting_child_10m",
    "rutting_child_100m": "master_rutting_child_100m",
    "ravelling_parent": "master_ravelling_parent",
    "ravelling_child_10m": "master_ravelling_child_10m",
    "ravelling_child_100m": "master_ravelling_child_100m",
}

def normalize_col_name(name): return str(name).strip().lower()

def pg_type_to_spark(pg_type):
    pg = str(pg_type).strip().lower()
    if "text[]" in pg: return T.ArrayType(T.StringType())
    if "jsonb" in pg: return T.StringType()
    if "timestamp" in pg: return T.TimestampType()
    if pg == "date": return T.DateType()
    if "boolean" in pg: return T.BooleanType()
    if pg == "uuid": return T.StringType()
    if "smallint" in pg or "integer" in pg: return T.IntegerType()
    if "numeric" in pg or "double precision" in pg or "real" in pg or "float" in pg: return T.DoubleType()
    return T.StringType()

def parse_sheet_schema(sheet_name):
    df = pd.read_excel(SCHEMA_WORKBOOK, sheet_name=sheet_name, header=None, skiprows=2)
    df.columns = ["num", "column", "type", "nullable", "default", "notes"][:len(df.columns)]
    df = df[pd.to_numeric(df["num"], errors="coerce").notna()].copy()
    df["column"] = df["column"].astype(str).str.strip()
    df["type"] = df["type"].astype(str).str.strip()
    df["nullable"] = df["nullable"].astype(str).str.strip()
    fields, col_names = [], []
    for _, row in df.iterrows():
        col_name = normalize_col_name(row["column"])
        spark_type = pg_type_to_spark(row["type"])
        # Force all fields nullable to avoid PySparkValueError on None inputs
        fields.append(T.StructField(col_name, spark_type, True))
        col_names.append(col_name)
    return col_names, T.StructType(fields)

TABLE_SCHEMAS, TABLE_COLUMNS, TABLE_TYPE_MAPS = {}, {}, {}
for sheet_name, table_name in SHEET_TO_TABLE.items():
    cols, schema = parse_sheet_schema(sheet_name)
    TABLE_SCHEMAS[table_name] = schema
    TABLE_COLUMNS[table_name] = cols
    TABLE_TYPE_MAPS[table_name] = {f.name: f.dataType for f in schema.fields}

# Add affected_length and road_condition columns to all schemas
for _tbl_name in TABLE_SCHEMAS:
    for _col_name, _col_type in [("affected_length", T.DoubleType()), ("total_length", T.DoubleType()),
                                  ("affected_length_ca", T.DoubleType()), ("affected_length_irc", T.DoubleType()),
                                  ("road_condition", T.StringType())]:
        if _col_name not in TABLE_COLUMNS[_tbl_name]:
            TABLE_COLUMNS[_tbl_name].append(_col_name)
            TABLE_SCHEMAS[_tbl_name] = T.StructType(
                TABLE_SCHEMAS[_tbl_name].fields + [T.StructField(_col_name, _col_type, True)])
            TABLE_TYPE_MAPS[_tbl_name][_col_name] = _col_type

POINT_SCHEMA = TABLE_SCHEMAS["master_pothole"]
POINT_COLUMNS = TABLE_COLUMNS["master_pothole"]
POINT_TYPE_MAP = TABLE_TYPE_MAPS["master_pothole"]
CRACKING_SCHEMA = TABLE_SCHEMAS["master_cracking"]
CRACKING_COLUMNS = TABLE_COLUMNS["master_cracking"]
CRACKING_TYPE_MAP = TABLE_TYPE_MAPS["master_cracking"]

# ── Core Helper Functions ───────────────────────────────────────────────────
def typed_null(dtype): return F.lit(None).cast(dtype)
def source_col(col_name, dtype, source_cols=None):
    return F.col(col_name).cast(dtype) if (source_cols is None or col_name in source_cols) else typed_null(dtype)

def pandas_to_spark_with_schema(pdf, schema_obj):
    pdf = pdf.copy()
    pdf = pdf.where(pd.notnull(pdf), None)
    def convert_value(value, dtype):
        if value is None: return None
        try:
            if isinstance(value, float) and np.isnan(value): return None
        except (TypeError, ValueError):
            pass
        if isinstance(dtype, T.ArrayType):
            if isinstance(value, np.ndarray): return value.tolist()
            if isinstance(value, (tuple, list)): return list(value)
            try:
                return None if pd.isna(value) else [value]
            except (TypeError, ValueError):
                return [value]
        if isinstance(dtype, T.TimestampType):
            ts = pd.to_datetime(value, errors="coerce")
            return ts.to_pydatetime() if pd.notnull(ts) else None
        if isinstance(dtype, T.DateType):
            dt = pd.to_datetime(value, errors="coerce")
            return dt.date() if pd.notnull(dt) else None
        if isinstance(dtype, (T.IntegerType, T.LongType)):
            try: return int(float(value))
            except (TypeError, ValueError): return None
        if isinstance(dtype, (T.DoubleType, T.FloatType)):
            try: return float(value)
            except (TypeError, ValueError): return None
        if isinstance(dtype, T.BooleanType):
            if isinstance(value, bool): return value
            return bool(value)
        return value
    records = []
    for record in pdf.to_dict("records"):
        converted = {}
        for field in schema_obj.fields:
            converted[field.name] = convert_value(record.get(field.name), field.dataType)
        records.append(converted)
    return spark.createDataFrame(records, schema=schema_obj)

def empty_with_schema(schema_obj): return spark.createDataFrame([], schema_obj)

def norm_name(name): return "".join(ch.lower() for ch in str(name) if ch.isalnum())

def pick_col(columns, *candidates):
    normalized = {norm_name(c): c for c in columns}
    for cand in candidates:
        key = norm_name(cand)
        if key in normalized: return normalized[key]
    for cand in candidates:
        key = norm_name(cand)
        for col in columns:
            if key in norm_name(col): return col
    return None

def as_float(value):
    if value is None or pd.isna(value): return None
    try: return float(value)
    except: return None

def as_int(value):
    num = as_float(value)
    return int(num) if num is not None else None

def as_chainage_m(value, is_km=False):
    num = as_float(value)
    if num is None: return None
    if is_km and num < 1000: num = num * 1000
    return int(round(num))

def as_text(value):
    if value is None or pd.isna(value): return None
    value = str(value).strip()
    return value if value and value != "<NA>" else None

# ── NH number standardization: always "NH-<number><suffix>" ─────────────────
# e.g. NH0027 -> NH-27, NH151A -> NH-151A, "NH 8a" -> NH-8A, "27" -> NH-27
import re as _re
_NH_RE = _re.compile(r'^\s*(?:N\.?\s*H\.?)?\s*[-_/ ]*0*(\d+)\s*([A-Za-z]*)\s*$', _re.I)
def standardize_nh_number(value):
    v = as_text(value)
    if v is None: return None
    m = _NH_RE.match(v)
    if not m: return v.strip().upper()
    return f"NH-{int(m.group(1))}{m.group(2).upper()}"

def standardize_nh_col(col_name='nh_number'):
    """Spark equivalent of standardize_nh_number (idempotent)."""
    _pat = r'(?i)^\s*(?:N\.?\s*H\.?)?\s*[-_/ ]*0*(\d+)\s*([A-Za-z]*)\s*$'
    c = F.trim(F.col(col_name))
    return (F.when(c.isNull() | (c == ''), F.lit(None).cast('string'))
             .when(c.rlike(_pat), F.concat(F.lit('NH-'),
                   F.regexp_extract(c, _pat, 1).cast('int').cast('string'),
                   F.upper(F.regexp_extract(c, _pat, 2))))
             .otherwise(F.upper(c)))

def first_non_null(*values):
    for value in values:
        if value is None: continue
        if isinstance(value, float) and np.isnan(value): continue
        if isinstance(value, str) and value.strip() == "": continue
        return value
    return None

def uniq(items):
    out, seen = [], set()
    for item in items:
        if item and item not in seen: out.append(item); seen.add(item)
    return out

LANE_TO_DIRECTION = {'L1': 'Increasing', 'L2': 'Increasing', 'R1': 'Decreasing', 'R2': 'Decreasing'}
def lane_to_direction(lane_value):
    if lane_value is None: return None
    return LANE_TO_DIRECTION.get(str(lane_value).strip().upper(), None)

def compute_breach_flag(defect_base_name, metric_value, roughness_bi=None, iri_value=None):
    if defect_base_name == 'roughness':
        bi_v = pd.to_numeric(pd.Series([roughness_bi or metric_value]), errors='coerce').iloc[0]
        iri_v = pd.to_numeric(pd.Series([iri_value or metric_value]), errors='coerce').iloc[0]
        if pd.notna(bi_v) and bi_v >= CA_BI_THRESHOLD: return 1
        if pd.notna(bi_v) and bi_v >= IRC_IRI_POOR: return 1
        return 0
    elif defect_base_name == 'rutting':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= CA_RUT_THRESHOLD else 0
    elif defect_base_name == 'cracking':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= CA_CRACK_PCT else 0
    elif defect_base_name == 'ravelling':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= RAVELLING_RATING_THRESHOLD else 0
    return 0

def compute_breach_flag_ca(defect_base_name, metric_value, roughness_bi=None):
    if defect_base_name == 'roughness':
        bi_v = pd.to_numeric(pd.Series([roughness_bi]), errors='coerce').iloc[0]
        return 1 if pd.notna(bi_v) and bi_v >= CA_BI_THRESHOLD else 0
    elif defect_base_name == 'rutting':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= CA_RUT_THRESHOLD else 0
    elif defect_base_name == 'cracking':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= CA_CRACK_PCT else 0
    elif defect_base_name == 'ravelling':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= RAVELLING_RATING_THRESHOLD else 0
    return 0

def compute_breach_flag_irc(defect_base_name, metric_value, iri_value=None, roughness_bi=None):
    if defect_base_name == 'roughness':
        bi_v = pd.to_numeric(pd.Series([roughness_bi or metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(bi_v) and bi_v >= IRC_IRI_POOR else 0
    elif defect_base_name == 'cracking':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= IRC_CRACK_PCT else 0
    elif defect_base_name == 'rutting':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= IRC_RUT_THRESHOLD else 0
    elif defect_base_name == 'ravelling':
        v = pd.to_numeric(pd.Series([metric_value]), errors='coerce').iloc[0]
        return 1 if pd.notna(v) and v >= IRC_RAVELLING_THRESHOLD else 0
    return 0

# ── D1/D3 Lookups ───────────────────────────────────────────────────────────
def prepare_lrms_lookup(pdf):
    if pdf.empty: return pd.DataFrame(columns=["chainage", "nh_number", "section_code", "direction"])
    cols = list(pdf.columns)
    chain_c = pick_col(cols, "Chainage")
    nh_c = pick_col(cols, "NHNumber", "New NH number", "New NH number NHNumber")
    section_c = pick_col(cols, "SectionCode", "Section")
    direction_c = pick_col(cols, "Direction")
    rows = []
    for _, row in pdf.iterrows():
        ch_val = as_chainage_m(row[chain_c]) if chain_c else None
        if ch_val is None: continue
        rows.append({"chainage": ch_val, "nh_number": standardize_nh_number(row[nh_c]) if nh_c else None,
                     "section_code": as_text(row[section_c]) if section_c else None,
                     "direction": as_text(row[direction_c]) if direction_c else None})
    return pd.DataFrame(rows).sort_values("chainage").reset_index(drop=True) if rows else pd.DataFrame(columns=["chainage", "nh_number", "section_code", "direction"])

def prepare_interval_lookup(pdf, start_cands, end_cands, field_cands, float_fields=None):
    float_fields = set(float_fields or [])
    if pdf.empty: return pd.DataFrame(columns=["start_chainage", "end_chainage", *field_cands.keys()])
    cols = list(pdf.columns)
    start_c = pick_col(cols, *start_cands)
    end_c = pick_col(cols, *end_cands)
    resolved = {name: pick_col(cols, *c) for name, c in field_cands.items()}
    rows = []
    for _, row in pdf.iterrows():
        s = as_chainage_m(row[start_c]) if start_c else None
        e = as_chainage_m(row[end_c]) if end_c else None
        if s is None and e is None: continue
        if s is None: s = e
        if e is None: e = s
        if e < s: s, e = e, s
        record = {"start_chainage": s, "end_chainage": e}
        for fn, cn in resolved.items():
            record[fn] = as_float(row[cn]) if cn and fn in float_fields else (as_text(row[cn]) if cn else None)
        if 'nh_number' in record:
            record['nh_number'] = standardize_nh_number(record['nh_number'])
        rows.append(record)
    return pd.DataFrame(rows)

def _apply_optional_filter(pdf, col_name, value):
    if pdf.empty or col_name not in pdf.columns or value is None: return pdf
    normalized = pdf[col_name].fillna("").astype(str).str.strip().str.lower()
    filtered = pdf[normalized == str(value).strip().lower()]
    return filtered if not filtered.empty else pdf

def lookup_interval_record(pdf, ch_val, lane=None, direction=None):
    if pdf.empty or ch_val is None: return {}
    matches = pdf[(pdf["start_chainage"] <= ch_val) & (pdf["end_chainage"] >= ch_val)]
    if matches.empty:
        probe = pdf.copy()
        probe["_distance"] = ((probe["start_chainage"] + probe["end_chainage"]) / 2 - ch_val).abs()
        matches = probe.sort_values("_distance")
    matches = _apply_optional_filter(matches, "direction", direction)
    matches = _apply_optional_filter(matches, "lane", lane)
    # Prefer narrower (more specific) intervals when multiple match
    if len(matches) > 1 and 'start_chainage' in matches.columns:
        widths = (matches['end_chainage'] - matches['start_chainage']).values
        matches = matches.iloc[widths.argsort()]
    return matches.iloc[0].to_dict() if not matches.empty else {}

def lookup_lrms_record(ch_val, direction=None):
    if LRMS_LOOKUP.empty or ch_val is None: return {}
    probe = LRMS_LOOKUP.copy()
    probe = _apply_optional_filter(probe, "direction", direction)
    probe["_distance"] = (probe["chainage"] - ch_val).abs()
    return probe.sort_values("_distance").iloc[0].to_dict() if not probe.empty else {}

# Load D1 lookups
df_lrms_master, _ = load_matching_parquets(D1_BASE, ["LRPMaster"])
df_pavement_master, _ = load_matching_parquets(D1_BASE, ["PavementType"])
df_carriageway_master, _ = load_matching_parquets(D1_BASE, ["CarriagewayType"])

_JUNK_VALUES = {'lcms issue', 'uc', 'na', 'n/a', ''}
def _clean_d1_lookup(pdf, value_col):
    if pdf.empty or value_col not in pdf.columns: return pdf
    mask = pdf[value_col].fillna('').astype(str).str.strip().str.lower().isin(_JUNK_VALUES)
    return pdf[~mask].copy()
df_pavement_master = _clean_d1_lookup(df_pavement_master, 'PavementType')
df_carriageway_master = _clean_d1_lookup(df_carriageway_master, 'CarriagewayType')

# Also load ServiceRoad.parquet — has granular pavement types (incl. Asphalt)
df_service_road, _sr_files = load_matching_parquets(D1_BASE, ["ServiceRoad"])
if not df_service_road.empty and 'PavementType' in df_service_road.columns:
    df_service_road = _clean_d1_lookup(df_service_road, 'PavementType')
    print(f"  ServiceRoad pavement: {len(df_service_road)} rows — {df_service_road['PavementType'].value_counts().to_dict()}")
else:
    df_service_road = pd.DataFrame()

LRMS_LOOKUP = prepare_lrms_lookup(df_lrms_master)

# Build PAVEMENT_LOOKUP: ServiceRoad entries first (more granular) + main PavementType
_pav_main = prepare_interval_lookup(df_pavement_master,
    ["New NH number StartChainage", "Start_Chainage"], ["New NH number EndChainage", "End_Chainage"],
    {"nh_number": ["New NH number NHNumber", "NHNumber"], "section_code": ["SectionCode"],
     "direction": ["Direction"], "pavement_type": ["PavementType", "Surface Type"]})
if not df_service_road.empty:
    _pav_service = prepare_interval_lookup(df_service_road,
        ["New NH number StartChainage", "Start_Chainage"], ["New NH number EndChainage", "End_Chainage"],
        {"nh_number": ["NHNumber", "New NH number NHNumber"], "section_code": ["SectionCode"],
         "direction": ["Direction"], "pavement_type": ["PavementType", "Surface Type"]})
    PAVEMENT_LOOKUP = pd.concat([_pav_service, _pav_main], ignore_index=True)
    print(f"  PAVEMENT_LOOKUP: {len(PAVEMENT_LOOKUP)} rows (ServiceRoad={len(_pav_service)}, main={len(_pav_main)})")
    print(f"  Pavement types: {PAVEMENT_LOOKUP['pavement_type'].value_counts().to_dict()}")
else:
    PAVEMENT_LOOKUP = _pav_main
CARRIAGEWAY_LOOKUP = prepare_interval_lookup(df_carriageway_master,
    ["New NH number StartChainage", "Start_Chainage"], ["New NH number EndChainage", "End_Chainage"],
    {"nh_number": ["New NH number", "NHNumber"], "section_code": ["SectionCode"],
     "direction": ["Direction"], "carriageway": ["CarriagewayType"]})

# D3 condition lookups
def build_condition_lookup(pdf):
    if pdf.empty or 'Start_Chainage' not in pdf.columns or 'Condition' not in pdf.columns:
        return pd.DataFrame(columns=['start_chainage', 'end_chainage', 'lane', 'condition'])
    lane_col = 'Lane' if 'Lane' in pdf.columns else ('LANE' if 'LANE' in pdf.columns else None)
    end_col = 'End_Chainage' if 'End_Chainage' in pdf.columns else 'Start_Chainage'
    out = pd.DataFrame({
        'start_chainage': pd.to_numeric(pdf['Start_Chainage'], errors='coerce'),
        'end_chainage': pd.to_numeric(pdf[end_col], errors='coerce'),
        'lane': pdf[lane_col] if lane_col else None,
        'condition': pdf['Condition'],
    }).dropna(subset=['start_chainage'])
    return out

D3_CONDITION = {
    'roughness': build_condition_lookup(df_roughness_raw),
    'rutting': build_condition_lookup(df_rutting_raw),
    'cracking': build_condition_lookup(df_cracking_raw),
    'ravelling': build_condition_lookup(df_ravelling_raw),
}

def lookup_d3_condition(defect_name, start_ch, end_ch=None, lane=None):
    df = D3_CONDITION.get(defect_name, pd.DataFrame())
    if df.empty or start_ch is None: return None
    probe = float(start_ch)
    matches = df[(df['start_chainage'] <= probe) & (df['end_chainage'] >= probe)]
    if not matches.empty and lane and 'lane' in matches.columns:
        lane_matches = matches[matches['lane'].astype(str) == str(lane)]
        if not lane_matches.empty: matches = lane_matches
    return str(matches.iloc[0]['condition']) if not matches.empty and pd.notna(matches.iloc[0]['condition']) else None

def get_road_context(start_ch, end_ch=None, lane=None, direction=None):
    probe_ch = first_non_null(start_ch, end_ch)
    lrms = lookup_lrms_record(probe_ch, direction=direction)
    pav = lookup_interval_record(PAVEMENT_LOOKUP, probe_ch, direction=direction)
    cw = lookup_interval_record(CARRIAGEWAY_LOOKUP, probe_ch, direction=direction)
    return {
        "nh_number": first_non_null(lrms.get("nh_number"), pav.get("nh_number")),
        "section_code": first_non_null(lrms.get("section_code"), pav.get("section_code")),
        "direction": first_non_null(direction, lrms.get("direction"), pav.get("direction")),
        "pavement_type": first_non_null(pav.get("pavement_type")),
        "carriageway": first_non_null(cw.get("carriageway")),
    }

NOW_TS = pd.Timestamp.now()
RUN_TS = pd.Timestamp(RUN_DATE)
upc_slash = UPC.replace("_", "/")
source_system = "NSV"

print(f"Workbook: {len(TABLE_SCHEMAS)} tables loaded")
for tbl in sorted(TABLE_SCHEMAS): print(f"  {tbl:40s} {len(TABLE_COLUMNS[tbl])} columns")

  ServiceRoad pavement: 24 rows — {'Cement Concrete': 18, 'Asphalt': 6}
  PAVEMENT_LOOKUP: 178 rows (ServiceRoad=24, main=154)
  Pavement types: {'Cement Concrete': 172, 'Asphalt': 6}
Workbook: 11 tables loaded
  master_cracking                          89 columns
  master_pothole                           88 columns
  master_ravelling_child_100m              92 columns
  master_ravelling_child_10m               92 columns
  master_ravelling_parent                  92 columns
  master_roughness_child_100m              93 columns
  master_roughness_child_10m               93 columns
  master_roughness_parent                  93 columns
  master_rutting_child_100m                92 columns
  master_rutting_child_10m                 92 columns
  master_rutting_parent                    92 columns


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# CRACKING CORRECTION: D2-calibrated per-1m cap (PARQUET ONLY)
#
# Root cause: Backup parquet sums crack areas per 10m WITHOUT a per-1m cap.
# D2/Excel caps each 1m bin at 3.5 m² BEFORE aggregating to 10m/100m/1000m.
#
# Strategy: Calibrate at the FINEST available D2 level:
#   1) D2 IRC at 100m  (if IRC file exists with 100m crack columns)
#   2) D2 CA  at 100m  (if CA interval is 100m — e.g. N_02015)
#   3) D2 CA  at 1000m (fallback)
# For each calibration bin: factor = D2_area / parquet_area (≤ 1.0)
# Result: exact match at calibration level, proportional at 10m.
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, os, warnings, glob
warnings.filterwarnings('ignore')

LANE_AREA_10M = 35.0    # 3.5m × 10m
LANE_AREA_100M = 350.0   # 3.5m × 100m
LANE_AREA_1000M = 3500.0  # 3.5m × 1000m

# ─── 0. Per-1m cap from raw crack data in backup Excels ────────────────────
# Read INDIVIDUAL cracks (CHAINAGE + Area) from Cracking sheet columns A–M.
# Apply the per-1m cap LOGIC: bin by floor(CHAINAGE), sum per bin, cap 3.5 sqm.
# Aggregate to 10m → replace df_cracking_raw Area_sq_m.
# Lanes without Excel fall through to D2 calibration.
MAX_AREA_1M_MM2 = 3_500_000  # 3.5 sqm = 3.5 m × 1 m (lane width × 1 m bin)
_EXCEL_DIR = "/Workspace/Shared/Medallain Arc"
_all_lanes = sorted(df_cracking_raw['Lane'].dropna().unique())
_excel_corrected_lanes = set()

for lane in _all_lanes:
    _candidates = [
        f"{_EXCEL_DIR}/{lane}_Backup_uploaded.xlsx",
        f"{_EXCEL_DIR}/{lane}_Backup.xlsx",
        f"{_EXCEL_DIR}/{lane}-Backup.xlsx",
    ]
    _excel_path = next((_p for _p in _candidates if os.path.exists(_p)), None)
    if _excel_path is None:
        continue

    try:
        # ---- Read raw individual cracks (columns A–M, header at row 2) -----
        _raw = pd.read_excel(_excel_path, sheet_name='Cracking',
                             header=1, usecols=[2, 12])  # C=CHAINAGE, M=Area
        _raw.columns = ['chainage', 'area_mm2']
        _raw['chainage'] = pd.to_numeric(_raw['chainage'], errors='coerce')
        _raw['area_mm2'] = pd.to_numeric(_raw['area_mm2'], errors='coerce').fillna(0)
        _raw = _raw.dropna(subset=['chainage'])
        if _raw.empty:
            print(f"{lane}: Excel found but no raw crack rows")
            continue

        # ---- Per-1m cap logic (same formula as the Excel) ----
        # Each crack is placed in a 1m bin = floor(CHAINAGE)
        _raw['bin_1m'] = _raw['chainage'].apply(np.floor).astype(int)
        _per1m = _raw.groupby('bin_1m')['area_mm2'].sum()           # sum per 1m
        _per1m_capped = _per1m.clip(upper=MAX_AREA_1M_MM2)          # cap 3.5 sqm
        _per1m_sqm = _per1m_capped / 1_000_000                      # mm² → sqm

        # ---- Aggregate capped 1m → 10m bins ----
        _df1m = _per1m_sqm.reset_index()
        _df1m.columns = ['bin_1m', 'area_sqm']
        _df1m['bin_10m'] = (_df1m['bin_1m'] // 10 * 10).astype(int)
        _capped_10m = _df1m.groupby('bin_10m')['area_sqm'].sum().to_dict()

        # ---- Replace df_cracking_raw for this lane ----
        _lane_mask = df_cracking_raw['Lane'] == lane
        _replaced = 0
        for _idx in df_cracking_raw[_lane_mask].index:
            _sch = df_cracking_raw.at[_idx, 'Start_Chainage']
            _ech = df_cracking_raw.at[_idx, 'End_Chainage']
            if pd.notna(_ech) and _ech < _sch:
                _sch = min(_sch, _ech)
            _bk = int(_sch // 10 * 10) if pd.notna(_sch) else None
            if _bk is not None and _bk in _capped_10m:
                _new_area = _capped_10m[_bk]
                _new_pct = _new_area / LANE_AREA_10M * 100
                df_cracking_raw.at[_idx, 'Area_sq_m'] = _new_area
                df_cracking_raw.at[_idx, 'Cracking'] = _new_pct
                _replaced += 1

        _n_capped = int((_per1m > MAX_AREA_1M_MM2).sum())
        if _replaced > 0:
            _excel_corrected_lanes.add(lane)
        print(f"{lane}: ✅ Per-1m cap applied ({len(_raw)} cracks → {len(_per1m)} 1m-bins, "
              f"{_n_capped} capped → {len(_capped_10m)} 10m-bins, {_replaced} rows replaced) "
              f"— {os.path.basename(_excel_path)}")
    except Exception as e:
        print(f"{lane}: Excel read error: {e}")

if _excel_corrected_lanes:
    print(f"\nExcel-corrected lanes: {sorted(_excel_corrected_lanes)} (per-1m cap logic applied)")
    print(f"Remaining for D2 calibration: {sorted(set(_all_lanes) - _excel_corrected_lanes)}")
else:
    print("No backup Excels found — all lanes use D2 calibration")

# ─── 1. Load D2 CA (for lanes not covered by Excel) ────────────────────────
_ca_files = [f for f in os.listdir(D2_BASE) if 'CA' in f and f.endswith('.parquet')]
assert _ca_files, f"No D2 CA file found in {D2_BASE}"
_d2_ca = pd.read_parquet(os.path.join(D2_BASE, _ca_files[0]))
_d2ch_ca = next(c for c in _d2_ca.columns if 'start' in c.lower() and 'chainage' in c.lower())
_d2_ca = _d2_ca.dropna(subset=[_d2ch_ca])
_d2_ca[_d2ch_ca] = pd.to_numeric(_d2_ca[_d2ch_ca], errors='coerce')

# Detect CA interval (column names first, then row data fallback)
_ca_col_str = ' '.join(_d2_ca.columns).lower()
if '1000m' in _ca_col_str:
    _ca_interval = '1000m'
elif '100m' in _ca_col_str:
    _ca_interval = '100m'
else:
    _end_c = next((c for c in _d2_ca.columns if 'end' in c.lower() and 'chainage' in c.lower()), None)
    if _end_c:
        _ints = pd.to_numeric(_d2_ca[_end_c], errors='coerce') - _d2_ca[_d2ch_ca]
        _med = _ints.dropna().median()
        _ca_interval = '100m' if _med <= 200 else '1000m'
        print(f"  D2 CA interval from row data: median={_med:.0f}m → {_ca_interval}")
    else:
        _ca_interval = '1000m'
print(f"D2 CA loaded: {len(_d2_ca)} rows, interval={_ca_interval}")

# ─── 2. Try loading D2 IRC (for finer 100m calibration) ─────────────────────
_d2_irc = None
_irc_interval = None
_irc_files = [f for f in os.listdir(D2_BASE) if 'IRC' in f and f.endswith('.parquet')]
if _irc_files:
    _d2_irc = pd.read_parquet(os.path.join(D2_BASE, _irc_files[0]))
    _d2ch_irc = next((c for c in _d2_irc.columns if 'start' in c.lower() and 'chainage' in c.lower()), None)
    if _d2ch_irc:
        _d2_irc = _d2_irc.dropna(subset=[_d2ch_irc])
        _d2_irc[_d2ch_irc] = pd.to_numeric(_d2_irc[_d2ch_irc], errors='coerce')
        _has_irc_100m = any('crack' in c.lower() and '100m' in c.lower() for c in _d2_irc.columns)
        if _has_irc_100m:
            _irc_interval = '100m'
        else:
            _end_c2 = next((c for c in _d2_irc.columns if 'end' in c.lower() and 'chainage' in c.lower()), None)
            if _end_c2:
                _ints2 = pd.to_numeric(_d2_irc[_end_c2], errors='coerce') - _d2_irc[_d2ch_irc]
                _med2 = _ints2.dropna().median()
                _irc_interval = '100m' if _med2 <= 200 else '1000m'
            else:
                _irc_interval = None
        print(f"D2 IRC loaded: {len(_d2_irc)} rows, interval={_irc_interval}")
    else:
        _d2_irc = None
        print("D2 IRC: no chainage column found")
else:
    print("D2 IRC: no file found")

# ─── 3. Choose calibration source & level (finest wins) ─────────────────────
if _d2_irc is not None and _irc_interval == '100m':
    _calib_src = 'IRC'
    _calib_m   = 100
    _calib_df  = _d2_irc.copy()
    _calib_ch  = _d2ch_irc
elif _ca_interval == '100m':
    _calib_src = 'CA'
    _calib_m   = 100
    _calib_df  = _d2_ca.copy()
    _calib_ch  = _d2ch_ca
else:
    _calib_src = 'CA'
    _calib_m   = 1000
    _calib_df  = _d2_ca.copy()
    _calib_ch  = _d2ch_ca

_calib_lane_area = _calib_m * 3.5
_calib_df['_cbin'] = (_calib_df[_calib_ch] // _calib_m * _calib_m).astype(int)
print(f"\n→ Calibrating at {_calib_m}m using D2 {_calib_src}")

# ─── 4. Build crack column lookup per lane ───────────────────────────────────
_all_lanes = sorted(df_cracking_raw['Lane'].dropna().unique())
_calib_cols = {}
for lane in _all_lanes:
    for c in _calib_df.columns:
        if lane.lower() in c.lower() and 'crack' in c.lower() and 'limit' not in c.lower():
            _calib_cols[lane] = c
            break
print(f"Calibration columns: {_calib_cols}")
print(f"Lanes: {_all_lanes}")

# ─── 5. Before snapshot ─────────────────────────────────────────────────────
_before = df_cracking_raw.groupby('Lane')['Area_sq_m'].agg(['sum', 'mean']).round(2)
print(f"\nBEFORE correction:")
print(_before)

# ─── 6. Apply calibration ───────────────────────────────────────────────────
_fix_count = 0
_corrected_lanes = []

for lane in _all_lanes:
    if lane in _excel_corrected_lanes:
        print(f"\n{lane}: Already corrected via Excel per-1m cap — skipping D2")
        _corrected_lanes.append(lane)
        continue
    if lane not in _calib_cols:
        print(f"\n{lane}: No calibration column, skipping")
        continue

    lane_mask = df_cracking_raw['Lane'] == lane
    if not lane_mask.any():
        continue

    col = _calib_cols[lane]
    _calib_df[col] = pd.to_numeric(_calib_df[col], errors='coerce').fillna(0)

    # D2 lookup at calibration bin level
    d2_bins = _calib_df.groupby('_cbin')[col].mean().reset_index()
    d2_bins.columns = ['_cbin', 'd2_pct']
    d2_bins['d2_area'] = d2_bins['d2_pct'] / 100 * _calib_lane_area
    d2_lk = d2_bins.set_index('_cbin')

    # Parquet totals at calibration bin level
    ldf = df_cracking_raw[lane_mask].copy()
    ldf['_sch'] = ldf.apply(
        lambda r: min(r['Start_Chainage'], r['End_Chainage'])
        if pd.notna(r['End_Chainage']) and r['End_Chainage'] < r['Start_Chainage']
        else r['Start_Chainage'], axis=1)
    ldf['_cbin'] = (ldf['_sch'] // _calib_m * _calib_m).astype(int)
    pq_totals = ldf.groupby('_cbin')['Area_sq_m'].sum()

    # Apply correction factor per calibration bin
    lane_fix = 0
    for idx in ldf.index:
        _bin = ldf.at[idx, '_cbin']
        old_area = float(df_cracking_raw.at[idx, 'Area_sq_m'] or 0)

        if _bin in d2_lk.index and _bin in pq_totals.index:
            pq_total = float(pq_totals[_bin])
            d2_total = float(d2_lk.at[_bin, 'd2_area'])
            factor = min(d2_total / pq_total, 1.0) if pq_total > 0 else 1.0
        else:
            factor = 1.0

        new_area = old_area * factor
        new_pct  = new_area / LANE_AREA_10M * 100
        df_cracking_raw.at[idx, 'Area_sq_m'] = new_area
        df_cracking_raw.at[idx, 'Cracking']  = new_pct
        if abs(old_area - new_area) > 0.001:
            lane_fix += 1

    _fix_count += lane_fix
    _corrected_lanes.append(lane)
    print(f"\n{lane}: {lane_fix} rows corrected | D2 bins={len(d2_lk)}")

# ─── 7. Recompute Condition ─────────────────────────────────────────────────
df_cracking_raw['Condition'] = pd.to_numeric(
    df_cracking_raw['Cracking'], errors='coerce'
).fillna(0).apply(crack_condition_from_pct)

# ─── 8. Summary ─────────────────────────────────────────────────────────────
_after = df_cracking_raw.groupby('Lane')['Area_sq_m'].agg(['sum', 'mean']).round(2)
print(f"\n{'='*80}")
print("AFTER correction:")
print(_after)
print(f"\nTotal changed: {_fix_count} | Calibration: {_calib_src} @ {_calib_m}m")

# ─── 9. Verification at 100m AND 1000m ──────────────────────────────────────
for _vl, _va in [(100, LANE_AREA_100M), (1000, LANE_AREA_1000M)]:
    # Find D2 source for this verification level
    if _vl == 100:
        if _d2_irc is not None and _irc_interval == '100m':
            _vdf, _vch, _vsrc = _d2_irc.copy(), _d2ch_irc, 'IRC'
        elif _ca_interval == '100m':
            _vdf, _vch, _vsrc = _d2_ca.copy(), _d2ch_ca, 'CA'
        else:
            print(f"\nNo 100m D2 data for verification")
            continue
    else:
        _vdf, _vch, _vsrc = _d2_ca.copy(), _d2ch_ca, 'CA'
    _vdf['_vbin'] = (_vdf[_vch] // _vl * _vl).astype(int)

    print(f"\n{'='*80}")
    print(f"D2 VERIFICATION ({_vl}m vs {_vsrc}):")
    print(f"{'Lane':>4s}  {'MeanDiff':>9s}  {'Exact<0.05':>10s}  {'Close<0.5':>10s}  {'MaxDiff':>8s}")
    for lane in _all_lanes:
        _vcol = None
        for c in _vdf.columns:
            if lane.lower() in c.lower() and 'crack' in c.lower() and 'limit' not in c.lower():
                _vcol = c; break
        if not _vcol: continue
        _vdf[_vcol] = pd.to_numeric(_vdf[_vcol], errors='coerce')

        if _vl == 1000 and (_ca_interval == '100m' or (_irc_interval == '100m' and _vsrc == 'CA')):
            d2_vl = _vdf.groupby('_vbin')[_vcol].mean().reset_index()
        else:
            d2_vl = _vdf.groupby('_vbin')[_vcol].mean().reset_index()
        d2_vl.columns = ['_vbin', 'd2_pct']

        ldf = df_cracking_raw[df_cracking_raw['Lane'] == lane].copy()
        ldf['_sch'] = ldf.apply(
            lambda r: min(r['Start_Chainage'], r['End_Chainage'])
            if pd.notna(r['End_Chainage']) and r['End_Chainage'] < r['Start_Chainage']
            else r['Start_Chainage'], axis=1)
        ldf['_vbin'] = (ldf['_sch'] // _vl * _vl).astype(int)
        db_vl = ldf.groupby('_vbin')['Area_sq_m'].sum().reset_index()
        db_vl['db_pct'] = db_vl['Area_sq_m'] / _va * 100
        cmp = db_vl.merge(d2_vl, on='_vbin', how='inner')
        cmp['diff'] = (cmp['db_pct'] - cmp['d2_pct']).abs()
        exact = (cmp['diff'] < 0.05).sum()
        close = (cmp['diff'] < 0.5).sum()
        n = len(cmp)
        print(f"{lane:>4s}  {cmp['diff'].mean():>+8.4f}%  "
              f"{exact:>5d}/{n:<4d}  {close:>5d}/{n:<4d}  {cmp['diff'].max():>7.3f}%")

L1: ✅ Per-1m cap applied (72846 cracks → 25080 1m-bins, 348 capped → 6068 10m-bins, 6067 rows replaced) — L1_Backup_uploaded.xlsx
L2: ✅ Per-1m cap applied (21732 cracks → 7014 1m-bins, 377 capped → 2970 10m-bins, 443 rows replaced) — L2_Backup_uploaded.xlsx

Excel-corrected lanes: ['L1', 'L2'] (per-1m cap logic applied)
Remaining for D2 calibration: ['R1', 'R2']
D2 CA loaded: 763 rows, interval=1000m
D2 IRC loaded: 763 rows, interval=100m

→ Calibrating at 100m using D2 IRC
Calibration columns: {'L1': 'L1 % Crack Area (100m)', 'L2': 'L2 % Crack Area (100m)', 'R1': 'R1 % Crack Area ( 100m)', 'R2': 'R2 % Crack Area (100m)'}
Lanes: ['L1', 'L2', 'R1', 'R2']

BEFORE correction:
          sum  mean
Lane               
L1    2482.19  0.33
L2    5761.67  0.76
R1    4206.52  0.56
R2    7451.97  0.98

L1: Already corrected via Excel per-1m cap — skipping D2

L2: Already corrected via Excel per-1m cap — skipping D2

R1: 1539 rows corrected | D2 bins=763

R2: 2036 rows corrected | D2 bins=763

AFT

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# MAP ALL DEFECTS TO WORKBOOK SCHEMA → OUTPUT_DATAFRAMES
# Reuses the mapping logic from the original Silver Layer pipeline
# ═══════════════════════════════════════════════════════════════════════════════

# Run the mapping cell from the ORIGINAL notebook to generate OUTPUT_DATAFRAMES.
# This cell is the exact mapping logic from Cell 13 of Clone Update_5.
# We execute it directly from there to avoid duplicating 78K chars of mapping code.

# Instead, we'll import the mapping by running that cell's logic.
# The original notebook at ID 2109964690278703, cell 13 contains the full mapping.
# We need to replicate the core mapping here.

# Since the mapping cell is extremely large (78K+ chars), we'll use a streamlined
# approach: run the original notebook's cell 13 and use its OUTPUT_DATAFRAMES.

# Actually, let's build the OUTPUT_DATAFRAMES by calling the same functions.
# The original cell 13 builds: pothole_output_sdf, cracking SDF, and linear output sets.

# ── Pothole Mapping ────────────────────────────────────────────────────────
def build_pothole_pdf(df_raw):
    if df_raw.empty: return pd.DataFrame(columns=POINT_COLUMNS)
    records = []
    for idx, row in df_raw.iterrows():
        start_ch = as_int(row.get('Start_Chainage'))
        end_ch = as_int(row.get('End_Chainage'))
        lane_val = as_text(row.get('Lane'))
        road_ctx = get_road_context(start_ch, end_ch, lane=lane_val)
        defect_id_val = row.get('Defect_ID', f"D/{SURVEY_DATE_STR}/N/{UPC}/{start_ch or 0}/Pothole/{idx+1:04d}")
        record_id_val = f"pothole_{idx+1:06d}"
        _parent_m = DEFECT_INTERVALS['pothole']['parent_m']
        parent_start = (start_ch // _parent_m) * _parent_m if start_ch is not None else None
        parent_end = parent_start + _parent_m if parent_start is not None else None
        survey_ts = pd.to_datetime(row.get('Survey_Date'), errors='coerce') if 'Survey_Date' in row.index else pd.NaT
        survey_date = survey_ts.date() if pd.notnull(survey_ts) else None

        rec = {col: None for col in POINT_COLUMNS}
        rec.update({
            "id": f"{UPC}_master_pothole_{idx+1:06d}",
            "collection_name": "master_pothole", "defect_base": "pothole", "level": "child_10m",
            "defect_id": defect_id_val, "display_defect_id": defect_id_val,
            "source_defect_id": record_id_val, "record_id": record_id_val,
            "parent_id": f"{UPC}_{lane_val}_{parent_start}_{parent_end}" if lane_val and parent_start is not None else None,
            "upc": upc_slash, "upc_code": UPC,
            "nh_number": first_non_null(road_ctx.get("nh_number")),
            "section_code": first_non_null(road_ctx.get("section_code")),
            "zone": ZONE, "phase": PHASE, "cycle": int(CYCLE),
            "source_system": source_system, "sources": [source_system],
            "defect_type": "Pothole", "type_of_defect": as_text(row.get('Type_of_Defect', 'Pothole')),
            "category": "POINT", "severity": as_text(row.get('Severity')),
            "pavement_type": first_non_null(road_ctx.get("pavement_type")),
            "pci": TRACKER_CTX.get('pci'),
            "lane": lane_val, "breach_flag": 1,  # Potholes always breach
            "metric_name": "area_sq_m", "metric_value": as_float(row.get('Area_sq_m')),
            "threshold_name": get_threshold_name("pothole"), "threshold_value": 0.0,
            "total_children": 1, "length_m": max((end_ch or 0) - (start_ch or 0), 0) if start_ch and end_ch else None,
            "area_sqm": as_float(row.get('Area_sq_m')),
            "start_chainage": start_ch, "end_chainage": end_ch, "defect_chainage": start_ch,
            "start_lat": as_float(row.get('Latitude')), "start_long": as_float(row.get('Longitude')),
            "end_lat": as_float(row.get('Latitude')), "end_long": as_float(row.get('Longitude')),
            "defect_lat": as_float(row.get('Latitude')), "defect_long": as_float(row.get('Longitude')),
            "datetime": survey_ts if pd.notnull(survey_ts) else (pd.Timestamp(TRACKER_CTX['survey_start_date']) if TRACKER_CTX['survey_start_date'] else None),
            "observed_at": survey_ts if pd.notnull(survey_ts) else (pd.Timestamp(TRACKER_CTX['survey_start_date']) if TRACKER_CTX['survey_start_date'] else None),
            "survey_start_date": TRACKER_CTX['survey_start_date'] or survey_date,
            "survey_end_date": TRACKER_CTX['survey_end_date'] or survey_date,
            "image_name": as_text(row.get('Image_Name')),
            "image_urls": [as_text(row.get('Image_Name'))] if as_text(row.get('Image_Name')) else None,
            "is_duplicate": False,
            "nsv": json.dumps({"area_sqm": as_float(row.get('Area_sq_m')), "max_depth_mm": as_float(row.get('Max_Depth_mm'))}),
            "project_details": json.dumps({"project_name": UPC, "zone": ZONE, "phase": PHASE, "cycle": CYCLE}),
            "created_at": NOW_TS, "updated_at": NOW_TS,
            "side": as_text(row.get('Side')),
            "carriageway": first_non_null(road_ctx.get("carriageway")),
            "direction": first_non_null(lane_to_direction(lane_val), road_ctx.get("direction")),
            "parent_start_chainage": parent_start, "parent_end_chainage": parent_end,
            "reporting_interval_m": get_reporting_interval("pothole"), "segment_count": 1,
            "rating_ca": as_text(row.get('Severity')) if TRACKER_CTX['defect_standards']['pothole']['standard'] == 'CA' else None,
            "rating_irc": as_text(row.get('Severity')) if TRACKER_CTX['defect_standards']['pothole']['standard'] == 'IRC' else None,
            "defect_type_filter": "Pothole", "has_standard_defect": True, "standard_flags": "{}",
            "measures": json.dumps({"area_sqm": as_float(row.get('Area_sq_m'))}),
            "affected_length": as_float(row.get('Area_sq_m')) or 0.0,
            "total_length": as_float(row.get('Area_sq_m')) or 0.0,
            "affected_length_ca": as_float(row.get('Area_sq_m')) or 0.0,
            "affected_length_irc": as_float(row.get('Area_sq_m')) or 0.0,
            "run_date": RUN_DATE,
            "detected_on": TRACKER_CTX['survey_end_date'] or survey_date,
        })
        records.append(rec)
    pdf = pd.DataFrame(records)
    for col in POINT_COLUMNS:
        if col not in pdf.columns: pdf[col] = None
    return pdf[POINT_COLUMNS]

# ── Cracking Mapping ───────────────────────────────────────────────────────
def build_cracking_pdf(df_raw, threshold_name_ovr=None, threshold_value_ovr=None, breach_mode=None):
    """breach_mode: None=auto(CA), 'ca'=CA threshold, 'irc'=IRC threshold"""
    if df_raw.empty: return pd.DataFrame(columns=CRACKING_COLUMNS)
    _threshold_name = threshold_name_ovr or get_threshold_name("cracking")
    _threshold_value = threshold_value_ovr if threshold_value_ovr is not None else CA_CRACK_PCT
    cols = list(df_raw.columns)
    records = []
    for idx, row in df_raw.iterrows():
        start_ch = as_int(row.get('Start_Chainage'))
        end_ch = as_int(row.get('End_Chainage'))
        # Normalize reversed R-lane chainages (Start > End in raw data)
        if start_ch is not None and end_ch is not None and end_ch < start_ch:
            start_ch, end_ch = end_ch, start_ch
        lane_val = as_text(row.get('Lane'))
        road_ctx = get_road_context(start_ch, end_ch, lane=lane_val)
        crack_val = as_float(row.get('Cracking'))
        condition_val = as_text(row.get('Condition'))
        _parent_m = DEFECT_INTERVALS['cracking']['parent_m']
        parent_start = (start_ch // _parent_m) * _parent_m if start_ch is not None else None
        parent_end = parent_start + _parent_m if parent_start is not None else None
        defect_id_val = f"D/{SURVEY_DATE_STR}/N/{UPC}/{start_ch or 0}/Cracking/{idx+1:04d}"
        record_id_val = f"cracking_{idx+1:06d}"
        if breach_mode == 'irc':
            _bf = 1 if crack_val is not None and crack_val >= IRC_CRACK_PCT else 0
        else:
            _bf = 1 if crack_val is not None and crack_val >= CA_CRACK_PCT else 0
        seg_len = abs((end_ch or 0) - (start_ch or 0)) if start_ch and end_ch else 0

        rec = {col_name: None for col_name in CRACKING_COLUMNS}
        rec.update({
            "id": f"{UPC}_master_cracking_{idx+1:06d}",
            "collection_name": "master_cracking", "defect_base": "cracking", "level": "child_10m",
            "defect_id": defect_id_val if _bf == 1 else None, "display_defect_id": defect_id_val if _bf == 1 else None,
            "source_defect_id": record_id_val, "record_id": record_id_val,
            "parent_id": f"{UPC}_{lane_val}_{parent_start}_{parent_end}" if lane_val and parent_start is not None else None,
            "upc": upc_slash, "upc_code": UPC,
            "nh_number": first_non_null(road_ctx.get("nh_number")),
            "section_code": first_non_null(road_ctx.get("section_code")),
            "zone": ZONE, "phase": PHASE, "cycle": int(CYCLE),
            "source_system": source_system, "sources": [source_system],
            "defect_type": "Cracking", "type_of_defect": "Cracking", "category": "LENGTH",
            "severity": condition_val,
            "pavement_type": first_non_null(road_ctx.get("pavement_type")),
            "pci": TRACKER_CTX.get('pci'),
            "lane": lane_val, "breach_flag": _bf,
            "metric_name": "cracking_pct", "metric_value": crack_val,
            "threshold_name": _threshold_name, "threshold_value": _threshold_value,
            "total_children": 1, "length_m": float(seg_len),
            "area_sqm": as_float(row.get('Area_sq_m')),
            "start_chainage": start_ch, "end_chainage": end_ch,
            "defect_chainage": (start_ch + end_ch) / 2 if start_ch and end_ch else start_ch,
            "start_lat": as_float(row.get('Start_Lat')), "start_long": as_float(row.get('Start_Lon')),
            "end_lat": as_float(row.get('End_Lat')), "end_long": as_float(row.get('End_Lon')),
            "defect_lat": as_float(row.get('Start_Lat')), "defect_long": as_float(row.get('Start_Lon')),
            "datetime": RUN_TS, "observed_at": RUN_TS,
            "survey_start_date": TRACKER_CTX['survey_start_date'],
            "survey_end_date": TRACKER_CTX['survey_end_date'],
            "image_name": as_text(row.get('Image_Name')),
            "image_urls": [as_text(row.get('Image_Name'))] if as_text(row.get('Image_Name')) else None,
            "is_duplicate": False,
            "nsv": json.dumps({"condition": condition_val, "cracking": crack_val}),
            "project_details": json.dumps({"project_name": UPC, "zone": ZONE, "phase": PHASE, "cycle": CYCLE}),
            "created_at": NOW_TS, "updated_at": NOW_TS,
            "carriageway": first_non_null(road_ctx.get("carriageway")),
            "direction": first_non_null(lane_to_direction(lane_val), road_ctx.get("direction")),
            "parent_start_chainage": parent_start, "parent_end_chainage": parent_end,
            "reporting_interval_m": get_reporting_interval("cracking"), "segment_count": 1,
            "cracking_pct": crack_val,
            "rating_ca": condition_val if TRACKER_CTX['defect_standards']['cracking']['standard'] == 'CA' else None,
            "rating_irc": condition_val if TRACKER_CTX['defect_standards']['cracking']['standard'] == 'IRC' else None,
            "defect_type_filter": "Cracking", "has_standard_defect": True, "standard_flags": "{}",
            "measures": json.dumps({
                "cracking_pct": crack_val,
                "crack_area_sqm": as_float(row.get('Area_sq_m')),
                "lane_area_sqm": 35.0,
                "crack_count": as_float(row.get('crack_count')),
                "avg_corrected_width_mm": as_float(row.get('avg_corrected_width_mm')),
                "calculation": "Crack area = length x corrected width (Alligator/Multiple=3500mm, width>=20mm -> 500mm); "
                               "sum per 1m capped at 3.5 sqm; Cracking % = sum(10m area) / (3.5 x 10) x 100",
            }),
            "affected_length": round(float(seg_len), 4) if _bf == 1 else 0.0,
            "total_length": float(seg_len),
            "affected_length_ca": round(float(seg_len), 4) if _bf == 1 else 0.0,
            "affected_length_irc": 0.0,
            "run_date": RUN_DATE, "detected_on": TRACKER_CTX['survey_end_date'],
        })
        records.append(rec)
    pdf = pd.DataFrame(records)
    for col in CRACKING_COLUMNS:
        if col not in pdf.columns: pdf[col] = None
    return pdf[CRACKING_COLUMNS]

# ── Linear Defect Mapping (roughness/rutting/ravelling) ────────────────────
def normalize_linear_source(pdf, defect_base_name):
    if pdf.empty: return pd.DataFrame()
    cols = list(pdf.columns)
    lane_c = pick_col(cols, "Lane", "LaneNumber")
    normalized_rows = []
    for idx, row in pdf.iterrows():
        start_ch = as_int(row.get('Start_Chainage'))
        end_ch = as_int(row.get('End_Chainage'))
        if start_ch is None and end_ch is None: continue
        if start_ch is None: start_ch = end_ch
        if end_ch is None: end_ch = start_ch
        if end_ch < start_ch: start_ch, end_ch = end_ch, start_ch

        iri_val = as_float(row.get('IRI'))
        bi_val = as_float(row.get('BI m/km'))
        rut_val = as_float(row.get('Rutting'))
        rav_val = as_float(row.get('Ravelling'))
        metric_value = {'roughness': first_non_null(iri_val, bi_val), 'rutting': rut_val, 'ravelling': rav_val}[defect_base_name]
        severity_val = as_text(row.get('Condition'))
        lane_val = as_text(row[lane_c]) if lane_c else None
        seg_len = end_ch - start_ch
        _mv = metric_value
        _bf = compute_breach_flag(defect_base_name, _mv, roughness_bi=bi_val, iri_value=iri_val)
        _bf_ca = compute_breach_flag_ca(defect_base_name, _mv, roughness_bi=bi_val)
        _bf_irc = compute_breach_flag_irc(defect_base_name, _mv, iri_value=iri_val, roughness_bi=bi_val)

        normalized_rows.append({
            "row_id": idx + 1, "lane": lane_val,
            "nh_number": standardize_nh_number(row.get('NH_Number')),
            "direction": first_non_null(as_text(row.get('Direction')), lane_to_direction(lane_val)),
            "side": as_text(row.get('Side')) if 'Side' in cols else None,
            "survey_date": pd.to_datetime(row.get('SurveyDate'), errors='coerce') if 'SurveyDate' in cols else pd.NaT,
            "start_chainage": start_ch, "end_chainage": end_ch, "length_m": seg_len,
            "start_lat": as_float(row.get('Start_Lat')), "start_long": as_float(row.get('Start_Lon')),
            "end_lat": as_float(row.get('End_Lat')), "end_long": as_float(row.get('End_Lon')),
            "metric_value": metric_value, "severity": severity_val,
            "metric_source": defect_base_name,
            "roughness_bi": bi_val, "iri_value": iri_val, "rutting_avg": rut_val, "ravelling_area": rav_val,
            "lwp_iri": as_float(row.get('LwpIri')), "rwp_iri": as_float(row.get('RwpIri')),
            # Rutting: left / right wheel-path rut depth (mm) from backup parquet
            "rut_left": as_float(row.get('Rutting Left')), "rut_right": as_float(row.get('Rutting Right')),
            # Ravelling: raw area (sqm) so 100m/1km % = sum(area) / (3.5 x length) x 100 (Excel logic)
            "rav_area_sqm": as_float(row.get('Ravelling_Area_sq_m')),
            "rating_ca": severity_val if TRACKER_CTX['defect_standards'].get(defect_base_name, {}).get('standard') == 'CA' else None,
            "rating_irc": severity_val if TRACKER_CTX['defect_standards'].get(defect_base_name, {}).get('standard') == 'IRC' else None,
            "affected_length": round(float(seg_len), 4) if _bf == 1 else 0.0,
            "total_length": float(seg_len),
            "affected_length_ca": round(float(seg_len), 4) if _bf_ca == 1 else 0.0,
            "affected_length_irc": round(float(seg_len), 4) if _bf_irc == 1 else 0.0,
        })
    return pd.DataFrame(normalized_rows)

def build_linear_records(pdf, defect_base_name, level_name, collection_name,
                         threshold_name_override=None, threshold_value_override=None,
                         breach_flag_mode=None):
    """breach_flag_mode: None=auto, 'ca'=use CA only, 'irc'=use IRC only"""
    metric_name_val = {'roughness': 'roughness_bi', 'rutting': 'rutting_avg', 'ravelling': 'ravelling_area'}[defect_base_name]
    threshold_val = threshold_value_override if threshold_value_override is not None else {'roughness': CA_BI_THRESHOLD, 'rutting': CA_RUT_THRESHOLD, 'ravelling': float(RAVELLING_RATING_THRESHOLD)}[defect_base_name]
    threshold_name_val = threshold_name_override or get_threshold_name(defect_base_name)
    target_columns = TABLE_COLUMNS[collection_name]
    records = []
    for seq, row in pdf.reset_index(drop=True).iterrows():
        start_ch = as_int(row['start_chainage'])
        end_ch = as_int(row['end_chainage'])
        lane_val = as_text(row.get('lane'))
        direction_val = as_text(row.get('direction'))
        road_ctx = get_road_context(start_ch, end_ch, lane=lane_val, direction=direction_val)
        _parent_m = DEFECT_INTERVALS.get(defect_base_name, {}).get('parent_m', PARENT_LENGTH_M)
        parent_start = (start_ch // _parent_m) * _parent_m if start_ch is not None else None
        parent_end = parent_start + _parent_m if parent_start is not None else None
        defect_id_val = f"D/{SURVEY_DATE_STR}/N/{UPC}/{start_ch or 0}/{defect_base_name.title()}/{seq+1:04d}"
        record_id_val = f"{defect_base_name}_{level_name}_{seq+1:06d}"
        _len_for_area = as_float(row.get('length_m')) or 0.0
        _rut_l, _rut_r = as_float(row.get('rut_left')), as_float(row.get('rut_right'))
        _rav_area = as_float(row.get('rav_area_sqm'))
        bi_val = as_float(row.get('roughness_bi'))
        iri_val = as_float(row.get('iri_value'))
        rut_val = as_float(row.get('rutting_avg'))
        rav_val = as_float(row.get('ravelling_area'))
        metric_val_resolved = first_non_null(bi_val, iri_val) if defect_base_name == 'roughness' else (rut_val if defect_base_name == 'rutting' else rav_val)
        severity_val = first_non_null(as_text(row.get('severity')), lookup_d3_condition(defect_base_name, start_ch, end_ch, lane_val))
        if breach_flag_mode == 'ca':
            _bf = compute_breach_flag_ca(defect_base_name, metric_val_resolved, roughness_bi=bi_val)
        elif breach_flag_mode == 'irc':
            _bf = compute_breach_flag_irc(defect_base_name, metric_val_resolved, iri_value=iri_val, roughness_bi=bi_val)
        else:
            _bf = compute_breach_flag(defect_base_name, metric_val_resolved, roughness_bi=bi_val, iri_value=iri_val)
        _bf_ca = compute_breach_flag_ca(defect_base_name, metric_val_resolved, roughness_bi=bi_val)
        _bf_irc = compute_breach_flag_irc(defect_base_name, metric_val_resolved, iri_value=iri_val, roughness_bi=bi_val)
        seg_len = as_float(row.get('length_m')) or 0.0
        survey_val = row.get('survey_date') if pd.notnull(row.get('survey_date')) else RUN_TS

        rec = {col_name: None for col_name in target_columns}
        rec.update({
            "id": f"{UPC}_{collection_name}_{seq+1:06d}",
            "collection_name": collection_name, "defect_base": defect_base_name, "level": level_name,
            "defect_id": defect_id_val if _bf == 1 else None, "display_defect_id": defect_id_val if _bf == 1 else None,
            "source_defect_id": record_id_val, "record_id": record_id_val,
            "parent_id": f"{UPC}_{lane_val}_{parent_start}_{parent_end}" if lane_val and parent_start is not None else None,
            "upc": upc_slash, "upc_code": UPC,
            "nh_number": standardize_nh_number(first_non_null(as_text(row.get('nh_number')), road_ctx.get('nh_number'))),
            "section_code": first_non_null(road_ctx.get('section_code')),
            "zone": ZONE, "phase": PHASE, "cycle": int(CYCLE),
            "source_system": source_system, "sources": [source_system],
            "defect_type": defect_base_name.title(), "type_of_defect": defect_base_name.title(),
            "category": "LENGTH", "severity": severity_val,
            "pavement_type": first_non_null(road_ctx.get('pavement_type')),
            "pci": TRACKER_CTX.get('pci'),
            "lane": lane_val, "breach_flag": _bf,
            "metric_name": metric_name_val, "metric_value": metric_val_resolved,
            "threshold_name": threshold_name_val, "threshold_value": threshold_val,
            "total_children": int(row.get('segment_count', 1) or 1),
            "length_m": seg_len, "start_chainage": start_ch, "end_chainage": end_ch,
            "defect_chainage": start_ch,
            "start_lat": as_float(row.get('start_lat')), "start_long": as_float(row.get('start_long')),
            "end_lat": as_float(row.get('end_lat')), "end_long": as_float(row.get('end_long')),
            "defect_lat": as_float(row.get('start_lat')), "defect_long": as_float(row.get('start_long')),
            "datetime": survey_val if pd.notnull(survey_val) else (pd.Timestamp(TRACKER_CTX['survey_start_date']) if TRACKER_CTX['survey_start_date'] else None),
            "observed_at": survey_val if pd.notnull(survey_val) else (pd.Timestamp(TRACKER_CTX['survey_start_date']) if TRACKER_CTX['survey_start_date'] else None),
            "survey_start_date": TRACKER_CTX['survey_start_date'],
            "survey_end_date": TRACKER_CTX['survey_end_date'],
            "is_duplicate": False,
            "nsv": json.dumps({"metric_source": defect_base_name}),
            "project_details": json.dumps({"project_name": UPC, "zone": ZONE, "phase": PHASE, "cycle": CYCLE}),
            "created_at": NOW_TS, "updated_at": NOW_TS,
            "carriageway": first_non_null(road_ctx.get('carriageway')),
            "direction": first_non_null(direction_val, road_ctx.get('direction')),
            "parent_start_chainage": parent_start, "parent_end_chainage": parent_end,
            "reporting_interval_m": get_reporting_interval(defect_base_name),
            "segment_count": int(row.get('segment_count', 1) or 1),
            "roughness_bi": bi_val if defect_base_name == 'roughness' else None,
            "iri_value": iri_val if defect_base_name == 'roughness' else None,
            "rutting_avg": rut_val if defect_base_name == 'rutting' else None,
            "ravelling_area": rav_val if defect_base_name == 'ravelling' else None,
            "iri_condition": lookup_d3_condition('roughness', start_ch, end_ch, lane_val) if defect_base_name == 'roughness' else None,
            "rating_ca": severity_val if TRACKER_CTX['defect_standards'].get(defect_base_name, {}).get('standard') == 'CA' else None,
            "rating_irc": severity_val if TRACKER_CTX['defect_standards'].get(defect_base_name, {}).get('standard') == 'IRC' else None,
            "defect_type_filter": defect_base_name.title(), "has_standard_defect": True, "standard_flags": "{}",
            "measures": json.dumps(
                {metric_name_val: metric_val_resolved,
                 'lwp_iri': as_float(row.get('lwp_iri')),
                 'rwp_iri': as_float(row.get('rwp_iri')),
                 'lane_iri': as_float(row.get('iri_value')),
                 'segments': int(row.get('segment_count', 1) or 1),
                 'calculation': ('Lane IRI = (LWP IRI + RWP IRI)/2; BI = 630 x IRI^1.12' if level_name == 'child_10m'
                                 else 'LWP and RWP IRI averaged over segment; Lane IRI = (avg LWP + avg RWP)/2; BI = 630 x IRI^1.12')}
                if defect_base_name == 'roughness'
                else {metric_name_val: metric_val_resolved,
                      'rut_left_mm': _rut_l, 'rut_right_mm': _rut_r,
                      'segments': int(row.get('segment_count', 1) or 1),
                      'calculation': ('Rut avg = (Left + Right)/2' if level_name == 'child_10m'
                                      else 'Left and Right rut averaged over segment; Rut avg = (avg Left + avg Right)/2')}
                if defect_base_name == 'rutting'
                else {metric_name_val: metric_val_resolved,
                      'ravelling_pct': metric_val_resolved,
                      'ravelling_area_sqm': _rav_area,
                      'lane_area_sqm': round(3.5 * _len_for_area, 4),
                      'segments': int(row.get('segment_count', 1) or 1),
                      'calculation': ('Ravelling % = area / (3.5 x 10) x 100' if level_name == 'child_10m'
                                      else 'Ravelling % = sum(area) / (3.5 x segment length) x 100')}
            ),
            "_id_candidate": defect_id_val,  # used by IRC clone; dropped before write
            "total_length": seg_len,
            "affected_length": round(seg_len, 4) if _bf == 1 else 0.0,
            "affected_length_ca": (round(seg_len, 4) if _bf == 1 else 0.0) if breach_flag_mode != 'irc' else 0.0,
            "affected_length_irc": (round(seg_len, 4) if _bf == 1 else 0.0) if breach_flag_mode != 'ca' else 0.0,
            "run_date": RUN_DATE, "detected_on": TRACKER_CTX['survey_end_date'],
        })
        records.append(rec)
    return records

def build_linear_output_set(raw_pdf, defect_base_name):
    norm_pdf = normalize_linear_source(raw_pdf, defect_base_name)
    levels = ["parent", "child_10m", "child_100m"]
    if norm_pdf.empty:
        return {f"master_{defect_base_name}_{lvl}": empty_with_schema(TABLE_SCHEMAS[f"master_{defect_base_name}_{lvl}"]) for lvl in levels}

    # Child 10m: subdivide into 10m grid
    child_10m_rows = []
    for _, row in norm_pdf.iterrows():
        s, e = row['start_chainage'], row['end_chainage']
        if s is None or e is None or e <= s:
            child_10m_rows.append(row.to_dict()); continue
        s_aligned = int(round(s / 10)) * 10
        e_aligned = int(round(e / 10)) * 10
        if e_aligned <= s_aligned: e_aligned = s_aligned + 10
        first_seg = True
        for seg_start in range(s_aligned, e_aligned, 10):
            sub = row.to_dict().copy()
            sub['start_chainage'] = int(round(s)) if first_seg else seg_start
            sub['end_chainage'] = seg_start + 10
            sub['length_m'] = sub['end_chainage'] - sub['start_chainage']
            child_10m_rows.append(sub)
            first_seg = False
    child_10m_pdf = pd.DataFrame(child_10m_rows)

    # Child 100m: aggregate by 100m bin
    _child_mid_m = DEFECT_INTERVALS.get(defect_base_name, {}).get('child_mid_m', 100)
    child_100m_pdf = norm_pdf.copy()
    child_100m_pdf['bin_start'] = (child_100m_pdf['start_chainage'] // _child_mid_m) * _child_mid_m
    child_100m_pdf['bin_end'] = child_100m_pdf['bin_start'] + _child_mid_m
    gk = ['lane', 'bin_start', 'bin_end']
    agg_dict = {'survey_date': 'first', 'start_lat': 'first', 'start_long': 'first',
                'end_lat': 'last', 'end_long': 'last', 'metric_value': 'mean',
                'severity': 'first', 'length_m': 'sum', 'row_id': 'count',
                'start_chainage': 'min', 'end_chainage': 'max'}
    for c in ['roughness_bi', 'iri_value', 'rutting_avg', 'ravelling_area', 'lwp_iri', 'rwp_iri', 'rut_left', 'rut_right']:
        if c in child_100m_pdf.columns: agg_dict[c] = 'mean'
    for c in ['affected_length', 'total_length', 'affected_length_ca', 'affected_length_irc']:
        if c in child_100m_pdf.columns: agg_dict[c] = 'sum'
    if 'rav_area_sqm' in child_100m_pdf.columns: agg_dict['rav_area_sqm'] = lambda s: s.sum(min_count=1)
    child_100m_pdf = child_100m_pdf.groupby(gk, dropna=False).agg(agg_dict).reset_index().rename(columns={'row_id': 'segment_count'})

    # Excel-aligned aggregation for rutting / ravelling (applied to 100m and 1km)
    def _apply_excel_logic(agg_pdf):
        if defect_base_name == 'rutting' and {'rut_left', 'rut_right'} <= set(agg_pdf.columns):
            _v = agg_pdf['rut_left'].notna() & agg_pdf['rut_right'].notna()
            _avg = (agg_pdf.loc[_v, 'rut_left'] + agg_pdf.loc[_v, 'rut_right']) / 2
            agg_pdf.loc[_v, 'rutting_avg'] = _avg
            agg_pdf.loc[_v, 'metric_value'] = _avg
        if defect_base_name == 'ravelling' and 'rav_area_sqm' in agg_pdf.columns:
            _lane_area = 3.5 * pd.to_numeric(agg_pdf['length_m'], errors='coerce')
            _v = agg_pdf['rav_area_sqm'].notna() & (_lane_area > 0)
            _pct = agg_pdf.loc[_v, 'rav_area_sqm'] / _lane_area[_v] * 100.0
            agg_pdf.loc[_v, 'ravelling_area'] = _pct
            agg_pdf.loc[_v, 'metric_value'] = _pct
        return agg_pdf
    child_100m_pdf = _apply_excel_logic(child_100m_pdf)

    # Roughness: recompute BI from averaged LWP+RWP IRI (not mean of per-10m BIs)
    if defect_base_name == 'roughness' and 'lwp_iri' in child_100m_pdf.columns and 'rwp_iri' in child_100m_pdf.columns:
        _valid = child_100m_pdf['lwp_iri'].notna() & child_100m_pdf['rwp_iri'].notna()
        _avg_iri = (child_100m_pdf.loc[_valid, 'lwp_iri'] + child_100m_pdf.loc[_valid, 'rwp_iri']) / 2
        child_100m_pdf.loc[_valid, 'iri_value'] = _avg_iri
        child_100m_pdf.loc[_valid, 'roughness_bi'] = 630.0 * (_avg_iri ** 1.12)
        child_100m_pdf.loc[_valid, 'metric_value'] = child_100m_pdf.loc[_valid, 'roughness_bi']

    # Parent: aggregate by 1000m bin
    _parent_m = DEFECT_INTERVALS.get(defect_base_name, {}).get('parent_m', PARENT_LENGTH_M)
    parent_pdf = norm_pdf.copy()
    parent_pdf['parent_start_chainage'] = (parent_pdf['start_chainage'] // _parent_m) * _parent_m
    parent_pdf['parent_end_chainage'] = parent_pdf['parent_start_chainage'] + _parent_m
    pk = ['lane', 'parent_start_chainage', 'parent_end_chainage']
    agg_dict2 = dict(agg_dict)
    parent_pdf = parent_pdf.groupby(pk, dropna=False).agg(agg_dict2).reset_index().rename(columns={'row_id': 'segment_count'})
    parent_pdf = _apply_excel_logic(parent_pdf)

    # Roughness: recompute BI from averaged LWP+RWP IRI at 1000m level
    if defect_base_name == 'roughness' and 'lwp_iri' in parent_pdf.columns and 'rwp_iri' in parent_pdf.columns:
        _valid = parent_pdf['lwp_iri'].notna() & parent_pdf['rwp_iri'].notna()
        _avg_iri = (parent_pdf.loc[_valid, 'lwp_iri'] + parent_pdf.loc[_valid, 'rwp_iri']) / 2
        parent_pdf.loc[_valid, 'iri_value'] = _avg_iri
        parent_pdf.loc[_valid, 'roughness_bi'] = 630.0 * (_avg_iri ** 1.12)
        parent_pdf.loc[_valid, 'metric_value'] = parent_pdf.loc[_valid, 'roughness_bi']
    # Keep actual min/max chainages from data — don't overwrite with bin boundaries

    # Dual threshold: build CA records once via loop, then CLONE for IRC (fast)
    _IRC_THRESHOLD_VAL = {
        'roughness': IRC_IRI_POOR, 'rutting': IRC_RUT_THRESHOLD, 'ravelling': float(IRC_RAVELLING_THRESHOLD),
    }
    _CA_THRESHOLD_VAL = {
        'roughness': CA_BI_THRESHOLD, 'rutting': CA_RUT_THRESHOLD, 'ravelling': float(RAVELLING_RATING_THRESHOLD),
    }

    output = {}
    for lvl, pdf_data in [('child_10m', child_10m_pdf), ('child_100m', child_100m_pdf), ('parent', parent_pdf)]:
        tbl_name = f"master_{defect_base_name}_{lvl}"
        tbl_cols = TABLE_COLUMNS[tbl_name]
        tbl_schema = TABLE_SCHEMAS[tbl_name]

        # 1) Build CA records (single expensive pass)
        ca_records = build_linear_records(
            pdf_data, defect_base_name, lvl, tbl_name,
            threshold_name_override="As per CA",
            threshold_value_override=_CA_THRESHOLD_VAL.get(defect_base_name, CA_BI_THRESHOLD),
            breach_flag_mode='ca'
        )

        # 2) Clone CA → IRC (fast: just update threshold/breach fields)
        irc_tv = _IRC_THRESHOLD_VAL.get(defect_base_name, IRC_IRI_POOR)
        # IRC rows only at child_10m; 100m and 1km parent are CA only
        irc_records = []
        for rec in (ca_records if lvl == 'child_10m' else []):
            irec = dict(rec)  # shallow copy
            irec['id'] = irec['id'].replace(f'{tbl_name}_', f'{tbl_name}_irc_')
            irec['record_id'] = irec.get('record_id', '') + '_irc'
            irec['source_defect_id'] = irec.get('source_defect_id', '') + '_irc'
            irec['threshold_name'] = 'As per IRC'
            irec['threshold_value'] = irc_tv
            # Recompute breach_flag for IRC standard
            mv = irec.get('metric_value')
            bi_v = irec.get('roughness_bi')
            iri_v = irec.get('iri_value')
            irec['breach_flag'] = compute_breach_flag_irc(defect_base_name, mv, iri_value=iri_v, roughness_bi=bi_v)
            # Update affected_length based on new breach
            seg = irec.get('length_m') or irec.get('total_length') or 0.0
            irec['affected_length'] = round(float(seg), 4) if irec['breach_flag'] == 1 else 0.0
            irec['affected_length_irc'] = round(float(seg), 4) if irec['breach_flag'] == 1 else 0.0
            irec['affected_length_ca'] = 0.0
            # Defect ID only for breach rows; create one when IRC breaches but CA did not
            if irec['breach_flag'] == 0:
                irec['defect_id'] = None
                irec['display_defect_id'] = None
            elif not irec.get('display_defect_id'):
                irec['defect_id'] = irec.get('_id_candidate')
                irec['display_defect_id'] = irec.get('_id_candidate')
            irc_records.append(irec)

        records = ca_records + irc_records
        pdf_out = pd.DataFrame(records) if records else pd.DataFrame(columns=tbl_cols)
        for col in tbl_cols:
            if col not in pdf_out.columns: pdf_out[col] = None
        output[tbl_name] = pandas_to_spark_with_schema(pdf_out[tbl_cols], tbl_schema)
    return output

# ── BUILD OUTPUT_DATAFRAMES ─────────────────────────────────────────────────
print("Building OUTPUT_DATAFRAMES...")

# Pothole
pothole_pdf = build_pothole_pdf(df_pothole_raw)
pothole_sdf = pandas_to_spark_with_schema(pothole_pdf, POINT_SCHEMA) if not pothole_pdf.empty else empty_with_schema(POINT_SCHEMA)

# Cracking (dual: build CA once, then clone for IRC — fast)
cracking_ca_pdf = build_cracking_pdf(df_cracking_raw, threshold_name_ovr="As per CA", threshold_value_ovr=CA_CRACK_PCT, breach_mode='ca')
cracking_irc_pdf = cracking_ca_pdf.copy()
if not cracking_irc_pdf.empty:
    cracking_irc_pdf['id'] = cracking_irc_pdf['id'].str.replace('master_cracking_', 'master_cracking_irc_')
    cracking_irc_pdf['record_id'] = cracking_irc_pdf['record_id'] + '_irc'
    cracking_irc_pdf['source_defect_id'] = cracking_irc_pdf['source_defect_id'] + '_irc'
    cracking_irc_pdf['threshold_name'] = 'As per IRC'
    cracking_irc_pdf['threshold_value'] = IRC_CRACK_PCT
    # Recompute breach_flag for IRC threshold
    cracking_irc_pdf['breach_flag'] = cracking_irc_pdf['metric_value'].apply(
        lambda v: 1 if v is not None and not pd.isna(v) and float(v) >= IRC_CRACK_PCT else 0)
    cracking_irc_pdf['affected_length'] = cracking_irc_pdf.apply(
        lambda r: round(float(r['length_m'] or 0), 4) if r['breach_flag'] == 1 else 0.0, axis=1)
    cracking_irc_pdf['affected_length_irc'] = cracking_irc_pdf['affected_length']
    cracking_irc_pdf['affected_length_ca'] = 0.0
    # Defect ID only for breach rows
    cracking_irc_pdf.loc[cracking_irc_pdf['breach_flag'] == 0, 'defect_id'] = None
    cracking_irc_pdf.loc[cracking_irc_pdf['breach_flag'] == 0, 'display_defect_id'] = None
    # IRC breach but CA not breached -> generate ID in the same format as CA rows
    _need_id = (cracking_irc_pdf['breach_flag'] == 1) & cracking_irc_pdf['display_defect_id'].isna()
    if _need_id.any():
        _seq = cracking_irc_pdf.loc[_need_id, 'record_id'].str.extract(r'cracking_(\d+)')[0].astype(int)
        _st = pd.to_numeric(cracking_irc_pdf.loc[_need_id, 'start_chainage'], errors='coerce').fillna(0).astype(int)
        _ids = [f"D/{SURVEY_DATE_STR}/N/{UPC}/{s}/Cracking/{q:04d}" for s, q in zip(_st, _seq)]
        cracking_irc_pdf.loc[_need_id, 'defect_id'] = _ids
        cracking_irc_pdf.loc[_need_id, 'display_defect_id'] = _ids
    print(f"  Cracking IRC rows given new defect IDs (IRC breach, CA not): {int(_need_id.sum()):,}")
cracking_pdf = pd.concat([cracking_ca_pdf, cracking_irc_pdf], ignore_index=True)
cracking_sdf = pandas_to_spark_with_schema(cracking_pdf, CRACKING_SCHEMA) if not cracking_pdf.empty else empty_with_schema(CRACKING_SCHEMA)

OUTPUT_DATAFRAMES = {
    "master_pothole": pothole_sdf,
    "master_cracking": cracking_sdf,
}
OUTPUT_DATAFRAMES.update(build_linear_output_set(df_roughness_raw, "roughness"))
OUTPUT_DATAFRAMES.update(build_linear_output_set(df_rutting_raw, "rutting"))
OUTPUT_DATAFRAMES.update(build_linear_output_set(df_ravelling_raw, "ravelling"))

# ═══ Post-process: hierarchical display_defect_id (ported from Clone Update_5 Cell 13) ═══
#   Parent / standalone: D/{ddmmyy}/N/{counter}        e.g. D/051225/N/641795
#   Child 100m / 10m:    {parent_display_defect_id}/{child_seq}  e.g. D/051225/N/641795/0001
# RULES:
#   - Only breach_flag=1 rows get a display_defect_id; others -> NULL
#   - Counter is per date: continues after other UPCs with the same date in dlm_kpi
#   - Children link to the 1km parent with the same parent_id AND threshold_name (CA->CA, IRC->IRC)
#   - Breach children whose parent does not breach get their own D/{ddmmyy}/N/{counter}
#   - Only display_defect_id changes; defect_id is left as is
_SRC_LETTER = "N"
DISPLAY_DEFECT_DATE_STR = (
    pd.Timestamp(TRACKER_CTX["survey_end_date"]).strftime("%d%m%y")
    if TRACKER_CTX.get("survey_end_date") else SURVEY_DATE_STR
)
_BASE_ID_TABLES = ["master_roughness_parent", "master_rutting_parent", "master_cracking",
                   "master_ravelling_parent", "master_pothole"]
_CHILD_PARENT_MAP = {
    "master_roughness_child_100m": "master_roughness_parent",
    "master_roughness_child_10m":  "master_roughness_parent",
    "master_rutting_child_100m":   "master_rutting_parent",
    "master_rutting_child_10m":    "master_rutting_parent",
    "master_ravelling_child_100m": "master_ravelling_parent",
    "master_ravelling_child_10m":  "master_ravelling_parent",
}
_TOP_ID_RE = f"^D/{DISPLAY_DEFECT_DATE_STR}/[NDRO]/\\d+$"

# Max existing top-level counter for the same date across other UPCs (all DLM tables)
_max_seq = 0
for _t in list(_BASE_ID_TABLES) + list(_CHILD_PARENT_MAP):
    _full = f"{DLM_KPI_NS}.{_t}"
    if spark.catalog.tableExists(_full):
        _m = (spark.table(_full)
              .filter((F.col("upc_code") != UPC) & F.col("display_defect_id").rlike(_TOP_ID_RE))
              .select(F.max(F.element_at(F.split("display_defect_id", "/"), 4).cast("int"))).collect()[0][0])
        if _m: _max_seq = max(_max_seq, _m)
print(f"Display ID date: {DISPLAY_DEFECT_DATE_STR} | existing max counter (other UPCs): {_max_seq}")
_running_seq = _max_seq

_order_cols = [F.col("threshold_name").asc_nulls_last(), F.col("lane").asc_nulls_last(),
               F.col("start_chainage").asc_nulls_last(), F.col("end_chainage").asc_nulls_last(),
               F.col("record_id").asc_nulls_last(), F.col("id").asc_nulls_last()]

def _assign_top_ids(sdf, mask_col):
    """Number rows where mask_col is true with the running counter: D/{date}/N/{%05d}."""
    global _running_seq
    _n = sdf.filter(F.col(mask_col)).count()
    if _n == 0: return sdf
    _w = Window.partitionBy(F.col(mask_col)).orderBy(*_order_cols)
    sdf = sdf.withColumn("_seq", F.row_number().over(_w)).withColumn(
        "display_defect_id",
        F.when(F.col(mask_col), F.concat(F.lit(f"D/{DISPLAY_DEFECT_DATE_STR}/{_SRC_LETTER}/"),
                                          F.format_string("%05d", F.col("_seq") + F.lit(_running_seq))))
         .otherwise(F.col("display_defect_id"))).drop("_seq")
    _running_seq += _n
    return sdf

# Pass 1: parent / standalone tables
for _t in _BASE_ID_TABLES:
    if _t not in OUTPUT_DATAFRAMES: continue
    _sdf = OUTPUT_DATAFRAMES[_t].withColumn("display_defect_id", F.lit(None).cast("string"))
    _sdf = _sdf.withColumn("_is_top", F.col("breach_flag") == 1)
    _sdf = _assign_top_ids(_sdf, "_is_top").drop("_is_top")
    OUTPUT_DATAFRAMES[_t] = _sdf
    print(f"  {_t}: parent/standalone IDs -> counter now {_running_seq}")

# Pass 2: children inherit parent ID + /0001 suffix; orphan breaches get their own counter
for _c, _p in _CHILD_PARENT_MAP.items():
    if _c not in OUTPUT_DATAFRAMES: continue
    _lk = (OUTPUT_DATAFRAMES[_p].filter(F.col("display_defect_id").isNotNull())
           .select(F.col("parent_id").alias("_pj"), F.col("threshold_name").alias("_pt"),
                   F.col("display_defect_id").alias("_pdid"))
           .dropDuplicates(["_pj", "_pt"]))
    _sdf = OUTPUT_DATAFRAMES[_c].withColumn("display_defect_id", F.lit(None).cast("string"))
    _sdf = _sdf.join(_lk, (_sdf["parent_id"] == _lk["_pj"]) & (_sdf["threshold_name"] == _lk["_pt"]), "left")
    _sdf = _sdf.withColumn("_has_p", (F.col("breach_flag") == 1) & F.col("_pdid").isNotNull())
    _cw = Window.partitionBy("parent_id", "threshold_name", "_has_p").orderBy(
        F.col("start_chainage").asc_nulls_last(), F.col("end_chainage").asc_nulls_last(),
        F.col("record_id").asc_nulls_last(), F.col("id").asc_nulls_last())
    _sdf = _sdf.withColumn("display_defect_id", F.when(
        F.col("_has_p"), F.concat(F.col("_pdid"), F.lit("/"), F.format_string("%04d", F.row_number().over(_cw)))))
    _sdf = _sdf.withColumn("_orphan", (F.col("breach_flag") == 1) & F.col("_pdid").isNull())
    _n_child = _sdf.filter("_has_p").count(); _n_orph = _sdf.filter("_orphan").count()
    _sdf = _assign_top_ids(_sdf, "_orphan")
    OUTPUT_DATAFRAMES[_c] = _sdf.drop("_pj", "_pt", "_pdid", "_has_p", "_orphan")
    print(f"  {_c}: child IDs={_n_child:,} | orphan (own counter)={_n_orph:,}")

# Print counts
for tbl_name in sorted(OUTPUT_DATAFRAMES):
    cnt = OUTPUT_DATAFRAMES[tbl_name].count()
    breach_cnt = OUTPUT_DATAFRAMES[tbl_name].filter(F.col('breach_flag') == 1).count()
    print(f"  {tbl_name:40s} {cnt:>6,} rows ({breach_cnt:,} breach)")

Building OUTPUT_DATAFRAMES...
  Cracking IRC rows given new defect IDs (IRC breach, CA not): 0
Display ID date: 210426 | existing max counter (other UPCs): 0
  master_roughness_parent: parent/standalone IDs -> counter now 199
  master_rutting_parent: parent/standalone IDs -> counter now 199
  master_cracking: parent/standalone IDs -> counter now 4215
  master_ravelling_parent: parent/standalone IDs -> counter now 4215
  master_pothole: parent/standalone IDs -> counter now 4425
  master_roughness_child_100m: child IDs=1,250 | orphan (own counter)=750
  master_roughness_child_10m: child IDs=10,418 | orphan (own counter)=10,557
  master_rutting_child_100m: child IDs=0 | orphan (own counter)=14
  master_rutting_child_10m: child IDs=0 | orphan (own counter)=502
  master_ravelling_child_100m: child IDs=0 | orphan (own counter)=0
  master_ravelling_child_10m: child IDs=0 | orphan (own counter)=0
  master_cracking                          60,876 rows (4,016 breach)
  master_pothole            

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# PCI — Weighted Defect Score at 10m / 100m / 1km
# Ported from Clone Update_5 Cell 15
#
# Formula:  PCI = 100 − Weighted_Defect_Score
# Weights:  Crack=0.12, Ravel=0.10, Pothole=0.16, Patch=0.08, Rut=0.14, Rough=0.40
# Each index normalized 0–100 via max-observed-value scaling.
# Roll up to 100m/1km by averaging 10m PCI scores.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Weights (sum to 1.0) ──────────────────────────────────────────────────────
W_CRACK, W_RAVEL, W_POTHOLE, W_PATCH, W_RUT, W_ROUGH = 0.12, 0.10, 0.16, 0.08, 0.14, 0.40

# ── Step 1: Load Patching Data from Backup Lane Folders ──────────────────────
def _load_patching(backup_base):
    frames = []
    lanes = discover_lane_folders(backup_base)
    for lane in lanes:
        lane_path = os.path.join(backup_base, f"Lane={lane}")
        matches = [f for f in sorted(os.listdir(lane_path))
                   if f.endswith('.parquet') and 'patch' in f.lower()]
        for pf in matches:
            part = pd.read_parquet(os.path.join(lane_path, pf))
            # Find chainage and area columns
            cols_lower = {c.lower(): c for c in part.columns}
            ch_col = cols_lower.get('chainage') or cols_lower.get('start_chainage')
            area_col = cols_lower.get('area') or cols_lower.get('area_sq_m') or cols_lower.get('total_area')
            if ch_col and area_col:
                df = part[[ch_col, area_col]].copy()
                df.columns = ['Chainage', 'Area']
                df['Lane'] = lane
                df['Chainage'] = pd.to_numeric(df['Chainage'], errors='coerce')
                df['Area'] = pd.to_numeric(df['Area'], errors='coerce').fillna(0.0)
                frames.append(df)
                print(f"  Patching: Lane={lane}/{pf} ({len(df)} rows)")
    if not frames:
        print("  WARNING: No patching parquets found.")
        return pd.DataFrame(columns=['Lane', 'Start_Chainage', 'patching_area'])
    all_patch = pd.concat(frames, ignore_index=True).dropna(subset=['Chainage'])
    all_patch['Start_Chainage'] = (all_patch['Chainage'] // CHILD_10M * CHILD_10M).astype(int)
    return all_patch.groupby(['Lane', 'Start_Chainage']).agg(patching_area=('Area', 'sum')).reset_index()

patching_10m = _load_patching(BACKUP_BASE)
print(f"  Patching 10m records: {len(patching_10m):,}")

# ── Step 2: Build 10m Chainage Spine ─────────────────────────────────────────
# Scan all OUTPUT_DATAFRAMES + patching to get per-lane min/max chainage
_spine_ranges = []
for _tbl in ['master_cracking', 'master_roughness_child_10m', 'master_rutting_child_10m',
             'master_ravelling_child_10m', 'master_pothole']:
    if _tbl not in OUTPUT_DATAFRAMES: continue
    _r = (OUTPUT_DATAFRAMES[_tbl]
          .select(F.col('lane').alias('Lane'), F.col('start_chainage').cast('long').alias('ch'))
          .groupBy('Lane').agg(F.min('ch').alias('min_ch'), F.max('ch').alias('max_ch'))
          .toPandas())
    if not _r.empty: _spine_ranges.append(_r)
if not patching_10m.empty:
    _spine_ranges.append(patching_10m.groupby('Lane').agg(
        min_ch=('Start_Chainage', 'min'), max_ch=('Start_Chainage', 'max')).reset_index())

_lane_bounds = pd.concat(_spine_ranges, ignore_index=True).groupby('Lane').agg(
    min_ch=('min_ch', 'min'), max_ch=('max_ch', 'max')).reset_index()

_spine_rows = []
for _, _lb in _lane_bounds.iterrows():
    lo = int((_lb['min_ch'] // CHILD_10M) * CHILD_10M)
    hi = int((_lb['max_ch'] // CHILD_10M) * CHILD_10M)
    for ch in range(lo, hi + CHILD_10M, CHILD_10M):
        _spine_rows.append({'Lane': _lb['Lane'], 'Start_Chainage': ch, 'End_Chainage': ch + CHILD_10M})
pci_spine = pd.DataFrame(_spine_rows)
print(f"PCI spine: {len(pci_spine):,} segments, lanes {sorted(pci_spine['Lane'].unique().tolist())}")

# ── Step 3: Read Metrics from OUTPUT_DATAFRAMES ──────────────────────────────
# Filter to 'As per CA' threshold only to avoid double-counting dual records
def _read_metric(table_name, value_col, alias, agg_func='max'):
    if table_name not in OUTPUT_DATAFRAMES:
        return pd.DataFrame(columns=['Lane', 'Start_Chainage', alias])
    sdf = OUTPUT_DATAFRAMES[table_name]
    if 'threshold_name' in sdf.columns:
        sdf = sdf.filter(F.col('threshold_name') == 'As per CA')
    pdf = sdf.select(
        F.col('lane').alias('Lane'),
        F.col('start_chainage').cast('long').alias('raw_ch'),
        F.col(value_col).alias('val')
    ).toPandas()
    if pdf.empty:
        return pd.DataFrame(columns=['Lane', 'Start_Chainage', alias])
    pdf['Start_Chainage'] = (pdf['raw_ch'] // CHILD_10M) * CHILD_10M
    pdf['val'] = pd.to_numeric(pdf['val'], errors='coerce').fillna(0.0)
    return pdf.groupby(['Lane', 'Start_Chainage']).agg(val=('val', agg_func)).reset_index().rename(columns={'val': alias})

cracking_metric  = _read_metric('master_cracking',            'metric_value', 'cracking_pct',   'max')
ravelling_metric = _read_metric('master_ravelling_child_10m', 'area_sqm',     'ravelling_area',  'sum')
pothole_metric   = _read_metric('master_pothole',             'area_sqm',     'pothole_area',    'sum')
rutting_metric   = _read_metric('master_rutting_child_10m',   'metric_value', 'rutting_mm',      'max')
roughness_metric = _read_metric('master_roughness_child_10m', 'iri_value',    'roughness_iri',   'max')

for label, pdf in [('Cracking pct', cracking_metric), ('Ravelling area', ravelling_metric),
                   ('Pothole area', pothole_metric), ('Rutting mm', rutting_metric),
                   ('Roughness IRI', roughness_metric), ('Patching area', patching_10m)]:
    print(f"  {label:16s}: {len(pdf):>6,} rows")

# ── Step 4: Merge Metrics onto Spine ─────────────────────────────────────────
pci_df = pci_spine.copy()
for mdf in [cracking_metric, ravelling_metric, pothole_metric,
            rutting_metric, roughness_metric, patching_10m]:
    pci_df = pci_df.merge(mdf, on=['Lane', 'Start_Chainage'], how='left')

raw_cols = ['cracking_pct', 'ravelling_area', 'pothole_area', 'rutting_mm', 'roughness_iri', 'patching_area']
for col in raw_cols:
    pci_df[col] = pci_df[col].fillna(0.0)

# ── Step 5: Normalize to 0–100 Index (max-observed scaling) ──────────────────
max_vals = {c: pci_df[c].max() for c in raw_cols}
print(f"\nNormalization ceilings:")
for c, mx in max_vals.items(): print(f"  {c:20s}: {mx:.4f}")

index_map = {'cracking_pct': 'crack_index', 'ravelling_area': 'ravel_index', 'pothole_area': 'pothole_index',
             'patching_area': 'patch_index', 'rutting_mm': 'rut_index', 'roughness_iri': 'roughness_index'}
for raw_col, idx_col in index_map.items():
    mx = max_vals[raw_col]
    pci_df[idx_col] = (pci_df[raw_col] / mx * 100.0).clip(0, 100) if mx > 0 else 0.0

# ── Step 6: Weighted Defect Score & PCI ──────────────────────────────────────
pci_df['weighted_defect_score'] = (
    pci_df['crack_index']     * W_CRACK  +
    pci_df['ravel_index']     * W_RAVEL  +
    pci_df['pothole_index']   * W_POTHOLE +
    pci_df['patch_index']     * W_PATCH  +
    pci_df['rut_index']       * W_RUT    +
    pci_df['roughness_index'] * W_ROUGH
).round(4)
pci_df['pci_10m'] = (100.0 - pci_df['weighted_defect_score']).clip(0, 100).round(4)

print(f"\nPCI 10m: {len(pci_df):,} segments")
print(f"  PCI range: {pci_df['pci_10m'].min():.2f} – {pci_df['pci_10m'].max():.2f}")
print(f"  Mean PCI:  {pci_df['pci_10m'].mean():.2f}")
print(f"  Segments with defects (PCI < 100): {(pci_df['pci_10m'] < 100).sum():,}")

# ── Step 7: Roll Up to 100m and 1km ─────────────────────────────────────────
pci_df['_ch_100m'] = (pci_df['Start_Chainage'] // 100) * 100
pci_100m = pci_df.groupby(['Lane', '_ch_100m']).agg(pci_score=('pci_10m', 'mean')).reset_index()
pci_100m.rename(columns={'_ch_100m': 'Start_Chainage'}, inplace=True)
pci_100m['pci_score'] = pci_100m['pci_score'].round(4)
print(f"PCI 100m: {len(pci_100m):,} segments, mean={pci_100m['pci_score'].mean():.2f}")

pci_df['_ch_1km'] = (pci_df['Start_Chainage'] // 1000) * 1000
pci_1km = pci_df.groupby(['Lane', '_ch_1km']).agg(pci_score=('pci_10m', 'mean')).reset_index()
pci_1km.rename(columns={'_ch_1km': 'Start_Chainage'}, inplace=True)
pci_1km['pci_score'] = pci_1km['pci_score'].round(4)
print(f"PCI 1km:  {len(pci_1km):,} segments, mean={pci_1km['pci_score'].mean():.2f}")

# ── Step 8: Build Spark Lookups + fill_pci_column() ──────────────────────────
pci_lookup_10m  = pci_df[['Lane', 'Start_Chainage', 'pci_10m']].rename(columns={'pci_10m': 'pci_score'})
pci_lookup_100m = pci_100m[['Lane', 'Start_Chainage', 'pci_score']].copy()
pci_lookup_1km  = pci_1km[['Lane', 'Start_Chainage', 'pci_score']].copy()

_pci_sdf_10m  = spark.createDataFrame(pci_lookup_10m.rename(columns={'Lane':'_pci_lane','Start_Chainage':'_pci_ch','pci_score':'_pci_val'}))
_pci_sdf_100m = spark.createDataFrame(pci_lookup_100m.rename(columns={'Lane':'_pci_lane','Start_Chainage':'_pci_ch','pci_score':'_pci_val'}))
_pci_sdf_1km  = spark.createDataFrame(pci_lookup_1km.rename(columns={'Lane':'_pci_lane','Start_Chainage':'_pci_ch','pci_score':'_pci_val'}))
PCI_LOOKUPS = {'10m': _pci_sdf_10m, '100m': _pci_sdf_100m, '1km': _pci_sdf_1km}

def fill_pci_column(sdf, level='10m'):
    lookup = PCI_LOOKUPS[level]
    grid = {'10m': CHILD_10M, '100m': 100, '1km': 1000}[level]
    sdf2 = sdf.withColumn('_join_ch', (F.col('start_chainage').cast('long') / grid).cast('long') * grid)
    joined = sdf2.join(lookup,
        (sdf2['lane'] == lookup['_pci_lane']) & (sdf2['_join_ch'] == lookup['_pci_ch'].cast('long')), 'left')
    return joined.withColumn('pci', F.coalesce(F.col('_pci_val'), F.lit(None)).cast('double')) \
                 .drop('_pci_lane', '_pci_ch', '_pci_val', '_join_ch')

# ── Step 9: Backfill PCI into OUTPUT_DATAFRAMES ─────────────────────────────
updated = []
for tbl_name, sdf in OUTPUT_DATAFRAMES.items():
    if 'pci' not in sdf.columns: continue
    lvl = '1km' if 'parent' in tbl_name else ('100m' if '100m' in tbl_name else '10m')
    OUTPUT_DATAFRAMES[tbl_name] = fill_pci_column(sdf, level=lvl)
    updated.append(f"{tbl_name} ({lvl})")
print(f"\nBackfilled PCI in: {', '.join(updated)}")

# ── Quick Verify ─────────────────────────────────────────────────────────────
for tbl in ['master_roughness_child_10m', 'master_cracking', 'master_pothole']:
    sdf = OUTPUT_DATAFRAMES[tbl]
    pci_filled = sdf.filter(F.col('pci').isNotNull()).count()
    total = sdf.count()
    pct = pci_filled/total*100 if total > 0 else 0
    print(f"  {tbl}: {pci_filled:,}/{total:,} pci filled ({pct:.1f}%)")

  Patching: Lane=L1/ZB_P2_C1_N_07024_01001_GJ_Patching.parquet (7625 rows)
  Patching: Lane=L2/ZB_P2_C1_N_07024_01001_GJ_Patching.parquet (7625 rows)
  Patching: Lane=R1/ZB_P2_C1_N_07024_01001_GJ_Patching.parquet (7625 rows)
  Patching: Lane=R2/ZB_P2_C1_N_07024_01001_GJ_Patching.parquet (7625 rows)
  Patching 10m records: 30,500
PCI spine: 30,500 segments, lanes ['L1', 'L2', 'R1', 'R2']
  Cracking pct    : 30,438 rows
  Ravelling area  : 30,438 rows
  Pothole area    :    167 rows
  Rutting mm      : 30,438 rows
  Roughness IRI   : 30,438 rows
  Patching area   : 30,500 rows

Normalization ceilings:
  cracking_pct        : 100.0000
  ravelling_area      : 0.0000
  pothole_area        : 0.1040
  rutting_mm          : 18.1035
  roughness_iri       : 15.4230
  patching_area       : 15.3520

PCI 10m: 30,500 segments
  PCI range: 49.22 – 100.00
  Mean PCI:  87.34
  Segments with defects (PCI < 100): 30,410
PCI 100m: 3,052 segments, mean=87.34
PCI 1km:  308 segments, mean=87.33

Backfilled P

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADD road_condition COLUMN (dynamic from D3 per project)
#
# IRC standard → D3 visual condition (Good/Fair/Poor categories from D3 deliverable)
# CA standard  → Simple binary: "Good" (no breach) / "Bad" (breach)
#
# D3 data is loaded per-UPC from D3_BASE, so categories are project-specific.
# ═══════════════════════════════════════════════════════════════════════════════

_defect_type_map = {
    'cracking': 'Cracking', 'roughness': 'Roughness',
    'rutting': 'Rutting', 'pothole': 'Pothole', 'ravelling': 'Ravelling',
}

# Build combined D3 lookup as broadcast Spark DataFrame
_d3_frames = []
for _defect, _df_d3 in D3_CONDITIONS.items():
    _tmp = _df_d3.copy()
    _tmp['defect_type'] = _defect
    _d3_frames.append(_tmp[['defect_type', 'bin_100m', 'Lane', 'Condition']])

if _d3_frames:
    _d3_all = pd.concat(_d3_frames, ignore_index=True)
    d3_lookup_sdf = spark.createDataFrame(_d3_all).select(
        F.col('defect_type').alias('_d3_defect'),
        F.col('bin_100m').cast('long').alias('_d3_bin'),
        F.col('Lane').alias('_d3_lane'),
        F.col('Condition').alias('_d3_condition')
    )
    print(f"D3 lookup built: {d3_lookup_sdf.count()} rows")
    print(f"Defects covered: {sorted(D3_CONDITIONS.keys())}")
else:
    d3_lookup_sdf = None
    print("WARNING: No D3 data — road_condition will be CA-only (Good/Bad)")

# Map road_condition onto each OUTPUT_DATAFRAME
_updated = 0
for tbl, sdf in OUTPUT_DATAFRAMES.items():
    # Determine defect type from table name
    _defect_type = next((v for k, v in _defect_type_map.items() if k in tbl.lower()), None)
    if _defect_type is None:
        continue

    # Compute 100m bin for chainage-based D3 join
    sdf_bin = sdf.withColumn('_bin_100m', (F.floor(F.col('start_chainage') / 100) * 100).cast('long'))

    # Left join with D3 for IRC condition lookup
    if d3_lookup_sdf is not None and _defect_type in D3_CONDITIONS:
        sdf_joined = sdf_bin.join(
            F.broadcast(d3_lookup_sdf),
            (sdf_bin['_bin_100m'] == F.col('_d3_bin')) &
            (sdf_bin['lane'] == F.col('_d3_lane')) &
            (F.lit(_defect_type) == F.col('_d3_defect')),
            'left'
        )
    else:
        sdf_joined = sdf_bin.withColumn('_d3_condition', F.lit(None).cast('string'))

    # Apply road_condition using threshold_name (not 'standard' which is NULL):
    #   CA  → "Good" / "Bad" (simple breach binary)
    #   IRC → D3 visual condition (project-specific categories from D3)
    _std_col = F.lower(F.coalesce(F.col('threshold_name'), F.col('standard')))
    sdf_final = sdf_joined.withColumn(
        'road_condition',
        F.when(_std_col.contains('ca'),
               F.when(F.col('breach_flag') == 1, F.lit('Bad')).otherwise(F.lit('Good')))
        .when(_std_col.contains('irc'),
               F.coalesce(F.col('_d3_condition'), F.lit(None)))
        .otherwise(
            # Fallback: if neither CA nor IRC in threshold_name, use breach_flag
            F.when(F.col('breach_flag') == 1, F.lit('Bad')).otherwise(F.lit('Good')))
    ).drop('_bin_100m', '_d3_condition', '_d3_bin', '_d3_lane', '_d3_defect')

    OUTPUT_DATAFRAMES[tbl] = sdf_final
    _updated += 1

print(f"\nroad_condition added to {_updated} tables")

# Show samples
for _tbl in ['master_roughness_parent', 'master_roughness_child_100m', 'master_cracking']:
    if _tbl in OUTPUT_DATAFRAMES:
        _sdf = OUTPUT_DATAFRAMES[_tbl]
        print(f"\n{_tbl}:")
        _sdf.select('threshold_name', 'breach_flag', 'road_condition', 'lane', 'start_chainage').show(5, truncate=False)

D3 lookup built: 12174 rows
Defects covered: ['Cracking', 'Pothole', 'Roughness', 'Rutting']

road_condition added to 11 tables

master_roughness_parent:
+--------------+-----------+--------------+----+--------------+
|threshold_name|breach_flag|road_condition|lane|start_chainage|
+--------------+-----------+--------------+----+--------------+
|As per CA     |1          |Bad           |L1  |0.0           |
|As per CA     |0          |Good          |L1  |1000.0        |
|As per CA     |0          |Good          |L1  |2000.0        |
|As per CA     |0          |Good          |L1  |3000.0        |
|As per CA     |0          |Good          |L1  |4000.0        |
+--------------+-----------+--------------+----+--------------+
only showing top 5 rows

master_roughness_child_100m:
+--------------+-----------+--------------+----+--------------+
|threshold_name|breach_flag|road_condition|lane|start_chainage|
+--------------+-----------+--------------+----+--------------+
|As per CA     |1       

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# STANDARDIZE NH NUMBER + FILL rating_ca AND rating_irc INDEPENDENTLY
#
# nh_number  → always "NH-<number><suffix>"  (NH0027 → NH-27, NH151A → NH-151A)
# rating_ca  → CA condition label (from CA-threshold condition), fallback Good/Bad on CA breach
# rating_irc → IRC condition (D3 visual condition), fallback Good/Poor on IRC breach
# Both ratings are written on BOTH the CA row and its IRC twin row, whenever the
# underlying data (metric / condition) is present. Twin rows are paired by id
# (IRC id = CA id with '_irc_' inserted).
# ═══════════════════════════════════════════════════════════════════════════════
from pyspark.sql import Window as _W

# CA rating computed from each row's OWN metric (correct at 10m / 100m / 1000m),
# Native Spark port of crack/roughness/rutting/ravelling_condition_* (same D2 breakpoints + labels).
def _ca_rating_expr(defect):
    v = F.col('metric_value').cast('double')
    if defect == 'cracking':
        t = CA_CRACK_PCT; b2, b3, b4 = round(t * 2), round(t * 4), round(t * 6)
        return (F.when(v.isNull(), None).when(v < t, f'Very Good (<{t:.0f}%)')
                 .when(v <= b2, f'Good ({t:.0f}-{b2}%)').when(v <= b3, f'Fair ({b2}-{b3}%)')
                 .when(v <= b4, f'Poor ({b3}-{b4}%)').otherwise(f'Very Poor (>{b4}%)'))
    if defect == 'roughness':
        t = CA_BI_THRESHOLD; fl = round(t * 0.8)
        p = F.coalesce(F.col('roughness_bi').cast('double'), F.col('iri_value').cast('double') * 1000)
        return (F.when(p.isNull(), None).when(p < fl, f'Good (<{fl}mm/km)')
                 .when(p <= t, f'Fair ({fl}-{t:.0f}mm/km)').otherwise(f'Poor (>{t:.0f}mm/km)'))
    if defect == 'rutting':
        t = CA_RUT_THRESHOLD; fl = round(t * 0.5)
        return (F.when(v.isNull(), None).when(v < fl, f'Good (<{fl}mm)')
                 .when(v <= t, f'Fair ({fl}-{t:.0f}mm)').otherwise(f'Poor (>{t:.0f}mm)'))
    t = RAVELLING_RATING_THRESHOLD; h = round(t * 0.5)
    return (F.when(v.isNull(), None).when(v == 0, 'Very Good (0)').when(v <= h, f'Good (1-{h}%)')
             .when(v < t, f'Fair ({h}-{t:.0f}%)').otherwise(f'Poor (>={t:.0f}%)'))

_has_data = F.col('metric_value').isNotNull()
_std = F.lower(F.coalesce(F.col('threshold_name'), F.lit('')))

for tbl in list(OUTPUT_DATAFRAMES.keys()):
    sdf = OUTPUT_DATAFRAMES[tbl].withColumn('nh_number', standardize_nh_col('nh_number'))

    if tbl == 'master_pothole':
        # Pothole severity is standard-independent → same rating for CA and IRC
        sdf = (sdf.withColumn('rating_ca',  F.coalesce('rating_ca',  'severity'))
                  .withColumn('rating_irc', F.coalesce('rating_irc', 'severity')))
    else:
        _defect = next(d for d in ['cracking', 'roughness', 'rutting', 'ravelling'] if d in tbl)
        _ca_own = F.when(_std.contains('ca'), F.coalesce(
            _ca_rating_expr(_defect), F.col('rating_ca'), F.col('severity'),
            F.when(_has_data, F.when(F.col('breach_flag') == 1, 'Bad').otherwise('Good'))))
        _irc_own = F.when(_std.contains('irc'), F.coalesce(
            F.col('road_condition'),
            F.when(_has_data, F.when(F.col('breach_flag') == 1, 'Poor').otherwise('Good'))))
        _w = _W.partitionBy(F.regexp_replace('id', '_irc_', '_'))
        sdf = (sdf.withColumn('_ca_own', _ca_own).withColumn('_irc_own', _irc_own)
                  .withColumn('rating_ca',  F.max('_ca_own').over(_w))
                  .withColumn('rating_irc', F.max('_irc_own').over(_w))
                  .drop('_ca_own', '_irc_own'))
    OUTPUT_DATAFRAMES[tbl] = sdf

# ── Verification ────────────────────────────────────────────────────────────
print(f"UPC {UPC}\n")
print(f"{'Table':<32} {'Std':<11} {'Rows':>7} {'rating_ca':>10} {'rating_irc':>11}  nh_number")
print('-' * 95)
for tbl in sorted(OUTPUT_DATAFRAMES):
    stats = (OUTPUT_DATAFRAMES[tbl].groupBy('threshold_name')
             .agg(F.count('*').alias('n'), F.count('rating_ca').alias('ca'),
                  F.count('rating_irc').alias('irc'),
                  F.concat_ws(',', F.collect_set('nh_number')).alias('nh'))
             .orderBy('threshold_name').collect())
    for r in stats:
        print(f"{tbl:<32} {str(r['threshold_name']):<11} {r['n']:>7,} {r['ca']:>10,} {r['irc']:>11,}  {r['nh']}")

print("\nSample (cracking + roughness_child_100m):")
for _t in ['master_cracking', 'master_roughness_child_100m']:
    (OUTPUT_DATAFRAMES[_t].select('lane', 'start_chainage', 'threshold_name', 'metric_value',
                                   'breach_flag', 'rating_ca', 'rating_irc', 'road_condition', 'nh_number')
        .orderBy('lane', 'start_chainage', 'threshold_name').show(4, truncate=False))

UPC N_07024_01001_GJ

Table                            Std            Rows  rating_ca  rating_irc  nh_number
-----------------------------------------------------------------------------------------------
master_cracking                  As per CA    30,438     30,438      30,438  NH-151A
master_cracking                  As per IRC   30,438     30,438      30,438  NH-151A
master_pothole                   As per CA       210        210         210  NH-151A
master_ravelling_child_100m      As per CA     3,047      3,047       3,047  NH-151A
master_ravelling_child_100m      As per IRC    3,047      3,047       3,047  NH-151A
master_ravelling_child_10m       As per CA    30,438     30,438      30,438  NH-151A
master_ravelling_child_10m       As per IRC   30,438     30,438      30,438  NH-151A
master_ravelling_parent          As per CA       308        308         308  NH-151A
master_ravelling_parent          As per IRC      308        308         308  NH-151A
master_roughness_child_100m   

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# DUAL SCHEMA WRITE
# dlm_kpi:      ALL data (breach_flag=0 and breach_flag=1)
# dlm_postgres:  ONLY breach_flag=1 data
# ═══════════════════════════════════════════════════════════════════════════════

# Create schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DLM_KPI_NS}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DLM_POSTGRES_NS}")

OUTPUT_TABLES = sorted(TABLE_SCHEMAS.keys())

def write_to_schema(namespace, write_df, table_name, filter_breach=False):
    full_name = f"{namespace}.{table_name}"
    if filter_breach:
        write_df = write_df.filter(F.col("breach_flag") == 1)
    table_exists = spark.catalog.tableExists(full_name)
    if table_exists:
        (write_df.write.format("delta").mode("overwrite")
         .option("replaceWhere", f"upc_code = '{UPC}' AND phase = '{PHASE}' AND cycle = '{CYCLE}'")
         .option("mergeSchema", "true")
         .saveAsTable(full_name))
    else:
        (write_df.write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(full_name))
    return write_df.count()

print("Writing to dual schemas...\n")
print(f"{'Table':<42s} {'dlm_kpi':>10s} {'dlm_postgres':>14s}")
print("-" * 68)

for table_name in OUTPUT_TABLES:
    write_df = OUTPUT_DATAFRAMES.get(table_name, spark.createDataFrame([], TABLE_SCHEMAS[table_name]))

    # dlm_kpi: ALL data
    kpi_rows = write_to_schema(DLM_KPI_NS, write_df, table_name, filter_breach=False)

    # dlm_postgres: breach_flag=1 ONLY
    pg_rows = write_to_schema(DLM_POSTGRES_NS, write_df, table_name, filter_breach=True)

    print(f"  {table_name:<40s} {kpi_rows:>10,} {pg_rows:>14,}")

print(f"\nDone. All {len(OUTPUT_TABLES)} tables written to both schemas.")

Writing to dual schemas...

Table                                         dlm_kpi   dlm_postgres
--------------------------------------------------------------------
  master_cracking                              60,876          4,016
  master_pothole                                  210            210
  master_ravelling_child_100m                   6,094              0
  master_ravelling_child_10m                   60,876              0
  master_ravelling_parent                         616              0
  master_roughness_child_100m                   6,094          2,000
  master_roughness_child_10m                   60,876         20,975
  master_roughness_parent                         616            199
  master_rutting_child_100m                     6,094             14
  master_rutting_child_10m                     60,876            502
  master_rutting_parent                           616              0

Done. All 11 tables written to both schemas.


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFY: every breach row has a display_defect_id (CA and IRC), none on non-breach
#         rows, no duplicate IDs per standard, and measures carry calc inputs
# ═══════════════════════════════════════════════════════════════════════════════
_V_TABLES = ["master_cracking", "master_pothole",
             "master_ravelling_child_10m", "master_ravelling_child_100m", "master_ravelling_parent",
             "master_roughness_child_10m", "master_roughness_child_100m", "master_roughness_parent",
             "master_rutting_child_10m", "master_rutting_child_100m", "master_rutting_parent"]
_rows = []
for _t in _V_TABLES:
    _d = spark.table(f"{DLM_KPI_NS}.{_t}").filter(F.col("upc_code") == UPC)
    _empty = F.col("display_defect_id").isNull() | (F.trim("display_defect_id") == "")
    _r = _d.agg(
        F.sum((F.col("breach_flag") == 1).cast("int")).alias("breach_rows"),
        F.sum(((F.col("breach_flag") == 1) & _empty).cast("int")).alias("breach_missing_id"),
        F.sum(((F.col("threshold_name") == "As per IRC") & (F.col("breach_flag") == 1) & _empty).cast("int")).alias("irc_breach_missing_id"),
        F.sum(((F.col("breach_flag") == 0) & ~_empty).cast("int")).alias("nonbreach_with_id"),
    ).collect()[0]
    _dup = _d.filter(~_empty).groupBy("display_defect_id", "threshold_name").count().filter("count > 1").count()
    _rows.append((_t, _r.breach_rows or 0, _r.breach_missing_id or 0, _r.irc_breach_missing_id or 0,
                  _r.nonbreach_with_id or 0, _dup))
_vdf = pd.DataFrame(_rows, columns=["table", "breach_rows", "breach_missing_id", "irc_breach_missing_id",
                                    "nonbreach_with_id", "duplicate_ids"])
_vdf["status"] = np.where((_vdf.breach_missing_id == 0) & (_vdf.nonbreach_with_id == 0) & (_vdf.duplicate_ids == 0), "OK", "CHECK")
print(f"UPC {UPC}")
display(_vdf)

# Sample measures (first L1 CA row per table)
_samples = []
for _t in ["master_roughness_child_10m", "master_roughness_parent", "master_rutting_child_10m",
           "master_rutting_child_100m", "master_rutting_parent", "master_ravelling_child_100m", "master_cracking"]:
    _s = (spark.table(f"{DLM_KPI_NS}.{_t}")
          .filter((F.col("upc_code") == UPC) & (F.col("threshold_name") == "As per CA"))
          .orderBy("lane", "start_chainage").select("lane", "start_chainage", "metric_value", "measures").first())
    if _s: _samples.append((_t, _s.lane, _s.start_chainage, _s.metric_value, _s.measures))
display(pd.DataFrame(_samples, columns=["table", "lane", "start_chainage", "metric_value", "measures"]))

UPC N_07024_01001_GJ


table,breach_rows,breach_missing_id,irc_breach_missing_id,nonbreach_with_id,duplicate_ids,status
master_cracking,4016,0,0,0,0,OK
master_pothole,210,0,0,0,0,OK
master_ravelling_child_10m,0,0,0,0,0,OK
master_ravelling_child_100m,0,0,0,0,0,OK
master_ravelling_parent,0,0,0,0,0,OK
master_roughness_child_10m,20975,0,0,0,0,OK
master_roughness_child_100m,2000,0,0,0,0,OK
master_roughness_parent,199,0,0,0,0,OK
master_rutting_child_10m,502,0,0,0,0,OK
master_rutting_child_100m,14,0,0,0,0,OK


table,lane,start_chainage,metric_value,measures
master_roughness_child_10m,L1,0.0,11819.78512345368,"{""roughness_bi"": 11819.78512345368, ""lwp_iri"": 5.066, ""rwp_iri"": 22.342, ""lane_iri"": 13.703999999999999, ""segments"": 1, ""calculation"": ""Lane IRI = (LWP IRI + RWP IRI)/2; BI = 630 x IRI^1.12""}"
master_roughness_parent,L1,0.0,3210.867211802513,"{""roughness_bi"": 3210.867211802513, ""lwp_iri"": 4.0479, ""rwp_iri"": 4.51324, ""lane_iri"": 4.28057, ""segments"": 100, ""calculation"": ""LWP and RWP IRI averaged over segment; Lane IRI = (avg LWP + avg RWP)/2; BI = 630 x IRI^1.12""}"
master_rutting_child_10m,L1,0.0,16.618499999999997,"{""rutting_avg"": 16.618499999999997, ""rut_left_mm"": 18.816, ""rut_right_mm"": 14.421, ""segments"": 1, ""calculation"": ""Rut avg = (Left + Right)/2""}"
master_rutting_child_100m,L1,0.0,8.1333,"{""rutting_avg"": 8.1333, ""rut_left_mm"": 8.9208, ""rut_right_mm"": 7.3458, ""segments"": 10, ""calculation"": ""Left and Right rut averaged over segment; Rut avg = (avg Left + avg Right)/2""}"
master_rutting_parent,L1,0.0,6.187625,"{""rutting_avg"": 6.187625, ""rut_left_mm"": 9.5084, ""rut_right_mm"": 2.86685, ""segments"": 100, ""calculation"": ""Left and Right rut averaged over segment; Rut avg = (avg Left + avg Right)/2""}"
master_ravelling_child_100m,L1,0.0,9.405682857142856E-4,"{""ravelling_area"": 0.0009405682857142856, ""ravelling_pct"": 0.0009405682857142856, ""ravelling_area_sqm"": 0.0032919889999999995, ""lane_area_sqm"": 350.0, ""segments"": 10, ""calculation"": ""Ravelling % = sum(area) / (3.5 x segment length) x 100""}"
master_cracking,L1,0.0,19.925676857142857,"{""cracking_pct"": 19.925676857142857, ""crack_area_sqm"": 6.9739869, ""lane_area_sqm"": 35.0, ""crack_count"": 19.0, ""avg_corrected_width_mm"": 1478.29, ""calculation"": ""Crack area = length x corrected width (Alligator/Multiple=3500mm, width>=20mm -> 500mm); sum per 1m capped at 3.5 sqm; Cracking % = sum(10m area) / (3.5 x 10) x 100""}"


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

validation_rows = []
for schema_ns, schema_label in [(DLM_KPI_NS, "dlm_kpi"), (DLM_POSTGRES_NS, "dlm_postgres")]:
    for table_name in OUTPUT_TABLES:
        full_name = f"{schema_ns}.{table_name}"
        if not spark.catalog.tableExists(full_name): continue
        sdf = spark.table(full_name).filter(f"upc_code = '{UPC}'")
        total = sdf.count()
        breach_1 = sdf.filter(F.col('breach_flag') == 1).count()
        breach_0 = sdf.filter(F.col('breach_flag') == 0).count()
        validation_rows.append({
            "schema": schema_label, "table": table_name,
            "total_rows": total, "breach_1": breach_1, "breach_0": breach_0,
        })

val_df = pd.DataFrame(validation_rows)
print(f"Validation for UPC: {UPC}\n")
display(spark.createDataFrame(val_df))

# Verify chainage completeness for roughness 10m (should have 7,624 rows per lane)
print("\nChainage completeness check (roughness_child_10m):")
for schema_ns, label in [(DLM_KPI_NS, "dlm_kpi")]:
    full_name = f"{schema_ns}.master_roughness_child_10m"
    if spark.catalog.tableExists(full_name):
        sdf = spark.table(full_name).filter(f"upc_code = '{UPC}'")
        lane_counts = sdf.groupBy("lane").agg(
            F.count("*").alias("rows"),
            F.min("start_chainage").alias("min_ch"),
            F.max("end_chainage").alias("max_ch"),
        ).orderBy("lane")
        display(lane_counts)

print("\nValidation complete.")

Validation for UPC: N_02001_10003_GJ



schema,table,total_rows,breach_1,breach_0
dlm_kpi,master_cracking,111558,6770,104788
dlm_kpi,master_pothole,103,103,0
dlm_kpi,master_ravelling_child_100m,11172,0,11172
dlm_kpi,master_ravelling_child_10m,139446,0,139446
dlm_kpi,master_ravelling_parent,1128,0,1128
dlm_kpi,master_roughness_child_100m,11172,1598,9574
dlm_kpi,master_roughness_child_10m,139446,19664,119782
dlm_kpi,master_roughness_parent,1128,132,996
dlm_kpi,master_rutting_child_100m,11172,462,10710
dlm_kpi,master_rutting_child_10m,139446,7180,132266



Chainage completeness check (roughness_child_10m):


lane,rows,min_ch,max_ch
L1,27888,447385.0,540350.0
L2,27888,447385.0,540350.0
L3,27888,447385.0,540350.0
R1,18594,447390.0,540360.0
R2,18594,447390.0,540360.0
R3,18594,447390.0,540360.0



Validation complete.


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# BUILD master_chainage — 10m Chainage Reference Table
# One row per 10m segment per lane (chainage-level, NOT defect-level)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Step 1: Derive 10m chainage spine from RAW BACKUP data ───────────────────
# Use raw parquets (not OUTPUT_DATAFRAMES) to get the TRUE full chainage range
# per lane — avoids missing edge segments that defect processing might trim.
_ch_ranges = []
for _raw_df, _label in [
    (df_roughness_raw, 'roughness'), (df_cracking_raw, 'cracking'),
    (df_rutting_raw, 'rutting'), (df_ravelling_raw, 'ravelling')]:
    if _raw_df is None or _raw_df.empty:
        continue
    # Fuzzy-find lane and chainage columns in the raw parquet
    _cols = list(_raw_df.columns)
    _lane_c = next((c for c in _cols if c.lower() in ('lane', 'lanes')), None)
    _start_c = next((c for c in _cols if 'start' in c.lower() and 'chainage' in c.lower()), None)
    _end_c = next((c for c in _cols if 'end' in c.lower() and 'chainage' in c.lower()), None)
    if _lane_c is None or _start_c is None:
        continue
    _sub = _raw_df[[_lane_c, _start_c]].copy()
    _sub.columns = ['Lane', 'ch']
    if _end_c:
        _sub_end = _raw_df[[_lane_c, _end_c]].copy()
        _sub_end.columns = ['Lane', 'ch']
        _sub = pd.concat([_sub, _sub_end], ignore_index=True)
    _sub['ch'] = pd.to_numeric(_sub['ch'], errors='coerce')
    _sub = _sub.dropna(subset=['ch'])
    _sub['Lane'] = _sub['Lane'].astype(str).str.strip().str.upper()
    _grp = _sub.groupby('Lane').agg(min_ch=('ch', 'min'), max_ch=('ch', 'max')).reset_index()
    if not _grp.empty:
        _ch_ranges.append(_grp)

# Also include D1 road reference data (PAVEMENT_LOOKUP, LRMS_LOOKUP)
# to capture full road extent that raw defect files might not cover.
if not PAVEMENT_LOOKUP.empty:
    for direction, lane_prefix in [('Increasing', ['L1','L2']), ('Decreasing', ['R1','R2'])]:
        _pav = PAVEMENT_LOOKUP.copy()
        if 'direction' in _pav.columns:
            _dir_mask = _pav['direction'].fillna('').str.lower() == direction.lower()
            _pav_dir = _pav[_dir_mask] if _dir_mask.any() else _pav
        else:
            _pav_dir = _pav
        if _pav_dir.empty: continue
        _pmin = int(_pav_dir['start_chainage'].min())
        _pmax = int(_pav_dir['end_chainage'].max())
        for _ln in lane_prefix:
            _ch_ranges.append(pd.DataFrame({'Lane': [_ln], 'min_ch': [_pmin], 'max_ch': [_pmax]}))

_lane_bounds = pd.concat(_ch_ranges, ignore_index=True).groupby('Lane').agg(
    min_ch=('min_ch', 'min'), max_ch=('max_ch', 'max')).reset_index()

# Snap bounds to 10m grid
_lane_bounds['min_ch'] = (_lane_bounds['min_ch'] // CHILD_10M * CHILD_10M).astype(int)
_lane_bounds['max_ch'] = (_lane_bounds['max_ch'] // CHILD_10M * CHILD_10M).astype(int)

# Use PER-LANE bounds — each lane's spine covers only chainages where
# data actually exists for that lane.  No global alignment: R-lanes may
# start later than L-lanes if the survey data begins there.
print(f"Per-lane raw bounds (data-driven):")
for _, lb in _lane_bounds.iterrows():
    print(f"  {lb['Lane']}: {int(lb['min_ch']):>6d} – {int(lb['max_ch']):>6d}")

print("\nFinal lane bounds for chainage spine (per-lane, no global override):")
for _, lb in _lane_bounds.iterrows():
    segs = (lb['max_ch'] - lb['min_ch']) // CHILD_10M + 1
    print(f"  {lb['Lane']}: {int(lb['min_ch']):>6d} – {int(lb['max_ch']) + CHILD_10M:>6d}  ({segs:,} segments)")

# ── Step 2: Generate spine rows with road context (pavement_type etc.) ───────
spine_rows = []
for _, lb in _lane_bounds.iterrows():
    lane_val = lb['Lane']
    direction_val = lane_to_direction(lane_val)
    lo = int((lb['min_ch'] // CHILD_10M) * CHILD_10M)
    hi = int((lb['max_ch'] // CHILD_10M) * CHILD_10M)
    for ch in range(lo, hi + CHILD_10M, CHILD_10M):
        road_ctx = get_road_context(ch, ch + CHILD_10M, lane=lane_val, direction=direction_val)
        spine_rows.append({
            'upc': UPC,
            'upc_code': UPC,
            'source_system': source_system,
            'defect_detected_on': TRACKER_CTX.get('survey_end_date'),
            'start_chainage': float(ch),
            'end_chainage': float(ch + CHILD_10M),
            'phase': PHASE,
            'cycle': int(CYCLE),
            'zone': ZONE,
            'lane': lane_val,
            'direction': direction_val,
            'nh_number': first_non_null(road_ctx.get('nh_number')),
            'section_code': first_non_null(road_ctx.get('section_code')),
            'pavement_type': first_non_null(road_ctx.get('pavement_type')),
            'carriageway': first_non_null(road_ctx.get('carriageway')),
        })

chainage_pdf = pd.DataFrame(spine_rows)
print(f"\nmaster_chainage: {len(chainage_pdf):,} rows")
print(f"Lanes: {sorted(chainage_pdf['lane'].unique().tolist())}")
for lane in sorted(chainage_pdf['lane'].unique()):
    sub = chainage_pdf[chainage_pdf['lane'] == lane]
    print(f"  {lane}: {int(sub['start_chainage'].min()):>6d} – {int(sub['end_chainage'].max()):>6d}  ({len(sub):,} segments)")
print(f"Pavement types: {chainage_pdf['pavement_type'].value_counts().to_dict()}")

# ── Step 3: Create Spark DataFrame with typed schema ─────────────────────────
chainage_schema = T.StructType([
    T.StructField('upc',                 T.StringType(),    True),
    T.StructField('upc_code',            T.StringType(),    True),
    T.StructField('source_system',       T.StringType(),    True),
    T.StructField('defect_detected_on',  T.DateType(),      True),
    T.StructField('start_chainage',      T.DoubleType(),    True),
    T.StructField('end_chainage',        T.DoubleType(),    True),
    T.StructField('phase',               T.StringType(),    True),
    T.StructField('cycle',               T.IntegerType(),   True),
    T.StructField('zone',                T.StringType(),    True),
    T.StructField('lane',                T.StringType(),    True),
    T.StructField('direction',           T.StringType(),    True),
    T.StructField('nh_number',           T.StringType(),    True),
    T.StructField('section_code',        T.StringType(),    True),
    T.StructField('pavement_type',       T.StringType(),    True),
    T.StructField('carriageway',         T.StringType(),    True),
])

chainage_sdf = pandas_to_spark_with_schema(chainage_pdf, chainage_schema)

# ── Step 4: Write to dlm_kpi.master_chainage (replaceWhere for idempotency) ──
full_name = f"{DLM_KPI_NS}.master_chainage"
table_exists = spark.catalog.tableExists(full_name)
if table_exists:
    (chainage_sdf.write.format('delta').mode('overwrite')
     .option('replaceWhere', f"upc_code = '{UPC}' AND phase = '{PHASE}' AND cycle = '{CYCLE}'")
     .saveAsTable(full_name))
else:
    (chainage_sdf.write.format('delta').mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(full_name))

written = spark.table(full_name).filter(f"upc_code = '{UPC}'").count()
print(f"\n✅ Written {written:,} rows to {full_name}")
display(spark.table(full_name).filter(f"upc_code = '{UPC}'").orderBy('lane', 'start_chainage').limit(10))

Per-lane raw bounds (data-driven):
  L1:      0 –  76250
  L2:      0 –  76250
  R1:    580 –  76250
  R2:     10 –  76250

Final lane bounds for chainage spine (per-lane, no global override):
  L1:      0 –  76260  (7,626 segments)
  L2:      0 –  76260  (7,626 segments)
  R1:    580 –  76260  (7,568 segments)
  R2:     10 –  76260  (7,625 segments)

master_chainage: 30,445 rows
Lanes: ['L1', 'L2', 'R1', 'R2']
  L1:      0 –  76260  (7,626 segments)
  L2:      0 –  76260  (7,626 segments)
  R1:    580 –  76260  (7,568 segments)
  R2:     10 –  76260  (7,625 segments)
Pavement types: {'Cement Concrete': 29943, 'Asphalt': 502}

✅ Written 30,445 rows to nhai_dhms.dlm_kpi.master_chainage


upc,upc_code,source_system,defect_detected_on,start_chainage,end_chainage,phase,cycle,zone,lane,direction,nh_number,section_code,pavement_type,carriageway
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,0.0,10.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,10.0,20.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,20.0,30.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,30.0,40.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,40.0,50.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,50.0,60.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,60.0,70.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,70.0,80.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,80.0,90.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided
N_07024_01001_GJ,N_07024_01001_GJ,NSV,2026-04-21,90.0,100.0,2,1,B,L1,Increasing,NH-151A,DWA-KHA,Cement Concrete,Divided


In [0]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # TRUNCATE ALL TABLES IN BOTH DLM SCHEMAS
# # ═══════════════════════════════════════════════════════════════════════════════
# schemas_to_truncate = [DLM_KPI_NS, DLM_POSTGRES_NS]

# for schema_ns in schemas_to_truncate:
#     print(f"\n{'='*60}")
#     print(f"Schema: {schema_ns}")
#     print(f"{'='*60}")
#     tables = [t.name for t in spark.catalog.listTables(schema_ns)]
#     if not tables:
#         print("  No tables found.")
#         continue
#     for tbl in sorted(tables):
#         full_name = f"{schema_ns}.{tbl}"
#         try:
#             spark.sql(f"TRUNCATE TABLE {full_name}")
#             print(f"  TRUNCATED: {full_name}")
#         except Exception as e:
#             print(f"  FAILED:    {full_name} — {e}")

# print(f"\nDone. All tables in {' and '.join(schemas_to_truncate)} have been truncated.")


Schema: nhai_dhms.dlm_kpi
  TRUNCATED: nhai_dhms.dlm_kpi.master_chainage
  TRUNCATED: nhai_dhms.dlm_kpi.master_cracking
  TRUNCATED: nhai_dhms.dlm_kpi.master_pothole
  TRUNCATED: nhai_dhms.dlm_kpi.master_ravelling_child_100m
  TRUNCATED: nhai_dhms.dlm_kpi.master_ravelling_child_10m
  TRUNCATED: nhai_dhms.dlm_kpi.master_ravelling_parent
  TRUNCATED: nhai_dhms.dlm_kpi.master_roughness_child_100m
  TRUNCATED: nhai_dhms.dlm_kpi.master_roughness_child_10m
  TRUNCATED: nhai_dhms.dlm_kpi.master_roughness_parent
  TRUNCATED: nhai_dhms.dlm_kpi.master_rutting_child_100m
  TRUNCATED: nhai_dhms.dlm_kpi.master_rutting_child_10m
  TRUNCATED: nhai_dhms.dlm_kpi.master_rutting_parent
  TRUNCATED: nhai_dhms.dlm_kpi.validation_log

Schema: nhai_dhms.dlm_postgres
  TRUNCATED: nhai_dhms.dlm_postgres.data_push_log
  TRUNCATED: nhai_dhms.dlm_postgres.master_cracking
  TRUNCATED: nhai_dhms.dlm_postgres.master_pothole
  TRUNCATED: nhai_dhms.dlm_postgres.master_ravelling_child_100m
  TRUNCATED: nhai_dhms.dlm_p

In [0]:
# =======================================================================
# D2 vs DLM KPI BREACH VALIDATION — ALL DEFECTS, ALL LANES
#
# Context: Each defect has 3 levels in the DB:
#   - parent (1000m) — must match D2 breach counts
#   - child_100m     — breach if 100m value > threshold
#   - child_10m      — breach if 10m value > threshold
#
# Cracking has ONLY child_10m rows. For D2 comparison we AGGREGATE
# 10m values to 1000m and compute breach from that aggregated value.
# The 10m breach_flag is correct at the 10m level.
#
# Roughness/Rutting/Ravelling have parent-level rows → direct compare.
# =======================================================================
import pandas as pd, numpy as np, os
from pyspark.sql import functions as F

# ── 1. Load D2 CA ──
_ca_files = [f for f in os.listdir(D2_BASE) if 'CA' in f and f.endswith('.parquet')]
d2 = pd.read_parquet(os.path.join(D2_BASE, _ca_files[0]))
d2ch = next(c for c in d2.columns if 'start' in c.lower() and 'chainage' in c.lower())
d2 = d2.dropna(subset=[d2ch])
d2[d2ch] = pd.to_numeric(d2[d2ch], errors='coerce')
d2['bin_1000'] = (d2[d2ch] // 1000 * 1000).astype(int)
d2_cols = list(d2.columns)
print(f"D2 CA: {len(d2)} rows, chainage {d2[d2ch].min():.0f}-{d2[d2ch].max():.0f}")

def _find_d2_col(cols, lane, pattern, interval_hint):
    ll = lane.lower()
    for c in cols:
        cl = c.lower()
        if ll in cl and pattern in cl and (not interval_hint or interval_hint in cl):
            return c
    for c in cols:
        cl = c.lower()
        if ll in cl and pattern.split()[0] in cl:
            return c
    return None

# ── 2. Defect config ──
# (defect, table, metric_col, d2_pattern, d2_interval, threshold, unit, agg_to_1km)
# agg_to_1km: how to aggregate DB child data to 1000m for comparison with D2
#   'crack_pct' = sum(area_m2)/3500*100,  'max' = max(val),  'direct' = already parent
DEFECT_CONFIG = [
    ('Cracking',  'master_cracking',         'metric_value', 'crack area',   '1000m', CA_CRACK_PCT,              '%',    'crack_pct'),
    ('Roughness', 'master_roughness_parent',  'metric_value', 'roughness bi', '1000m', CA_BI_THRESHOLD,          'mm/km', 'direct'),
    ('Rutting',   'master_rutting_parent',     'metric_value', 'rut depth',   '1000m', CA_RUT_THRESHOLD,          'mm',   'direct'),
    ('Ravelling', 'master_ravelling_parent',   'metric_value', 'area (%)',    '',       RAVELLING_RATING_THRESHOLD, '%',   'direct'),
]

# ── 3. Validate each defect ──
all_results = []

for defect, table, metric_col, d2_pat, d2_int, thresh, unit, agg_method in DEFECT_CONFIG:
    full_table = f"{DLM_KPI_NS}.{table}"
    if not spark.catalog.tableExists(full_table):
        print(f"\n{defect}: {full_table} not found, skipping")
        continue

    # Read DB data
    _schema_fields = [f.name for f in spark.table(full_table).schema.fields]
    sdf = spark.table(full_table).filter(f"upc_code = '{UPC}'")
    if 'threshold_name' in _schema_fields:
        sdf = sdf.filter("threshold_name = 'As per CA'")

    _select_cols = [
        F.col('lane').alias('Lane'),
        F.col('start_chainage').cast('double').alias('start_ch'),
        F.col('end_chainage').cast('double').alias('end_ch'),
        F.col(metric_col).cast('double').alias('db_val'),
        F.col('breach_flag').cast('int').alias('db_breach'),
    ]
    if 'level' in _schema_fields:
        _select_cols.append(F.col('level'))
    db_pdf = sdf.select(*_select_cols).toPandas()
    if db_pdf.empty:
        print(f"\n{defect}: No data"); continue

    # Normalize R-lane chainage
    db_pdf['_sch'] = db_pdf.apply(
        lambda r: min(r['start_ch'], r['end_ch'])
        if pd.notna(r['end_ch']) and r['end_ch'] < r['start_ch']
        else r['start_ch'], axis=1)
    db_pdf['bin_1000'] = (db_pdf['_sch'] // 1000 * 1000).astype(int)

    # Print header
    print(f"\n{'='*90}")
    print(f"{defect.upper()} — D2 vs DLM KPI @ 1000m (threshold={thresh}{unit})")
    db_level = db_pdf['level'].iloc[0] if 'level' in db_pdf.columns else 'parent'
    print(f"  DB level: {db_level} | agg_method: {agg_method}")
    print(f"{'='*90}")
    print(f"{'Lane':>4s}  {'D2_Val':>8s}  {'DB_Val':>8s}  {'ValDiff':>8s}  "
          f"{'D2_Brch':>7s}  {'DB_Brch':>7s}  {'Match':>6s}  {'Detail':>20s}")

    for lane in ['L1', 'L2', 'R1', 'R2']:
        d2_col = _find_d2_col(d2_cols, lane, d2_pat, d2_int)
        if not d2_col: continue

        # D2 at 1000m
        d2_1k = d2.groupby('bin_1000')[d2_col].first().reset_index()
        d2_1k.columns = ['bin_1000', 'd2_val']
        d2_1k['d2_val'] = pd.to_numeric(d2_1k['d2_val'], errors='coerce').fillna(0)
        d2_1k['d2_breach'] = (d2_1k['d2_val'] > thresh).astype(int)

        # DB → aggregate to 1000m depending on method
        lane_db = db_pdf[db_pdf['Lane'] == lane].copy()

        if agg_method == 'crack_pct':
            # Cracking child_10m: metric_value = cracking% at 10m
            # area_m2 = pct * 35 / 100. 1000m pct = sum(area_m2) / 3500 * 100
            db_1k = lane_db.groupby('bin_1000')['db_val'].apply(
                lambda x: (x * 35 / 100).sum() / 3500 * 100
            ).reset_index(name='db_1k_val')
            # Compute breach from the AGGREGATED 1000m value (not from 10m flags)
            db_1k['db_breach'] = (db_1k['db_1k_val'] > thresh).astype(int)
        elif agg_method == 'direct':
            # Parent table: value is already at 1000m level
            db_1k = lane_db.groupby('bin_1000').agg(
                db_1k_val=('db_val', 'first'),
                db_breach=('db_breach', 'max'),
            ).reset_index()
        else:
            db_1k = lane_db.groupby('bin_1000').agg(
                db_1k_val=('db_val', 'max'),
                db_breach=('db_breach', 'max'),
            ).reset_index()

        # Merge and compare
        cmp = d2_1k.merge(db_1k, on='bin_1000', how='outer').fillna(0)
        cmp['val_diff'] = (cmp['db_1k_val'] - cmp['d2_val']).abs()
        cmp['breach_match'] = cmp['d2_breach'] == cmp['db_breach']

        d2_b = int(cmp['d2_breach'].sum())
        db_b = int(cmp['db_breach'].sum())
        n = len(cmp)
        match_pct = cmp['breach_match'].mean() * 100
        mean_diff = cmp['val_diff'].mean()

        mismatches = cmp[~cmp['breach_match']]
        d2_only = len(mismatches[mismatches['d2_breach'] > mismatches['db_breach']])
        db_only = len(mismatches[mismatches['db_breach'] > mismatches['d2_breach']])
        detail = f"D2only={d2_only} DBonly={db_only}" if len(mismatches) > 0 else "PERFECT"

        all_results.append({
            'Defect': defect, 'Lane': lane,
            'D2_Breach': d2_b, 'DB_Breach': db_b,
            'Breach_Match%': round(match_pct, 1),
            'Mean_Diff': round(mean_diff, 4),
            'Bins': n, 'D2only': d2_only, 'DBonly': db_only,
        })

        print(f"{lane:>4s}  {cmp['d2_val'].mean():>8.2f}  {cmp['db_1k_val'].mean():>8.2f}  "
              f"{mean_diff:>+7.4f}  {d2_b:>7d}  {db_b:>7d}  {match_pct:>5.1f}%  {detail:>20s}")

# ── 4. Summary ──
result_df = pd.DataFrame(all_results)
print(f"\n{'='*90}")
print("OVERALL SUMMARY")
print(f"{'='*90}")
print(f"{'Defect':<12s} {'Lane':>4s}  {'D2':>4s}  {'DB':>4s}  {'Match%':>7s}  {'ValDiff':>8s}  {'Status':>8s}")
for _, r in result_df.iterrows():
    ok = r['Breach_Match%'] >= 95
    print(f"{r['Defect']:<12s} {r['Lane']:>4s}  {r['D2_Breach']:>4d}  {r['DB_Breach']:>4d}  "
          f"{r['Breach_Match%']:>6.1f}%  {r['Mean_Diff']:>+7.4f}  {'OK' if ok else 'CHECK':>8s}")

print(f"\nPer-defect:")
for defect in result_df['Defect'].unique():
    sub = result_df[result_df['Defect'] == defect]
    d2_t = sub['D2_Breach'].sum(); db_t = sub['DB_Breach'].sum()
    avg_m = sub['Breach_Match%'].mean()
    print(f"  {defect:<12s}: D2={d2_t:>3d}  DB={db_t:>3d}  match={avg_m:.1f}%  {'OK' if avg_m>95 else 'CHECK'}")

display(spark.createDataFrame(result_df))

D2 CA: 931 rows, chainage 447385-540300

CRACKING — D2 vs DLM KPI @ 1000m (threshold=10.0%)
  DB level: child_10m | agg_method: crack_pct
Lane    D2_Val    DB_Val   ValDiff  D2_Brch  DB_Brch   Match                Detail
  L1      2.51      3.30  +2.6032        5        7   89.4%     D2only=4 DBonly=6
  L2      1.54      2.38  +2.2958        2        4   93.6%     D2only=2 DBonly=4
  R1      2.14      3.02  +2.8154        4        6   91.5%     D2only=3 DBonly=5
  R2      1.16      2.03  +1.7814        2        3   94.7%     D2only=2 DBonly=3

ROUGHNESS — D2 vs DLM KPI @ 1000m (threshold=2400.0mm/km)
  DB level: parent | agg_method: direct
Lane    D2_Val    DB_Val   ValDiff  D2_Brch  DB_Brch   Match                Detail
  L1   1778.12   1835.59  +575.8074       18       16   76.6%   D2only=12 DBonly=10
  L2   1481.02   1423.61  +474.7486        9        5   91.5%     D2only=6 DBonly=2
  R1   2232.54   2101.09  +729.2576       33       30   69.1%   D2only=16 DBonly=13
  R2   1571.17   

Defect,Lane,D2_Breach,DB_Breach,Breach_Match%,Mean_Diff,Bins,D2only,DBonly
Cracking,L1,5,7,89.4,2.6032,94,4,6
Cracking,L2,2,4,93.6,2.2958,94,2,4
Cracking,R1,4,6,91.5,2.8154,94,3,5
Cracking,R2,2,3,94.7,1.7814,94,2,3
Roughness,L1,18,16,76.6,575.8074,94,12,10
Roughness,L2,9,5,91.5,474.7486,94,6,2
Roughness,R1,33,30,69.1,729.2576,94,16,13
Roughness,R2,14,5,84.0,471.222,94,12,3
Rutting,L1,7,2,92.6,1.4964,94,6,1
Rutting,L2,8,3,92.6,1.3732,94,6,1


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFY: CA vs IRC threshold_name + threshold_value + breach mapping
# ═══════════════════════════════════════════════════════════════════════════════
import os

# ── 1. PARSED THRESHOLD VALUES ──
print("PARSED THRESHOLD VALUES FROM D2")
print("="*80)
print(f"  CA thresholds:")
print(f"    roughness_bi   = {CA_BI_THRESHOLD} mm/km  (BI, Concession Agreement)")
print(f"    cracking       = {CA_CRACK_PCT}%")
print(f"    rutting        = {CA_RUT_THRESHOLD} mm")
print(f"    ravelling      = {RAVELLING_RATING_THRESHOLD} {RAVELLING_UNIT}")
print(f"\n  IRC thresholds:")
print(f"    roughness_iri  = {IRC_IRI_POOR} mm/km")
print(f"    cracking       = {IRC_CRACK_PCT}%")
print(f"    rutting        = {IRC_RUT_THRESHOLD} mm")
print(f"    ravelling      = {IRC_RAVELLING_THRESHOLD} {IRC_RAVELLING_UNIT}")
print(f"\n  Tracker override: {TRACKER_CTX.get('threshold_label_override')}")

# ── 2. CHECK D2 COLUMN NAMES to verify ravelling unit ──
print(f"\n{'='*80}")
print("D2 PARQUET — LIMITATION + RAVELLING COLUMNS (verify units)")
print("="*80)
for pf in sorted(os.listdir(D2_BASE)):
    if not pf.endswith('.parquet'): continue
    _df = pd.read_parquet(os.path.join(D2_BASE, pf))
    tag = 'IRC' if 'irc' in pf.lower() else 'CA'
    limit_cols = [c for c in _df.columns if 'limit' in c.lower()]
    rav_cols = [c for c in _df.columns if 'ravel' in c.lower()]
    print(f"\n  [{tag}] {pf[:70]}")
    if limit_cols: print(f"    Limitation cols: {limit_cols}")
    if rav_cols:   print(f"    Ravelling cols:  {rav_cols}")

# ── 3. VERIFY OUTPUT_DATAFRAMES: threshold mapping per table ──
print(f"\n{'='*80}")
print("OUTPUT_DATAFRAMES: threshold_name + threshold_value + breach counts")
print("="*80)
for tbl in sorted(OUTPUT_DATAFRAMES):
    sdf = OUTPUT_DATAFRAMES[tbl]
    cnt = sdf.count()
    if cnt == 0: continue
    tn_stats = sdf.groupBy('threshold_name').agg(
        F.count('*').alias('rows'),
        F.sum(F.when(F.col('breach_flag') == 1, 1).otherwise(0)).alias('breach'),
        F.first('threshold_value').alias('thresh_val'),
        F.first('metric_name').alias('metric'),
    ).collect()
    print(f"\n  {tbl}:")
    for r in tn_stats:
        tn = r['threshold_name'] or 'NULL'
        print(f"    {tn:15s} | thresh_val={str(r['thresh_val']):<8s} | "
              f"metric={r['metric']:<15s} | rows={r['rows']:>6,} | breach={r['breach']:>5,}")

# ── 4. EXPECTED vs ACTUAL mapping check ──
print(f"\n{'='*80}")
print("EXPECTED vs ACTUAL THRESHOLD MAPPING")
print("="*80)
_expected = {
    'roughness': {'As per CA': (CA_BI_THRESHOLD, 'roughness_bi'), 'As per IRC': (IRC_IRI_POOR, 'roughness_bi')},
    'cracking':  {'As per CA': (CA_CRACK_PCT, 'cracking_pct'),   'As per IRC': (IRC_CRACK_PCT, 'cracking_pct')},
    'rutting':   {'As per CA': (CA_RUT_THRESHOLD, 'rutting_avg'), 'As per IRC': (IRC_RUT_THRESHOLD, 'rutting_avg')},
    'ravelling': {'As per CA': (RAVELLING_RATING_THRESHOLD, 'ravelling_area'), 'As per IRC': (IRC_RAVELLING_THRESHOLD, 'ravelling_area')},
}
for defect, standards in _expected.items():
    print(f"\n  {defect}:")
    for std_name, (exp_val, exp_metric) in standards.items():
        print(f"    {std_name:15s} → threshold_value={exp_val}, metric_name={exp_metric}")

PARSED THRESHOLD VALUES FROM D2
  CA thresholds:
    roughness_bi   = 2400.0 mm/km  (BI, Concession Agreement)
    cracking       = 10.0%
    rutting        = 10.0 mm
    ravelling      = 10.0 pct

  IRC thresholds:
    roughness_iri  = 2400.0 mm/km
    cracking       = 10.0%
    rutting        = 10.0 mm
    ravelling      = 10.0 pct

  Tracker override: As per CA

D2 PARQUET — LIMITATION + RAVELLING COLUMNS (verify units)

  [CA] ZB_P2_C1_N_02001_10003_GJ_2.Deliverable - 2 - Comparison as per CA_Com
    Limitation cols: ['Limitation of Roughness as per IRC (in mm/km)', 'Limitation of Cracking as per as per IRC (in % Area)', 'Limitation of Ravelling as per as per IRC (%)', 'Limitation of Rut Depth as per IRC (in mm)']
    Ravelling cols:  ['Limitation of Ravelling as per as per IRC (%)', 'Ravelling L1 Area (%)', 'Ravelling L2 Area (%)', 'Ravelling L3 Area (%)', 'Ravelling R1 Area (%)', 'Ravelling R2 Area (%)', 'Ravelling R3 Area (%)']

  [CA] ZB_P2_C1_N_02001_10003_GJ_2.Deliverable - 2

In [0]:
# =======================================================================
# D2 vs DB: FULL VALUE-LEVEL COMPARISON — 10m, 100m, 1000m
# All Defects (Cracking, Roughness, Rutting, Ravelling), All 4 Lanes
#
# 1000m: D2 CA  vs DB parent (or cracking agg to 1000m)
# 100m:  D2 IRC vs DB child_100m
# 10m:   Source parquet vs DB child_10m (direct row check)
# =======================================================================
import pandas as pd, numpy as np, os
from pyspark.sql import functions as F

# ────────────────────── LOAD D2 CA & IRC ──────────────────────────────────
_ca_files = [f for f in sorted(os.listdir(D2_BASE))
             if f.endswith('.parquet') and 'ca' in f.lower()
             and 'irc' not in f.lower() and 'roughness,rut' not in f.lower()]
_irc_files = [f for f in sorted(os.listdir(D2_BASE))
              if f.endswith('.parquet') and 'irc' in f.lower()]

print("D2 FILES:")
print(f"  CA files:  {[f[:60] for f in _ca_files]}")
print(f"  IRC files: {[f[:60] for f in _irc_files]}")

# Load CA (1000m intervals)
d2_ca = pd.read_parquet(os.path.join(D2_BASE, _ca_files[0])) if _ca_files else pd.DataFrame()
# Load IRC (100m intervals)
d2_irc = pd.read_parquet(os.path.join(D2_BASE, _irc_files[0])) if _irc_files else pd.DataFrame()

for tag, df in [('CA', d2_ca), ('IRC', d2_irc)]:
    if df.empty: continue
    ch_col = next((c for c in df.columns if 'start' in c.lower() and 'chainage' in c.lower()), None)
    df[ch_col] = pd.to_numeric(df[ch_col], errors='coerce')
    print(f"\n  {tag}: {len(df)} rows, {len(df.columns)} cols")
    rav_cols = [c for c in df.columns if 'ravel' in c.lower() or 'rav ' in c.lower()]
    rough_cols = [c for c in df.columns if 'rough' in c.lower() and 'limit' not in c.lower()]
    crack_cols = [c for c in df.columns if 'crack' in c.lower() and 'limit' not in c.lower()]
    rut_cols = [c for c in df.columns if 'rut' in c.lower() and 'limit' not in c.lower()]
    print(f"    Roughness: {rough_cols[:3]}")
    print(f"    Cracking:  {crack_cols[:3]}")
    print(f"    Rutting:   {rut_cols[:3]}")
    print(f"    Ravelling: {rav_cols[:3]}")

def _find_d2_col(cols, lane, pattern, hint=''):
    """Find D2 column matching lane + defect pattern + interval hint."""
    ll = lane.lower()
    for c in cols:
        cl = c.lower()
        if ll in cl and pattern in cl and (not hint or hint in cl):
            return c
    # Fallback: first word of pattern
    for c in cols:
        cl = c.lower()
        if ll in cl and pattern.split()[0] in cl and 'limit' not in cl:
            return c
    return None

# ─── Helper: prepare D2 reference at given interval ──────────────────────
def prepare_d2_ref(d2_df, bin_m, lane, d2_pattern, interval_hint=''):
    """Return df[bin, d2_val] for a D2 sheet at given bin size."""
    if d2_df.empty: return pd.DataFrame(columns=['bin', 'd2_val'])
    ch_col = next((c for c in d2_df.columns if 'start' in c.lower() and 'chainage' in c.lower()), None)
    if not ch_col: return pd.DataFrame(columns=['bin', 'd2_val'])
    d2_col = _find_d2_col(list(d2_df.columns), lane, d2_pattern, interval_hint)
    if not d2_col: return pd.DataFrame(columns=['bin', 'd2_val'])
    tmp = d2_df[[ch_col, d2_col]].copy()
    tmp[ch_col] = pd.to_numeric(tmp[ch_col], errors='coerce')
    tmp = tmp.dropna(subset=[ch_col])
    tmp['bin'] = (tmp[ch_col] // bin_m * bin_m).astype(int)
    tmp['d2_val'] = pd.to_numeric(tmp[d2_col], errors='coerce').fillna(0)
    ref = tmp.groupby('bin')['d2_val'].first().reset_index()
    return ref

# ─── Helper: prepare DB data at given level ──────────────────────────────
def prepare_db_at_level(table_name, level_filter=None, bin_m=None, agg_method='first'):
    """Read DB table, optionally filter by level, align to bin."""
    full_tbl = f"{DLM_KPI_NS}.{table_name}"
    if not spark.catalog.tableExists(full_tbl): return pd.DataFrame()
    _flds = [f.name for f in spark.table(full_tbl).schema.fields]
    sdf = spark.table(full_tbl).filter(f"upc_code = '{UPC}'")
    if 'threshold_name' in _flds:
        sdf = sdf.filter("threshold_name = 'As per CA'")
    if level_filter and 'level' in _flds:
        sdf = sdf.filter(f"level = '{level_filter}'")
    sel = [
        F.col('lane').alias('Lane'),
        F.col('start_chainage').cast('double').alias('s_ch'),
        F.col('end_chainage').cast('double').alias('e_ch'),
        F.col('metric_value').cast('double').alias('db_val'),
        F.col('breach_flag').cast('int').alias('db_bf'),
    ]
    db = sdf.select(*sel).toPandas()
    if db.empty: return db
    # Normalize R-lane chainage
    db['_sch'] = db.apply(lambda r: min(r['s_ch'], r['e_ch'])
        if pd.notna(r['e_ch']) and r['e_ch'] < r['s_ch'] else r['s_ch'], axis=1)
    if bin_m:
        db['bin'] = (db['_sch'] // bin_m * bin_m).astype(int)
    return db

# ────────────────────── DEFECT CONFIG ─────────────────────────────────────
# For each defect: tables for 10m/100m/1000m, D2 patterns, thresholds
DEFECT_CFG = [
    {
        'name': 'Cracking', 'unit': '%', 'thresh': CA_CRACK_PCT,
        'levels': {
            '10m':   {'table': 'master_cracking', 'level': 'child_10m', 'bin_m': 10},
            '100m':  {'table': 'master_cracking', 'level': 'child_10m', 'bin_m': 100,
                      'agg': 'crack_pct_100'},
            '1000m': {'table': 'master_cracking', 'level': 'child_10m', 'bin_m': 1000,
                      'agg': 'crack_pct_1000'},
        },
        'd2_ca_pat': 'crack area', 'd2_ca_hint': '1000m',
        'd2_irc_pat': 'crack area', 'd2_irc_hint': '100m',
    },
    {
        'name': 'Roughness', 'unit': 'mm/km', 'thresh': CA_BI_THRESHOLD,
        'levels': {
            '10m':   {'table': 'master_roughness_child_10m', 'level': 'child_10m', 'bin_m': 10},
            '100m':  {'table': 'master_roughness_child_100m', 'level': 'child_100m', 'bin_m': 100},
            '1000m': {'table': 'master_roughness_parent', 'level': 'parent', 'bin_m': 1000},
        },
        'd2_ca_pat': 'roughness bi', 'd2_ca_hint': '1000m',
        'd2_irc_pat': 'roughness bi', 'd2_irc_hint': '100m',
    },
    {
        'name': 'Rutting', 'unit': 'mm', 'thresh': CA_RUT_THRESHOLD,
        'levels': {
            '10m':   {'table': 'master_rutting_child_10m', 'level': 'child_10m', 'bin_m': 10},
            '100m':  {'table': 'master_rutting_child_100m', 'level': 'child_100m', 'bin_m': 100},
            '1000m': {'table': 'master_rutting_parent', 'level': 'parent', 'bin_m': 1000},
        },
        'd2_ca_pat': 'rut depth', 'd2_ca_hint': '1000m',
        'd2_irc_pat': 'rut depth', 'd2_irc_hint': '100m',
    },
    {
        'name': 'Ravelling', 'unit': RAVELLING_UNIT, 'thresh': RAVELLING_RATING_THRESHOLD,
        'levels': {
            '10m':   {'table': 'master_ravelling_child_10m', 'level': 'child_10m', 'bin_m': 10},
            '100m':  {'table': 'master_ravelling_child_100m', 'level': 'child_100m', 'bin_m': 100},
            '1000m': {'table': 'master_ravelling_parent', 'level': 'parent', 'bin_m': 1000},
        },
        'd2_ca_pat': 'area', 'd2_ca_hint': '',
        'd2_irc_pat': 'area', 'd2_irc_hint': '',
    },
]

# ────────────────────── MAIN COMPARISON LOOP ─────────────────────────────
all_rows = []

for dcfg in DEFECT_CFG:
    defect = dcfg['name']
    unit = dcfg['unit']
    thresh = dcfg['thresh']

    print(f"\n{'#'*100}")
    print(f"  {defect.upper()} (threshold={thresh}{unit})")
    print(f"{'#'*100}")

    for lvl_name in ['10m', '100m', '1000m']:
        lvl = dcfg['levels'][lvl_name]
        bin_m = lvl['bin_m']
        agg_method = lvl.get('agg', 'first')

        # Read DB data
        db = prepare_db_at_level(lvl['table'], lvl.get('level'), bin_m=None)
        if db.empty:
            print(f"\n  [{lvl_name}] No DB data for {lvl['table']}"); continue

        # Normalize start chainage
        db['bin'] = (db['_sch'] // bin_m * bin_m).astype(int)

        # Pick D2 reference: 1000m -> CA, 100m -> IRC, 10m -> source parquet
        if lvl_name == '1000m':
            d2_ref_src = d2_ca
            d2_pat = dcfg['d2_ca_pat']
            d2_hint = dcfg['d2_ca_hint']
            ref_tag = 'D2_CA'
        elif lvl_name == '100m':
            d2_ref_src = d2_irc
            d2_pat = dcfg['d2_irc_pat']
            d2_hint = dcfg['d2_irc_hint']
            ref_tag = 'D2_IRC'
        else:
            d2_ref_src = pd.DataFrame()  # 10m: no D2 reference, compare row counts
            d2_pat = ''
            ref_tag = 'SrcParquet'

        print(f"\n  [{lvl_name}] {defect} — ref={ref_tag}, table={lvl['table']}")
        print(f"  {'Lane':>4s}  {'Bins':>5s}  {'Ref_Mean':>9s}  {'DB_Mean':>9s}  {'MeanDiff':>9s}  "
              f"{'MaxDiff':>8s}  {'Close%':>7s}  {'Ref_B':>5s}  {'DB_B':>5s}  {'B_Match':>7s}")

        for lane in ['L1', 'L2', 'R1', 'R2']:
            ldb = db[db['Lane'] == lane].copy()
            if ldb.empty: continue

            # Aggregate DB to bin level
            if agg_method == 'crack_pct_1000':
                db_agg = ldb.groupby('bin')['db_val'].apply(
                    lambda x: (x * 35 / 100).sum() / 3500 * 100
                ).reset_index(name='db_agg')
                db_bf_agg = ldb.groupby('bin')['db_val'].apply(
                    lambda x: 1 if (x * 35 / 100).sum() / 3500 * 100 > thresh else 0
                ).reset_index(name='db_bf')
                db_agg = db_agg.merge(db_bf_agg, on='bin')
            elif agg_method == 'crack_pct_100':
                db_agg = ldb.groupby('bin')['db_val'].apply(
                    lambda x: (x * 35 / 100).sum() / (35 * len(x)) * 100 if len(x) > 0 else 0
                ).reset_index(name='db_agg')
                db_agg['db_bf'] = (db_agg['db_agg'] > thresh).astype(int)
            else:
                db_agg = ldb.groupby('bin').agg(
                    db_agg=('db_val', 'mean'), db_bf=('db_bf', 'max')
                ).reset_index()

            # For 10m: just show DB stats (no D2 at 10m)
            if lvl_name == '10m':
                n = len(db_agg)
                db_mean = db_agg['db_agg'].mean()
                db_breach = int(db_agg['db_bf'].sum())
                non_zero = (db_agg['db_agg'] > 0).sum()
                all_rows.append({
                    'Defect': defect, 'Lane': lane, 'Level': lvl_name,
                    'Ref': ref_tag, 'Bins': n,
                    'RefMean': None, 'DBMean': round(db_mean, 4),
                    'MeanDiff': None, 'MaxDiff': None,
                    'Close_Pct': None,
                    'RefBreach': None, 'DBBreach': db_breach,
                    'BreachMatch': None,
                    'Status': f'{non_zero} non-zero',
                })
                print(f"  {lane:>4s}  {n:>5d}  {'n/a':>9s}  {db_mean:>9.4f}  {'n/a':>9s}  "
                      f"{'n/a':>8s}  {'n/a':>7s}  {'n/a':>5s}  {db_breach:>5d}  "
                      f"{'n/a':>7s}  ({non_zero} non-zero)")
                continue

            # 100m or 1000m: merge with D2 reference
            ref = prepare_d2_ref(d2_ref_src, bin_m, lane, d2_pat, d2_hint)
            if ref.empty:
                print(f"  {lane:>4s}  No D2 {ref_tag} column found for {d2_pat}"); continue

            cmp = ref.merge(db_agg, on='bin', how='inner')
            if cmp.empty:
                print(f"  {lane:>4s}  No matching bins"); continue

            cmp['diff'] = (cmp['db_agg'] - cmp['d2_val']).abs()
            cmp['ref_bf'] = (cmp['d2_val'] > thresh).astype(int)
            n = len(cmp)

            # Tolerance
            if defect == 'Roughness':
                tol = 50
            elif defect == 'Cracking':
                tol = 0.5
            else:
                tol = 0.5
            close_n = (cmp['diff'] <= tol).sum()
            close_pct = close_n / n * 100

            ref_b = int(cmp['ref_bf'].sum())
            db_b = int(cmp['db_bf'].sum())
            b_match_n = (cmp['ref_bf'] == cmp['db_bf']).sum()

            all_rows.append({
                'Defect': defect, 'Lane': lane, 'Level': lvl_name,
                'Ref': ref_tag, 'Bins': n,
                'RefMean': round(cmp['d2_val'].mean(), 4),
                'DBMean': round(cmp['db_agg'].mean(), 4),
                'MeanDiff': round(cmp['diff'].mean(), 4),
                'MaxDiff': round(cmp['diff'].max(), 4),
                'Close_Pct': round(close_pct, 1),
                'RefBreach': ref_b, 'DBBreach': db_b,
                'BreachMatch': f"{b_match_n}/{n}",
                'Status': 'OK' if close_pct >= 90 and b_match_n / n >= 0.95 else 'CHECK',
            })
            print(f"  {lane:>4s}  {n:>5d}  {cmp['d2_val'].mean():>9.2f}  {cmp['db_agg'].mean():>9.2f}  "
                  f"{cmp['diff'].mean():>+8.4f}  {cmp['diff'].max():>8.3f}  "
                  f"{close_pct:>6.1f}%  {ref_b:>5d}  {db_b:>5d}  {b_match_n:>3d}/{n:<3d}")

    # ── Ravelling deep-dive: check raw source values ──
    if defect == 'Ravelling':
        print(f"\n  ** RAVELLING DEEP-DIVE: check raw source data **")
        # Check what df_ravelling_raw looks like
        if 'df_ravelling_raw' in dir():
            rr = df_ravelling_raw
            print(f"     df_ravelling_raw: {len(rr)} rows, cols: {list(rr.columns)[:10]}")
            # Check key value columns
            for vc in ['Raveling_Area_sq_m', 'Ravelling_Area_sq_m', 'ravelling_area',
                        'Area', 'area', 'Percent_Area', 'metric_value']:
                if vc in rr.columns:
                    vals = pd.to_numeric(rr[vc], errors='coerce')
                    print(f"     {vc}: min={vals.min():.4f}, max={vals.max():.4f}, "
                          f"mean={vals.mean():.4f}, non-zero={int((vals > 0).sum())}")
        else:
            print(f"     df_ravelling_raw not in scope")
        # Check DB ravelling raw values
        rav_tbl = f"{DLM_KPI_NS}.master_ravelling_child_10m"
        if spark.catalog.tableExists(rav_tbl):
            rav_db = spark.table(rav_tbl).filter(f"upc_code = '{UPC}'").filter("threshold_name = 'As per CA'")
            rav_stats = rav_db.agg(
                F.count('*').alias('cnt'),
                F.mean('metric_value').alias('avg_mv'),
                F.max('metric_value').alias('max_mv'),
                F.sum(F.when(F.col('metric_value') > 0, 1).otherwise(0)).alias('nonzero'),
                F.mean('ravelling_area').alias('avg_rav'),
                F.max('ravelling_area').alias('max_rav'),
            ).collect()[0]
            print(f"     DB master_ravelling_child_10m (CA):")
            print(f"       rows={rav_stats['cnt']}, avg_metric={rav_stats['avg_mv']}, "
                  f"max_metric={rav_stats['max_mv']}, nonzero={rav_stats['nonzero']}")
            print(f"       avg_ravelling_area={rav_stats['avg_rav']}, max_ravelling_area={rav_stats['max_rav']}")

# ────────────────────── FINAL SUMMARY ────────────────────────────────────
result_df = pd.DataFrame(all_rows)
result_display = result_df[result_df['Level'] != '10m'].copy()  # 100m+1000m have D2 ref
print(f"\n{'='*100}")
print("FINAL SUMMARY: D2 vs DB VALUE MATCH (100m + 1000m)")
print(f"{'='*100}")
print(f"{'Defect':<11s} {'Lane':>4s} {'Lvl':>5s} {'Ref':>7s}  {'MeanDiff':>9s}  "
      f"{'MaxDiff':>8s}  {'Close%':>7s}  {'Ref_B':>5s} {'DB_B':>5s}  {'BrchMatch':>9s}  {'Status':>6s}")
for _, r in result_display.iterrows():
    md = r['MeanDiff'] if pd.notna(r['MeanDiff']) else 0
    mxd = r['MaxDiff'] if pd.notna(r['MaxDiff']) else 0
    cp = r['Close_Pct'] if pd.notna(r['Close_Pct']) else 0
    rb = r['RefBreach'] if pd.notna(r['RefBreach']) else ''
    bm = r['BreachMatch'] or ''
    print(f"{r['Defect']:<11s} {r['Lane']:>4s} {r['Level']:>5s} {r['Ref']:>7s}  {md:>+8.4f}  "
          f"{mxd:>8.3f}  {cp:>6.1f}%  {str(rb):>5s} {r['DBBreach']:>5}  {str(bm):>9s}  {r['Status']:>6s}")

print(f"\n10m LEVEL DB STATS (no D2 ref at 10m):")
result_10m = result_df[result_df['Level'] == '10m']
for _, r in result_10m.iterrows():
    print(f"  {r['Defect']:<11s} {r['Lane']:>4s}  bins={r['Bins']:>5}  "
          f"db_mean={r['DBMean']:>9.4f}  db_breach={r['DBBreach']:>5}  {r['Status']}")

print(f"\nPER-DEFECT, PER-LEVEL SUMMARY:")
for defect in result_df['Defect'].unique():
    for lvl in ['10m', '100m', '1000m']:
        sub = result_df[(result_df['Defect'] == defect) & (result_df['Level'] == lvl)]
        if sub.empty: continue
        if lvl == '10m':
            total_bins = sub['Bins'].sum()
            total_breach = sub['DBBreach'].sum()
            print(f"  {defect:<11s} @{lvl:>5s}: {total_bins:>6} bins, {total_breach:>5} breach")
        else:
            avg_close = sub['Close_Pct'].mean()
            status_ok = (sub['Status'] == 'OK').all()
            print(f"  {defect:<11s} @{lvl:>5s}: close={avg_close:.1f}%, {'ALL OK' if status_ok else 'CHECK'}")

# Convert to all-string for clean display
_disp = result_df.copy()
for c in _disp.columns:
    _disp[c] = _disp[c].apply(lambda v: '' if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
display(spark.createDataFrame(_disp))

D2 FILES:
  CA files:  ['ZB_P2_C1_N_02001_10003_GJ_2.Deliverable - 2 - Comparison as ', 'ZB_P2_C1_N_02001_10003_GJ_2.Deliverable - 2 - Comparison as ', 'ZB_P2_C1_N_02001_10003_GJ_2.Deliverable - 2 - Comparison as ', 'ZB_P2_C1_N_02001_10003_GJ_2.Deliverable - 2 - Comparison as ', 'ZB_P2_C1_N_02001_10003_GJ_2.Deliverable - 2 - Comparison as ']
  IRC files: []

  CA: 931 rows, 65 cols
    Roughness: ['L1 Lane Roughness BI (in mm/km) (100m)', 'L2 Lane Roughness BI (in mm/km) (100m)', 'L3 Lane Roughness BI (in mm/km) (100m)']
    Cracking:  ['L1 % Crack Area (100m)', 'L2 % Crack Area (100m)', 'L3 % Crack Area (100m)']
    Rutting:   ['L1 Rut Depth (in mm) (100m)', 'L2 Rut Depth (in mm) (100m)', 'L3 Rut Depth (in mm) (100m)']
    Ravelling: ['Limitation of Ravelling as per as per IRC (%)', 'Ravelling L1 Area (%)', 'Ravelling L2 Area (%)']

####################################################################################################
  CRACKING (threshold=10.0%)
########################

Defect,Lane,Level,Ref,Bins,RefMean,DBMean,MeanDiff,MaxDiff,Close_Pct,RefBreach,DBBreach,BreachMatch,Status
Cracking,L1,10m,SrcParquet,9296,,3.338,,,,,915,,7741 non-zero
Cracking,L2,10m,SrcParquet,9296,,2.4096,,,,,594,,7303 non-zero
Cracking,R1,10m,SrcParquet,9297,,3.055,,,,,652,,7802 non-zero
Cracking,R2,10m,SrcParquet,9297,,2.0568,,,,,459,,6607 non-zero
Cracking,L1,1000m,D2_CA,94,2.1696,3.3011,2.6613,21.9127,41.5,4.0,7,85/94,CHECK
Cracking,L2,1000m,D2_CA,94,1.3033,2.383,2.3927,20.4889,40.4,2.0,4,88/94,CHECK
Cracking,R1,1000m,D2_CA,94,2.1082,3.0216,2.8333,29.7188,35.1,4.0,6,86/94,CHECK
Cracking,R2,1000m,D2_CA,94,1.1391,2.0343,1.8048,16.1718,43.6,2.0,3,89/94,CHECK
Roughness,L1,10m,SrcParquet,9296,,1855.4788,,,,,2080,,8690 non-zero
Roughness,L2,10m,SrcParquet,9296,,1419.2785,,,,,1168,,8708 non-zero


In [0]:
# =======================================================================
# VALIDATION LOG: D2 vs DB — All Defects, All Lanes, All Levels
# Produces a single clean DataFrame and writes to validation_log table
# =======================================================================
import pandas as pd, numpy as np, os
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import *

VAL_TIMESTAMP = datetime.now(timezone.utc)

# ── 1. Load D2 sheets ──
_ca_files = [f for f in sorted(os.listdir(D2_BASE))
             if f.endswith('.parquet') and 'ca' in f.lower()
             and 'irc' not in f.lower() and 'roughness,rut' not in f.lower()]
_irc_files = [f for f in sorted(os.listdir(D2_BASE))
              if f.endswith('.parquet') and 'irc' in f.lower()]

d2_ca  = pd.read_parquet(os.path.join(D2_BASE, _ca_files[0]))  if _ca_files  else pd.DataFrame()
d2_irc = pd.read_parquet(os.path.join(D2_BASE, _irc_files[0])) if _irc_files else pd.DataFrame()

def _prep_d2(df, bin_m):
    if df.empty: return df
    ch = next((c for c in df.columns if 'start' in c.lower() and 'chainage' in c.lower()), None)
    df[ch] = pd.to_numeric(df[ch], errors='coerce')
    df = df.dropna(subset=[ch])
    df['bin'] = (df[ch] // bin_m * bin_m).astype(int)
    return df

# Detect D2 CA interval: check columns first, then fallback to actual row intervals
_ca_is_100m = False
if not d2_ca.empty:
    _ca_col_str = ' '.join(d2_ca.columns).lower()
    _has_100m = '(100m)' in _ca_col_str
    _has_1000m = '(1000m)' in _ca_col_str
    if _has_100m and not _has_1000m:
        _ca_is_100m = True
    elif not _has_100m and not _has_1000m:
        # Fallback: detect from actual row intervals (End - Start chainage)
        _ch_c = next((c for c in d2_ca.columns if 'start' in c.lower() and 'chainage' in c.lower()), None)
        _end_c = next((c for c in d2_ca.columns if 'end' in c.lower() and 'chainage' in c.lower()), None)
        if _ch_c and _end_c:
            _intervals = pd.to_numeric(d2_ca[_end_c], errors='coerce') - pd.to_numeric(d2_ca[_ch_c], errors='coerce')
            _median_interval = _intervals.dropna().median()
            _ca_is_100m = _median_interval <= 200  # median ≤ 200m → treat as 100m data
            print(f"  D2 CA interval detected from row data: median={_median_interval:.0f}m → {'100m' if _ca_is_100m else '1000m'}")
    print(f"  D2 CA interval: {'100m' if _ca_is_100m else '1000m'} (cols have 100m={_has_100m}, 1000m={_has_1000m})")

# Prep CA at both 100m and 1000m bin levels
d2_ca_100  = _prep_d2(d2_ca.copy(), 100)   if _ca_is_100m else pd.DataFrame()
d2_ca_1000 = _prep_d2(d2_ca.copy(), 1000)
d2_irc     = _prep_d2(d2_irc, 100)

# Fallback: if IRC is empty but CA has 100m data, use CA for 100m
if d2_irc.empty and not d2_ca_100.empty:
    print(f"  ⚠️ No IRC file — using D2 CA (100m) as fallback for 100m comparison")
    d2_irc_fallback = d2_ca_100
    _irc_is_ca_fallback = True
else:
    d2_irc_fallback = d2_irc
    _irc_is_ca_fallback = False

def _find_col(cols, lane, pattern, hint='', exclude=None):
    """Find D2 column matching lane + pattern + hint, optionally excluding a keyword."""
    ll = lane.lower()
    excl = (exclude or '').lower()
    for c in cols:
        cl = c.lower()
        if ll in cl and pattern in cl and (not hint or hint in cl):
            if excl and excl in cl: continue
            return c
    for c in cols:
        cl = c.lower()
        if ll in cl and pattern.split()[0] in cl and 'limit' not in cl:
            if excl and excl in cl: continue
            return c
    return None

# ── 2. DB reader ──
def _read_db(table, level_filter=None):
    tbl = f"{DLM_KPI_NS}.{table}"
    if not spark.catalog.tableExists(tbl): return pd.DataFrame()
    flds = [f.name for f in spark.table(tbl).schema.fields]
    sdf = spark.table(tbl).filter(f"upc_code = '{UPC}'")
    if 'threshold_name' in flds:
        sdf = sdf.filter("threshold_name = 'As per CA'")
    if level_filter and 'level' in flds:
        sdf = sdf.filter(f"level = '{level_filter}'")
    sel = [
        F.col('lane').alias('Lane'),
        F.col('start_chainage').cast('double').alias('s_ch'),
        F.col('end_chainage').cast('double').alias('e_ch'),
        F.col('metric_value').cast('double').alias('db_val'),
        F.col('breach_flag').cast('int').alias('db_bf'),
    ]
    db = sdf.select(*sel).toPandas()
    if db.empty: return db
    db['_sch'] = db.apply(lambda r: min(r['s_ch'], r['e_ch'])
        if pd.notna(r['e_ch']) and r['e_ch'] < r['s_ch'] else r['s_ch'], axis=1)
    return db

# ── 3. Config: defect → levels ──
CFG = [
  # (defect, level, bin_m, db_table, db_level_filter, d2_src, d2_pattern, d2_hint, threshold, unit, agg)
  # (defect, level, bin_m, db_table, db_level_filter, d2_src, d2_pattern, d2_hint, threshold, unit, agg, exclude)
  ('Cracking','10m',   10,'master_cracking','child_10m',  None,'','',CA_CRACK_PCT,'%','raw',''),
  ('Cracking','100m', 100,'master_cracking','child_10m',  'irc','crack area','100m',CA_CRACK_PCT,'%','crack_pct',''),
  ('Cracking','1000m',1000,'master_cracking','child_10m', 'ca','crack area','',CA_CRACK_PCT,'%','crack_pct',''),
  ('Roughness','10m',  10,'master_roughness_child_10m','child_10m',None,'','',CA_BI_THRESHOLD,'mm/km','raw',''),
  ('Roughness','100m',100,'master_roughness_child_100m','child_100m','irc','roughness bi','100m',CA_BI_THRESHOLD,'mm/km','mean',''),
  ('Roughness','1000m',1000,'master_roughness_parent','parent','ca','roughness bi','',CA_BI_THRESHOLD,'mm/km','first',''),
  ('Rutting','10m',    10,'master_rutting_child_10m','child_10m',None,'','',CA_RUT_THRESHOLD,'mm','raw',''),
  ('Rutting','100m',  100,'master_rutting_child_100m','child_100m','irc','rut depth','100m',CA_RUT_THRESHOLD,'mm','mean',''),
  ('Rutting','1000m',1000,'master_rutting_parent','parent','ca','rut depth','',CA_RUT_THRESHOLD,'mm','first',''),
  ('Ravelling','10m',  10,'master_ravelling_child_10m','child_10m',None,'','',RAVELLING_RATING_THRESHOLD,RAVELLING_UNIT,'raw',''),
  ('Ravelling','100m',100,'master_ravelling_child_100m','child_100m','irc','area (%)','',RAVELLING_RATING_THRESHOLD,RAVELLING_UNIT,'mean','crack'),
  ('Ravelling','1000m',1000,'master_ravelling_parent','parent','ca','area (sq','',RAVELLING_RATING_THRESHOLD,RAVELLING_UNIT,'first','crack'),
]

# ── 4. Build validation rows ──
val_rows = []

for defect, level, bin_m, db_table, db_lvl, d2_src, d2_pat, d2_hint, thresh, unit, agg, excl_kw in CFG:
    db = _read_db(db_table, db_lvl)
    if db.empty: continue
    db['bin'] = (db['_sch'] // bin_m * bin_m).astype(int)

    if d2_src == 'ca':
        d2_ref = d2_ca_1000
        ref_tag = 'D2_CA'
    elif d2_src == 'irc':
        d2_ref = d2_irc_fallback
        ref_tag = 'D2_CA_100m' if _irc_is_ca_fallback else 'D2_IRC'
    else:
        d2_ref = pd.DataFrame()
        ref_tag = 'Source'

    for lane in ['L1', 'L2', 'R1', 'R2']:
        ldb = db[db['Lane'] == lane]
        if ldb.empty: continue

        # Aggregate DB to bin level
        if agg == 'crack_pct':
            db_agg = ldb.groupby('bin')['db_val'].apply(
                lambda x: (x * 35 / 100).sum() / (bin_m / 10 * 35) * 100
            ).reset_index(name='db_agg')
            db_agg['db_bf'] = (db_agg['db_agg'] >= thresh).astype(int)
        elif agg == 'mean':
            db_agg = ldb.groupby('bin').agg(db_agg=('db_val','mean'), db_bf=('db_bf','max')).reset_index()
        elif agg == 'first':
            db_agg = ldb.groupby('bin').agg(db_agg=('db_val','first'), db_bf=('db_bf','max')).reset_index()
        else:  # raw - 10m level, no D2 ref
            n_bins = len(ldb)
            db_mean = ldb['db_val'].mean()
            db_breach = int(ldb['db_bf'].sum())
            non_zero = int((ldb['db_val'] > 0).sum())
            val_rows.append({
                'upc_code': UPC, 'zone': ZONE, 'phase': PHASE, 'cycle': CYCLE,
                'defect': defect, 'lane': lane, 'level': level,
                'reference_source': ref_tag, 'bin_size_m': bin_m,
                'total_bins': n_bins, 'non_zero_bins': non_zero,
                'd2_mean': None, 'db_mean': round(db_mean, 6),
                'mean_abs_diff': None, 'max_abs_diff': None,
                'close_match_pct': None,
                'd2_breach_count': None, 'db_breach_count': db_breach,
                'breach_match_pct': None,
                'threshold_value': thresh, 'threshold_unit': unit,
                'status': 'OK' if non_zero > 0 else 'NO_DATA',
                'validation_ts': VAL_TIMESTAMP,
            })
            continue

        # Merge with D2 reference
        if d2_ref.empty:
            continue
        d2_col = _find_col(list(d2_ref.columns), lane, d2_pat, d2_hint, exclude=excl_kw)
        if not d2_col:
            val_rows.append({
                'upc_code': UPC, 'zone': ZONE, 'phase': PHASE, 'cycle': CYCLE,
                'defect': defect, 'lane': lane, 'level': level,
                'reference_source': ref_tag, 'bin_size_m': bin_m,
                'total_bins': len(db_agg), 'non_zero_bins': None,
                'd2_mean': None, 'db_mean': round(db_agg['db_agg'].mean(), 6),
                'mean_abs_diff': None, 'max_abs_diff': None,
                'close_match_pct': None,
                'd2_breach_count': None, 'db_breach_count': int(db_agg['db_bf'].sum()),
                'breach_match_pct': None,
                'threshold_value': thresh, 'threshold_unit': unit,
                'status': 'NO_D2_COL',
                'validation_ts': VAL_TIMESTAMP,
            })
            continue

        # Aggregate D2 to target bin: use mean when D2 has finer granularity
        _d2_vals = d2_ref[['bin', d2_col]].copy()
        _d2_vals[d2_col] = pd.to_numeric(_d2_vals[d2_col], errors='coerce').fillna(0)
        if d2_src == 'ca' and _ca_is_100m and bin_m == 1000:
            # D2 CA is at 100m but we compare at 1000m → must aggregate properly
            if defect == 'Cracking':
                # Cracking %: weighted by segment area → mean of 100m pcts
                ref = _d2_vals.groupby('bin')[d2_col].mean().reset_index()
            else:
                ref = _d2_vals.groupby('bin')[d2_col].mean().reset_index()
        else:
            ref = _d2_vals.groupby('bin')[d2_col].first().reset_index()
        ref.columns = ['bin', 'd2_val']

        cmp = ref.merge(db_agg, on='bin', how='inner')
        if cmp.empty: continue

        cmp['diff'] = (cmp['db_agg'] - cmp['d2_val']).abs()
        cmp['d2_bf'] = (cmp['d2_val'] >= thresh).astype(int)
        n = len(cmp)

        # Tolerance: "close" = within acceptable range for defect
        tol = {  'Cracking': 0.5, 'Roughness': 50, 'Rutting': 0.5, 'Ravelling': 0.5  }.get(defect, 0.5)
        close_n = int((cmp['diff'] <= tol).sum())
        close_pct = round(close_n / n * 100, 1)
        b_match_n = int((cmp['d2_bf'] == cmp['db_bf']).sum())
        b_match_pct = round(b_match_n / n * 100, 1)

        # Status logic
        if close_pct >= 95 and b_match_pct >= 95:
            status = 'MATCH'
        elif close_pct >= 80 and b_match_pct >= 90:
            status = 'CLOSE_MATCH'
        elif b_match_pct >= 90:
            status = 'BREACH_OK_VALUE_DIFF'
        else:
            status = 'MISMATCH'

        val_rows.append({
            'upc_code': UPC, 'zone': ZONE, 'phase': PHASE, 'cycle': CYCLE,
            'defect': defect, 'lane': lane, 'level': level,
            'reference_source': ref_tag, 'bin_size_m': bin_m,
            'total_bins': n, 'non_zero_bins': int((cmp['db_agg'] > 0).sum()),
            'd2_mean': round(float(cmp['d2_val'].mean()), 6),
            'db_mean': round(float(cmp['db_agg'].mean()), 6),
            'mean_abs_diff': round(float(cmp['diff'].mean()), 6),
            'max_abs_diff': round(float(cmp['diff'].max()), 6),
            'close_match_pct': close_pct,
            'd2_breach_count': int(cmp['d2_bf'].sum()),
            'db_breach_count': int(cmp['db_bf'].sum()),
            'breach_match_pct': b_match_pct,
            'threshold_value': thresh, 'threshold_unit': unit,
            'status': status,
            'validation_ts': VAL_TIMESTAMP,
        })

# ── 5. Build DataFrame and display ──
val_pdf = pd.DataFrame(val_rows)

# Clean summary print
print(f"{'='*120}")
print(f"  VALIDATION LOG: {UPC} | Zone={ZONE} Phase={PHASE} Cycle={CYCLE} | {VAL_TIMESTAMP:%Y-%m-%d %H:%M}")
print(f"{'='*120}")
print(f"{'Defect':<11s} {'Lane':>4s} {'Level':>5s} {'Ref':>7s}  {'Bins':>5s}  "
      f"{'D2_Mean':>9s}  {'DB_Mean':>9s}  {'MeanDiff':>9s}  {'MaxDiff':>8s}  "
      f"{'Close%':>7s}  {'D2_B':>5s} {'DB_B':>5s} {'B_Match%':>8s}  {'Status':>20s}")
print(f"{'-'*120}")
for _, r in val_pdf.iterrows():
    d2m = f"{r['d2_mean']:.4f}" if pd.notna(r['d2_mean']) else 'n/a'
    dbm = f"{r['db_mean']:.4f}" if pd.notna(r['db_mean']) else 'n/a'
    mad = f"{r['mean_abs_diff']:.4f}" if pd.notna(r['mean_abs_diff']) else 'n/a'
    mxd = f"{r['max_abs_diff']:.4f}" if pd.notna(r['max_abs_diff']) else 'n/a'
    cpct = f"{r['close_match_pct']:.1f}%" if pd.notna(r['close_match_pct']) else 'n/a'
    d2b = str(int(r['d2_breach_count'])) if pd.notna(r['d2_breach_count']) else 'n/a'
    dbb = str(int(r['db_breach_count'])) if pd.notna(r['db_breach_count']) else 'n/a'
    bmp = f"{r['breach_match_pct']:.1f}%" if pd.notna(r['breach_match_pct']) else 'n/a'
    # Status with emoji
    st = r['status']
    icon = {'MATCH': '\u2705', 'CLOSE_MATCH': '\U0001F7E1', 'BREACH_OK_VALUE_DIFF': '\U0001F7E0',
            'MISMATCH': '\u274C', 'OK': '\u2705', 'NO_DATA': '\u26A0\uFE0F', 'NO_D2_COL': '\u2753'}.get(st, '')
    print(f"{r['defect']:<11s} {r['lane']:>4s} {r['level']:>5s} {r['reference_source']:>7s}  "
          f"{r['total_bins']:>5}  {d2m:>9s}  {dbm:>9s}  {mad:>9s}  {mxd:>8s}  "
          f"{cpct:>7s}  {d2b:>5s} {dbb:>5s} {bmp:>8s}  {icon} {st}")

# ── Per-defect summary ──
print(f"\n{'='*120}")
print("PER-DEFECT SUMMARY:")
for defect in val_pdf['defect'].unique():
    sub = val_pdf[val_pdf['defect'] == defect]
    for lvl in ['10m', '100m', '1000m']:
        lsub = sub[sub['level'] == lvl]
        if lsub.empty: continue
        statuses = lsub['status'].unique()
        if lvl == '10m':
            total_b = lsub['db_breach_count'].sum()
            total_bins = lsub['total_bins'].sum()
            print(f"  {defect:<11s} @{lvl:>5s}: {int(total_bins):>6} bins, {int(total_b):>5} breaches")
        else:
            avg_close = lsub['close_match_pct'].mean()
            avg_breach = lsub['breach_match_pct'].mean()
            all_ok = all(s in ('MATCH', 'CLOSE_MATCH', 'OK') for s in statuses)
            print(f"  {defect:<11s} @{lvl:>5s}: close={avg_close:.1f}% breach_match={avg_breach:.1f}% "
                  f"{'ALL OK' if all_ok else ' | '.join(statuses)}")

# ── 6. Write to validation_log table ──
val_schema = StructType([
    StructField('upc_code', StringType()),
    StructField('zone', StringType()),
    StructField('phase', StringType()),
    StructField('cycle', StringType()),
    StructField('defect', StringType()),
    StructField('lane', StringType()),
    StructField('level', StringType()),
    StructField('reference_source', StringType()),
    StructField('bin_size_m', IntegerType()),
    StructField('total_bins', IntegerType()),
    StructField('non_zero_bins', IntegerType()),
    StructField('d2_mean', DoubleType()),
    StructField('db_mean', DoubleType()),
    StructField('mean_abs_diff', DoubleType()),
    StructField('max_abs_diff', DoubleType()),
    StructField('close_match_pct', DoubleType()),
    StructField('d2_breach_count', IntegerType()),
    StructField('db_breach_count', IntegerType()),
    StructField('breach_match_pct', DoubleType()),
    StructField('threshold_value', DoubleType()),
    StructField('threshold_unit', StringType()),
    StructField('status', StringType()),
    StructField('validation_ts', TimestampType()),
])

# Convert pandas NaN -> None for Spark (robust approach)
_int_cols = ['total_bins','non_zero_bins','d2_breach_count','db_breach_count','bin_size_m']
_dbl_cols = ['d2_mean','db_mean','mean_abs_diff','max_abs_diff','close_match_pct','breach_match_pct','threshold_value']
_records = []
for _, row in val_pdf.iterrows():
    rec = {}
    for k, v in row.items():
        if pd.isna(v) if isinstance(v, (float, int, np.floating, np.integer)) and not isinstance(v, bool) else (v is None):
            rec[k] = None
        elif k in _int_cols:
            rec[k] = int(v)
        elif k in _dbl_cols:
            rec[k] = float(v)
        else:
            rec[k] = v
    _records.append(rec)

val_sdf = spark.createDataFrame(_records, schema=val_schema)

VAL_TABLE = f"{DLM_KPI_NS}.validation_log"

val_sdf.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .option('replaceWhere', f"upc_code = '{UPC}'") \
    .saveAsTable(VAL_TABLE)

written = spark.table(VAL_TABLE).filter(f"upc_code = '{UPC}'").count()
print(f"\n\u2705 Written {written} rows to {VAL_TABLE} (replaceWhere upc_code='{UPC}')")

# ── Display final table ──
display(val_sdf.orderBy('defect', 'lane', 'level'))

  D2 CA interval detected from row data: median=100m → 100m
  D2 CA interval: 100m (cols have 100m=False, 1000m=False)
  ⚠️ No IRC file — using D2 CA (100m) as fallback for 100m comparison
  VALIDATION LOG: N_02015_03002_GJ | Zone=B Phase=1 Cycle=1 | 2026-10-09 17:20
Defect      Lane Level     Ref   Bins    D2_Mean    DB_Mean   MeanDiff   MaxDiff   Close%   D2_B  DB_B B_Match%                Status
------------------------------------------------------------------------------------------------------------------------
Cracking      L1   10m  Source   5154        n/a     4.5767        n/a       n/a      n/a    n/a   603      n/a  ✅ OK
Cracking      L2   10m  Source   5150        n/a     4.4214        n/a       n/a      n/a    n/a   620      n/a  ✅ OK
Cracking      R1   10m  Source   5146        n/a     4.5722        n/a       n/a      n/a    n/a   588      n/a  ✅ OK
Cracking      R2   10m  Source   5146        n/a     5.8403        n/a       n/a      n/a    n/a   765      n/a  ✅ OK
Crack

upc_code,zone,phase,cycle,defect,lane,level,reference_source,bin_size_m,total_bins,non_zero_bins,d2_mean,db_mean,mean_abs_diff,max_abs_diff,close_match_pct,d2_breach_count,db_breach_count,breach_match_pct,threshold_value,threshold_unit,status,validation_ts
N_02015_03002_GJ,B,1,1,Cracking,L1,1000m,D2_CA,1000,52,52,4.590264,4.532292,0.057972,3.014527,98.1,5,4,98.1,10.0,%,MATCH,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,L1,100m,D2_CA_100m,100,515,505,4.534284,4.576295,0.716477,12.043784,65.2,64,80,96.1,10.0,%,BREACH_OK_VALUE_DIFF,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,L1,10m,Source,10,5154,4225,null,4.576711,null,null,null,null,603,null,10.0,%,OK,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,L2,1000m,D2_CA,1000,52,52,4.384693,4.378846,0.005847,0.304025,100.0,3,3,100.0,10.0,%,MATCH,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,L2,100m,D2_CA_100m,100,515,504,4.37317,4.421359,0.731362,14.836846,61.4,64,71,97.5,10.0,%,BREACH_OK_VALUE_DIFF,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,L2,10m,Source,10,5150,4417,null,4.421359,null,null,null,null,620,null,10.0,%,OK,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,R1,1000m,D2_CA,1000,52,51,4.535616,4.524706,0.01091,0.567313,98.1,4,4,100.0,10.0,%,MATCH,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,R1,100m,D2_CA_100m,100,515,504,4.55338,4.568635,1.010679,10.399019,57.3,70,74,96.1,10.0,%,BREACH_OK_VALUE_DIFF,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,R1,10m,Source,10,5146,4254,null,4.572187,null,null,null,null,588,null,10.0,%,OK,2026-10-09T17:20:02.817542Z
N_02015_03002_GJ,B,1,1,Cracking,R2,1000m,D2_CA,1000,52,52,5.779607,5.779607,0.0,0.0,100.0,7,7,100.0,10.0,%,MATCH,2026-10-09T17:20:02.817542Z


In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# SILVER-LAYER VALIDATION: Backup vs DB (Silver) vs D2 (Ground Truth)
# All Defects × All Lanes × Parent/Child Levels
# Handles R-side reversed chainage (Start > End in backup)
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, os
from pyspark.sql import functions as F

print(f"{'='*130}")
print(f"  SILVER-LAYER VALIDATION: {UPC} (Zone={ZONE}, Phase={PHASE}, Cycle={CYCLE})")
print(f"  Backup (source parquet) vs DB (Silver tables) vs D2 (Ground Truth)")
print(f"{'='*130}")

# ── 1. Helper: canonical chainage = min(Start, End) for reversed R-side ──
def _canon_ch(df, bin_m):
    """Add canonical bin column handling reversed R-side chainage."""
    df = df.copy()
    df['Start_Chainage'] = pd.to_numeric(df['Start_Chainage'], errors='coerce')
    if 'End_Chainage' in df.columns:
        df['End_Chainage'] = pd.to_numeric(df['End_Chainage'], errors='coerce')
        df['_canon_ch'] = df[['Start_Chainage', 'End_Chainage']].min(axis=1)
    else:
        df['_canon_ch'] = df['Start_Chainage']
    df[f'bin_{bin_m}'] = (df['_canon_ch'] // bin_m * bin_m).astype(int)
    return df

# ── 2. Load D2 CA and IRC ──
_d2_files = sorted(os.listdir(D2_BASE)) if os.path.exists(D2_BASE) else []
_ca_f = next((f for f in _d2_files if f.endswith('.parquet') and 'ca' in f.lower()
              and 'irc' not in f.lower() and 'roughness,rut' not in f.lower()), None)
_irc_f = next((f for f in _d2_files if f.endswith('.parquet') and 'irc' in f.lower()), None)

d2_ca = pd.read_parquet(os.path.join(D2_BASE, _ca_f)) if _ca_f else pd.DataFrame()
d2_irc = pd.read_parquet(os.path.join(D2_BASE, _irc_f)) if _irc_f else pd.DataFrame()

def _prep_d2(df, bin_m):
    if df.empty: return df
    ch = next((c for c in df.columns if 'start' in c.lower() and 'chainage' in c.lower()), None)
    if not ch: return pd.DataFrame()
    df = df.copy()
    df[ch] = pd.to_numeric(df[ch], errors='coerce')
    df = df.dropna(subset=[ch])
    df[f'bin'] = (df[ch] // bin_m * bin_m).astype(int)
    return df

d2_ca_1k = _prep_d2(d2_ca.copy(), 1000)
d2_ca_100 = _prep_d2(d2_ca.copy(), 100)
d2_irc_100 = _prep_d2(d2_irc.copy(), 100) if not d2_irc.empty else pd.DataFrame()

# Detect D2 CA interval
_ca_is_100m = False
if not d2_ca.empty:
    _end_c = next((c for c in d2_ca.columns if 'end' in c.lower() and 'chainage' in c.lower()), None)
    _st_c = next((c for c in d2_ca.columns if 'start' in c.lower() and 'chainage' in c.lower()), None)
    if _end_c and _st_c:
        _med = (pd.to_numeric(d2_ca[_end_c], errors='coerce') - pd.to_numeric(d2_ca[_st_c], errors='coerce')).dropna().abs().median()
        _ca_is_100m = _med <= 200
print(f"  D2 CA interval: {'100m' if _ca_is_100m else '1000m'} | IRC available: {not d2_irc.empty}")

def _find_d2_col(cols, lane, *patterns, exclude=None):
    for c in cols:
        cl = c.lower()
        if lane.lower() not in cl: continue
        if exclude and exclude.lower() in cl: continue
        if all(p.lower() in cl for p in patterns): return c
    # Fallback: looser match
    for c in cols:
        cl = c.lower()
        if lane.lower() not in cl: continue
        if exclude and exclude.lower() in cl: continue
        if 'limit' in cl: continue
        if patterns and patterns[0].lower() in cl: return c
    return None

# ── 3. Discover backup files ──
_lanes = ['L1', 'L2', 'R1', 'R2']

def _load_backup(lane, defect_key):
    lane_path = os.path.join(BACKUP_BASE, f"Lane={lane}")
    if not os.path.exists(lane_path): return pd.DataFrame()
    matches = [f for f in os.listdir(lane_path) if f.endswith('.parquet') and defect_key.lower() in f.lower()]
    if not matches: return pd.DataFrame()
    return pd.read_parquet(os.path.join(lane_path, matches[0]))

# ── 4. DB reader ──
def _read_db_silver(table, lane, level_filter=None):
    tbl = f"{DLM_KPI_NS}.{table}"
    if not spark.catalog.tableExists(tbl): return pd.DataFrame()
    flds = [f.name for f in spark.table(tbl).schema.fields]
    sdf = spark.table(tbl).filter(f"upc_code = '{UPC}' AND lane = '{lane}'")
    if 'threshold_name' in flds:
        sdf = sdf.filter("threshold_name = 'As per CA'")
    if level_filter and 'level' in flds:
        sdf = sdf.filter(f"level = '{level_filter}'")
    db = sdf.select(
        F.col('start_chainage').cast('double').alias('s_ch'),
        F.col('end_chainage').cast('double').alias('e_ch'),
        F.col('metric_value').cast('double').alias('db_val'),
        F.col('breach_flag').cast('int').alias('db_bf')
    ).toPandas()
    if db.empty: return db
    # Canonical chainage
    db['_sch'] = db.apply(lambda r: min(r['s_ch'], r['e_ch'])
        if pd.notna(r['e_ch']) and r['e_ch'] < r['s_ch'] else r['s_ch'], axis=1)
    return db

# ── 5. Validation Config ──
# (defect, bk_file_key, bk_val_col, d2_patterns, d2_exclude, db_tables, thresholds)
VAL_CFG = [
    {
        'defect': 'Roughness',
        'bk_key': 'Roughness_IRI',
        'bk_col': 'RoughnessIndex',  # BI in backup
        'd2_ca_pats': ['roughness', 'bi'],
        'd2_irc_pats': ['roughness', 'bi'],
        'd2_excl': 'limit',
        'db_parent': ('master_roughness_parent', 'parent'),
        'db_child100': ('master_roughness_child_100m', 'child_100m'),
        'db_child10': ('master_roughness_child_10m', 'child_10m'),
        'threshold': CA_BI_THRESHOLD,
        'unit': 'mm/km',
        'close_tol': 50,  # tolerance for "close match"
    },
    {
        'defect': 'Rutting',
        'bk_key': 'Rutting',
        'bk_col': 'Rutting Average',
        'd2_ca_pats': ['rut', 'depth'],
        'd2_irc_pats': ['rut', 'depth'],
        'd2_excl': 'limit',
        'db_parent': ('master_rutting_parent', 'parent'),
        'db_child100': ('master_rutting_child_100m', 'child_100m'),
        'db_child10': ('master_rutting_child_10m', 'child_10m'),
        'threshold': CA_RUT_THRESHOLD,
        'unit': 'mm',
        'close_tol': 0.5,
    },
    {
        'defect': 'Cracking',
        'bk_key': 'Cracking',
        'bk_col': 'Percent_Area',  # already % in backup
        'd2_ca_pats': ['crack', 'area'],
        'd2_irc_pats': ['crack', 'area'],
        'd2_excl': 'limit',
        'db_parent': ('master_cracking', 'child_10m'),  # cracking uses child_10m always
        'db_child100': ('master_cracking', 'child_10m'),
        'db_child10': ('master_cracking', 'child_10m'),
        'threshold': CA_CRACK_PCT,
        'unit': '%',
        'close_tol': 0.5,
    },
    {
        'defect': 'Ravelling',
        'bk_key': 'Raveling',
        'bk_col': 'Raveling_Area_m2',  # sqm in backup
        'd2_ca_pats': ['area', 'sq'],
        'd2_irc_pats': ['area'],
        'd2_excl': 'limit',
        'db_parent': ('master_ravelling_parent', 'parent'),
        'db_child100': ('master_ravelling_child_100m', 'child_100m'),
        'db_child10': ('master_ravelling_child_10m', 'child_10m'),
        'threshold': RAVELLING_RATING_THRESHOLD,
        'unit': RAVELLING_UNIT,
        'close_tol': 0.5,
    },
]

# ── 6. Run Validation ──
results = []

for cfg in VAL_CFG:
    defect = cfg['defect']
    
    for lane in _lanes:
        # Load backup
        bk = _load_backup(lane, cfg['bk_key'])
        if bk.empty:
            print(f"  ⚠️ {defect} {lane}: No backup data")
            continue
        
        bk_col = cfg['bk_col']
        if bk_col not in bk.columns:
            # Try fuzzy match
            bk_col = next((c for c in bk.columns if cfg['bk_col'].lower().replace('_',' ') in c.lower().replace('_',' ')), None)
            if not bk_col:
                print(f"  ⚠️ {defect} {lane}: Column '{cfg['bk_col']}' not found. Cols: {list(bk.columns)}")
                continue
        
        bk[bk_col] = pd.to_numeric(bk[bk_col], errors='coerce').fillna(0)
        
        # ── Roughness fallback: compute BI when RoughnessIndex is empty ──
        if defect == 'Roughness' and (bk[bk_col] == 0).all():
            _lwp = next((c for c in bk.columns if 'lwpiri' in c.lower().replace('_','')), None)
            _rwp = next((c for c in bk.columns if 'rwpiri' in c.lower().replace('_','')), None)
            if _lwp and _rwp:
                bk[_lwp] = pd.to_numeric(bk[_lwp], errors='coerce').fillna(0)
                bk[_rwp] = pd.to_numeric(bk[_rwp], errors='coerce').fillna(0)
                bk[bk_col] = 630 * ((bk[_lwp] + bk[_rwp]) / 2) ** 1.12
                print(f"  ℹ️ {lane}: RoughnessIndex empty — computed BI from LwpIri+RwpIri (formula: 630×IRI^1.12)")
        
        # ── Ravelling: convert sqm → pct for 10m/100m child comparison ──
        if defect == 'Ravelling' and 'area' in bk_col.lower():
            bk['_rav_pct'] = bk[bk_col] / 35.0 * 100  # sqm → %
        
        tol = cfg['close_tol']
        
        # ── 10m Level: Backup vs DB ──
        bk10 = _canon_ch(bk, 10)
        _bk10_col = '_rav_pct' if (defect == 'Ravelling' and '_rav_pct' in bk10.columns) else bk_col
        bk10_grp = bk10.groupby('bin_10')[_bk10_col].first().reset_index(name='bk_val')
        
        db10 = _read_db_silver(cfg['db_child10'][0], lane, cfg['db_child10'][1])
        if not db10.empty:
            db10['bin_10'] = (db10['_sch'] // 10 * 10).astype(int)
            if defect == 'Cracking':
                # DB cracking is %
                db10_grp = db10.groupby('bin_10')['db_val'].first().reset_index()
            else:
                db10_grp = db10.groupby('bin_10')['db_val'].first().reset_index()
            m10 = bk10_grp.merge(db10_grp, on='bin_10', how='inner')
            n10 = len(m10)
            close10 = ((m10['bk_val'] - m10['db_val']).abs() < tol).sum()
            results.append((defect, lane, '10m', n10, close10, n10, close10/n10*100 if n10 else 0,
                           (m10['bk_val']-m10['db_val']).abs().mean(), '-', '-', '-'))
        
        # ── 100m Level: Backup vs DB vs D2-IRC ──
        bk100 = _canon_ch(bk, 100)
        _bk100_col = '_rav_pct' if (defect == 'Ravelling' and '_rav_pct' in bk100.columns) else bk_col
        bk100_grp = bk100.groupby('bin_100')[_bk100_col].mean().reset_index(name='bk_val')
        
        db100 = _read_db_silver(cfg['db_child100'][0], lane, cfg['db_child100'][1])
        if not db100.empty:
            db100['bin_100'] = (db100['_sch'] // 100 * 100).astype(int)
            if defect == 'Cracking':
                db100_grp = db100.groupby('bin_100')['db_val'].mean().reset_index()
            else:
                db100_grp = db100.groupby('bin_100')['db_val'].mean().reset_index()
            
            # D2 IRC at 100m
            d2_ref = d2_irc_100 if not d2_irc_100.empty else (d2_ca_100 if _ca_is_100m else pd.DataFrame())
            d2_col_100 = _find_d2_col(d2_ref.columns if not d2_ref.empty else [], lane,
                                       *cfg['d2_irc_pats'], exclude=cfg['d2_excl']) if not d2_ref.empty else None
            if d2_col_100:
                d2_ref[d2_col_100] = pd.to_numeric(d2_ref[d2_col_100], errors='coerce')
                d2_100_grp = d2_ref.groupby('bin')[d2_col_100].first().reset_index(name='d2_val')
                d2_100_grp.rename(columns={'bin': 'bin_100'}, inplace=True)
            else:
                d2_100_grp = pd.DataFrame(columns=['bin_100', 'd2_val'])
            
            m100 = bk100_grp.merge(db100_grp, on='bin_100', how='inner')
            if not d2_100_grp.empty:
                m100 = m100.merge(d2_100_grp, on='bin_100', how='left')
            n100 = len(m100)
            bk_db_100 = ((m100['bk_val'] - m100['db_val']).abs() < tol).sum()
            db_d2_100 = ((m100['db_val'] - m100['d2_val']).abs() < tol).sum() if 'd2_val' in m100 else '-'
            db_d2_pct = db_d2_100/n100*100 if isinstance(db_d2_100, int) and n100 else '-'
            results.append((defect, lane, '100m', n100, bk_db_100, n100, bk_db_100/n100*100 if n100 else 0,
                           (m100['bk_val']-m100['db_val']).abs().mean(),
                           db_d2_100, db_d2_pct,
                           (m100['db_val']-m100['d2_val']).abs().mean() if 'd2_val' in m100 else '-'))
        
        # ── 1000m Level: Backup vs DB vs D2-CA ──
        bk1k = _canon_ch(bk, 1000)
        bk1k_grp = bk1k.groupby('bin_1000')[bk_col].mean().reset_index(name='bk_val')
        
        db1k = _read_db_silver(cfg['db_parent'][0], lane, cfg['db_parent'][1])
        if not db1k.empty:
            db1k['bin_1000'] = (db1k['_sch'] // 1000 * 1000).astype(int)
            if defect == 'Cracking':
                db1k_grp = db1k.groupby('bin_1000')['db_val'].mean().reset_index()
            else:
                db1k_grp = db1k.groupby('bin_1000')['db_val'].first().reset_index()
            
            # D2 CA at 1000m
            d2_col_ca = _find_d2_col(d2_ca_1k.columns if not d2_ca_1k.empty else [], lane,
                                      *cfg['d2_ca_pats'], exclude=cfg['d2_excl']) if not d2_ca_1k.empty else None
            if d2_col_ca:
                d2_ca_1k[d2_col_ca] = pd.to_numeric(d2_ca_1k[d2_col_ca], errors='coerce')
                if _ca_is_100m:
                    d2_1k_grp = d2_ca_1k.groupby('bin')[d2_col_ca].mean().reset_index(name='d2_val')
                else:
                    d2_1k_grp = d2_ca_1k.groupby('bin')[d2_col_ca].first().reset_index(name='d2_val')
                d2_1k_grp.rename(columns={'bin': 'bin_1000'}, inplace=True)
            else:
                d2_1k_grp = pd.DataFrame(columns=['bin_1000', 'd2_val'])
            
            m1k = bk1k_grp.merge(db1k_grp, on='bin_1000', how='inner')
            if not d2_1k_grp.empty:
                m1k = m1k.merge(d2_1k_grp, on='bin_1000', how='left')
            n1k = len(m1k)
            bk_db_1k = ((m1k['bk_val'] - m1k['db_val']).abs() < tol).sum()
            db_d2_1k = ((m1k['db_val'] - m1k['d2_val']).abs() < tol).sum() if 'd2_val' in m1k else '-'
            db_d2_pct_1k = db_d2_1k/n1k*100 if isinstance(db_d2_1k, int) and n1k else '-'
            results.append((defect, lane, '1000m', n1k, bk_db_1k, n1k, bk_db_1k/n1k*100 if n1k else 0,
                           (m1k['bk_val']-m1k['db_val']).abs().mean(),
                           db_d2_1k, db_d2_pct_1k,
                           (m1k['db_val']-m1k['d2_val']).abs().mean() if 'd2_val' in m1k else '-'))

# ── 7. Print Summary ──
print(f"\n{'Defect':<11s} {'Lane':>4s} {'Level':>6s} {'Bins':>5s}  {'BK≈DB':>7s} {'Close%':>7s} {'AvgDiff':>10s}  {'DB≈D2':>7s} {'Close%':>7s} {'AvgDiff':>10s}  {'STATUS':>12s}")
print("─" * 130)
current_defect = ""
for r in results:
    defect, lane, level, n, bk_db, _, bk_db_pct, bk_db_diff, db_d2, db_d2_pct, db_d2_diff = r
    if defect != current_defect:
        if current_defect: print()
        current_defect = defect
    status = '✅ MATCH' if bk_db_pct >= 95 else ('🟡 CLOSE' if bk_db_pct >= 80 else ('🟠 D2-CALIBRATED' if defect == 'Cracking' and str(db_d2_pct) != '-' and float(db_d2_pct) >= 95 else ('🟠 CHECK' if bk_db_pct >= 40 else '❌ MISMATCH')))
    print(f"{defect:<11s} {lane:>4s} {level:>6s} {n:>5d}  "
          f"{str(bk_db):>7s} {bk_db_pct:>6.0f}% {str(round(bk_db_diff,4)):>10s}  "
          f"{str(db_d2):>7s} {str(round(float(db_d2_pct),0)) if db_d2_pct != '-' else '-':>7s} "
          f"{str(round(float(db_d2_diff),4)) if db_d2_diff != '-' else '-':>10s}  {status}")

# ── 8. Summary per defect ──
print(f"\n{'='*130}")
print("PER-DEFECT SUMMARY:")
for d in ['Roughness', 'Rutting', 'Cracking', 'Ravelling']:
    d_rows = [r for r in results if r[0] == d]
    for level in ['10m', '100m', '1000m']:
        lv_rows = [r for r in d_rows if r[2] == level]
        if not lv_rows: continue
        avg_pct = np.mean([r[6] for r in lv_rows])
        status_set = set()
        for r in lv_rows:
            if r[6] >= 95: status_set.add('MATCH')
            elif r[6] >= 80: status_set.add('CLOSE')
            else: status_set.add('CHECK')
        print(f"  {d:<12s} @{level:>5s}: avg_close={avg_pct:.1f}%  [{' | '.join(sorted(status_set))}]")
print(f"\n✅ Silver-layer validation complete for {UPC}")

  SILVER-LAYER VALIDATION: N_07024_01001_GJ (Zone=B, Phase=2, Cycle=1)
  Backup (source parquet) vs DB (Silver tables) vs D2 (Ground Truth)
  D2 CA interval: 100m | IRC available: True

Defect      Lane  Level  Bins    BK≈DB  Close%    AvgDiff    DB≈D2  Close%    AvgDiff        STATUS
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Roughness     L1    10m  7624     7624    100%        0.0        -       -          -  ✅ MATCH
Roughness     L1   100m   763      763    100%        0.0      757       -    10.0604  ✅ MATCH
Roughness     L1  1000m    77       77    100%        0.0       76       -    14.4474  ✅ MATCH
Roughness     L2    10m  7624     7595    100%        nan        -       -          -  ✅ MATCH
Roughness     L2   100m   763      759     99%        nan      756       -    10.6581  ✅ MATCH
Roughness     L2  1000m    77       76     99%     9.6224       76       -    26.2117  ✅ MATCH
Roughness    

In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# RECONCILIATION EXPORT: Backup vs DB vs D2 — All Defects × Both UPCs
# Saves a multi-sheet Excel to UC Volume
# ═══════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np, os
from pyspark.sql import functions as F
from datetime import datetime

OUTPUT_DIR = "/Volumes/nhai_dhms/silver_layer/validation_reports"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
OUTPUT_PATH = f"{OUTPUT_DIR}/DLM_Reconciliation_{TIMESTAMP}.xlsx"

BRONZE_ROOT = "/Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output"
DLM_KPI = "nhai_dhms.dlm_kpi"

UPC_LIST = [
    {"upc": "N_07024_01001_GJ", "zone": "B", "phase": "2", "cycle": "1",
     "thresholds": {"roughness_bi": 2750.0, "cracking": 5.0, "rutting": 10.0, "ravelling": 10.0}},
    {"upc": "N_02015_03002_GJ", "zone": "B", "phase": "1", "cycle": "1",
     "thresholds": {"roughness_bi": 3000.0, "cracking": 10.0, "rutting": 10.0, "ravelling": 3.0}},
]

def _find_paths(upc_cfg):
    upc, zone, phase, cycle = upc_cfg['upc'], upc_cfg['zone'], upc_cfg['phase'], upc_cfg['cycle']
    nsv_root = f"{BRONZE_ROOT}/Source=NSV/Zone={zone}/Phase={phase}/Cycle={cycle}/UPC={upc}"
    bk_root = f"{BRONZE_ROOT}/Source=NSV_Backup/Zone={zone}/Phase={phase}/Cycle={cycle}/UPC={upc}"
    def _latest(base):
        if not os.path.exists(base): return base
        for y in sorted(os.listdir(base), reverse=True):
            if not y.startswith("year="): continue
            for m in sorted(os.listdir(os.path.join(base,y)), reverse=True):
                if not m.startswith("month="): continue
                for d in sorted(os.listdir(os.path.join(base,y,m)), reverse=True):
                    if d.startswith("day="): return f"{base}/{y}/{m}/{d}"
        return base
    def _find_del(base):
        if not os.path.exists(base): return base
        ch = sorted(os.listdir(base))
        if any(c.startswith("Deliverable=") for c in ch): return base
        for c in ch:
            cp = os.path.join(base, c)
            if os.path.isdir(cp) and any(gc.startswith("Deliverable=") for gc in os.listdir(cp)): return cp
        return base
    def _find_lane(base):
        if not os.path.exists(base): return None
        ch = sorted(os.listdir(base))
        if any(c.startswith("Lane=") for c in ch): return base
        for c in ch:
            cp = os.path.join(base, c)
            if os.path.isdir(cp):
                r = _find_lane(cp)
                if r: return r
        return None
    nsv_date = _latest(nsv_root)
    d2 = f"{_find_del(nsv_date)}/Deliverable=Deliverable - 2"
    bk = _find_lane(_latest(bk_root)) or _latest(bk_root)
    return d2, bk

def _canon_ch_df(df, bin_m):
    df = df.copy()
    df['Start_Chainage'] = pd.to_numeric(df['Start_Chainage'], errors='coerce')
    if 'End_Chainage' in df.columns:
        df['End_Chainage'] = pd.to_numeric(df['End_Chainage'], errors='coerce')
        df['_ch'] = df[['Start_Chainage','End_Chainage']].min(axis=1)
    else:
        df['_ch'] = df['Start_Chainage']
    df['bin'] = (df['_ch'] // bin_m * bin_m).astype(int)
    return df

def _find_col(cols, lane, *pats, excl=None):
    for c in cols:
        cl = c.lower()
        if lane.lower() not in cl: continue
        if excl and excl.lower() in cl: continue
        if all(p.lower() in cl for p in pats): return c
    return None

# ── Build reconciliation data ──
all_summary = []
all_detail = {d: [] for d in ['Roughness','Rutting','Cracking','Ravelling']}
all_row_counts = []
all_breach = []

for upc_cfg in UPC_LIST:
    upc = upc_cfg['upc']
    thresh = upc_cfg['thresholds']
    d2_base, bk_base = _find_paths(upc_cfg)
    print(f"\n{'='*80}")
    print(f"  Processing: {upc}")
    print(f"  D2: {d2_base}")
    print(f"  BK: {bk_base}")
    
    # Load D2
    d2f = sorted(os.listdir(d2_base)) if os.path.exists(d2_base) else []
    ca_f = next((f for f in d2f if f.endswith('.parquet') and 'ca' in f.lower() and 'irc' not in f.lower() and 'roughness,rut' not in f.lower()), None)
    d2 = pd.read_parquet(os.path.join(d2_base, ca_f)) if ca_f else pd.DataFrame()
    ch_col = next((c for c in d2.columns if 'start' in c.lower() and 'chainage' in c.lower()), None) if not d2.empty else None
    if ch_col:
        d2[ch_col] = pd.to_numeric(d2[ch_col], errors='coerce')
        d2['bin_1000'] = (d2[ch_col] // 1000 * 1000).astype(int)
    end_c = next((c for c in d2.columns if 'end' in c.lower() and 'chainage' in c.lower()), None) if not d2.empty else None
    is_100m = False
    if end_c and ch_col:
        med = (pd.to_numeric(d2[end_c], errors='coerce') - d2[ch_col]).dropna().abs().median()
        is_100m = med <= 200
    
    # Row counts per table
    for tbl in ['master_roughness_parent','master_roughness_child_100m','master_roughness_child_10m',
                'master_rutting_parent','master_rutting_child_100m','master_rutting_child_10m',
                'master_cracking','master_ravelling_parent','master_ravelling_child_100m',
                'master_ravelling_child_10m','master_pothole','master_chainage']:
        fq = f"{DLM_KPI}.{tbl}"
        try:
            cnt = spark.table(fq).filter(f"upc_code='{upc}'").count()
        except: cnt = 0
        all_row_counts.append({'upc': upc, 'table': tbl, 'db_rows': cnt})
    
    # Defect configs
    DEFECTS = [
        ('Roughness', ['roughness','bi'], 'master_roughness_parent', thresh['roughness_bi'], 'mm/km', 50),
        ('Rutting',    ['rut','depth'],   'master_rutting_parent',    thresh['rutting'],       'mm',    0.5),
        ('Cracking',   ['crack','area'],  'master_cracking',          thresh['cracking'],      '%',     0.5),
        ('Ravelling',  ['area','sq'],     'master_ravelling_parent',  thresh['ravelling'],     'sqm',   0.5),
    ]
    
    for defect, d2_pats, db_tbl, thr, unit, tol in DEFECTS:
        for lane in ['L1','L2','R1','R2']:
            # D2 column
            d2_col = _find_col(d2.columns if not d2.empty else [], lane, *d2_pats, excl='limit')
            if d2_col:
                d2[d2_col] = pd.to_numeric(d2[d2_col], errors='coerce')
                if is_100m:
                    d2_1k = d2.groupby('bin_1000')[d2_col].mean().reset_index(name='d2_val')
                else:
                    d2_1k = d2.groupby('bin_1000')[d2_col].first().reset_index(name='d2_val')
                d2_1k.rename(columns={'bin_1000':'bin'}, inplace=True)
            else:
                d2_1k = pd.DataFrame(columns=['bin','d2_val'])
            
            # DB
            fq = f"{DLM_KPI}.{db_tbl}"
            lvl = 'child_10m' if defect == 'Cracking' else 'parent'
            sdf = spark.table(fq).filter(f"upc_code='{upc}' AND lane='{lane}' AND threshold_name='As per CA'")
            if defect == 'Cracking':
                sdf = sdf.filter("level='child_10m'")
            db = sdf.select(F.col('start_chainage').cast('double').alias('sch'),
                           F.col('end_chainage').cast('double').alias('ech'),
                           F.col('metric_value').cast('double').alias('db_val'),
                           F.col('breach_flag').cast('int').alias('db_bf')).toPandas()
            if not db.empty:
                db['_sch'] = db.apply(lambda r: min(r['sch'],r['ech']) if pd.notna(r['ech']) and r['ech']<r['sch'] else r['sch'], axis=1)
                db['bin'] = (db['_sch'] // 1000 * 1000).astype(int)
                if defect == 'Cracking':
                    db_1k = db.groupby('bin').agg(db_val=('db_val','mean'), db_bf=('db_bf','max')).reset_index()
                else:
                    db_1k = db.groupby('bin').agg(db_val=('db_val','first'), db_bf=('db_bf','max')).reset_index()
            else:
                db_1k = pd.DataFrame(columns=['bin','db_val','db_bf'])
            
            # Backup
            lp = os.path.join(bk_base, f"Lane={lane}")
            bk_map = {'Roughness':'Roughness_IRI','Rutting':'Rutting','Cracking':'Cracking','Ravelling':'Raveling'}
            bk_col_map = {'Roughness':'RoughnessIndex','Rutting':'Rutting Average','Cracking':'Percent_Area','Ravelling':'Raveling_Area_m2'}
            bk = pd.DataFrame()
            if os.path.exists(lp):
                bk_files = [f for f in os.listdir(lp) if f.endswith('.parquet') and bk_map[defect].lower() in f.lower()]
                if bk_files:
                    bk = pd.read_parquet(os.path.join(lp, bk_files[0]))
            if not bk.empty:
                bk_c = bk_col_map[defect]
                if bk_c not in bk.columns:
                    bk_c = next((c for c in bk.columns if bk_col_map[defect].lower().replace('_',' ') in c.lower().replace('_',' ')), None)
                if bk_c:
                    bk[bk_c] = pd.to_numeric(bk[bk_c], errors='coerce').fillna(0)
                    if defect == 'Roughness' and (bk[bk_c] == 0).all():
                        lwp = next((c for c in bk.columns if 'lwpiri' in c.lower().replace('_','')), None)
                        rwp = next((c for c in bk.columns if 'rwpiri' in c.lower().replace('_','')), None)
                        if lwp and rwp:
                            bk[lwp] = pd.to_numeric(bk[lwp], errors='coerce').fillna(0)
                            bk[rwp] = pd.to_numeric(bk[rwp], errors='coerce').fillna(0)
                            bk[bk_c] = 630 * ((bk[lwp] + bk[rwp]) / 2) ** 1.12
                    if defect == 'Ravelling' and 'area' in bk_c.lower():
                        bk['_val'] = bk[bk_c] / 35.0 * 100
                    else:
                        bk['_val'] = bk[bk_c]
                    bk = _canon_ch_df(bk, 1000)
                    bk_1k = bk.groupby('bin')['_val'].mean().reset_index(name='bk_val')
                else:
                    bk_1k = pd.DataFrame(columns=['bin','bk_val'])
            else:
                bk_1k = pd.DataFrame(columns=['bin','bk_val'])
            
            # Merge
            m = db_1k.merge(d2_1k, on='bin', how='outer').merge(bk_1k, on='bin', how='outer').sort_values('bin')
            m['upc'] = upc
            m['defect'] = defect
            m['lane'] = lane
            m['unit'] = unit
            m['threshold'] = thr
            m['d2_breach'] = (m['d2_val'].fillna(0) >= thr).astype(int)
            m['bk_db_diff'] = (m['bk_val'] - m['db_val']).abs()
            m['db_d2_diff'] = (m['db_val'] - m['d2_val']).abs()
            m['bk_db_close'] = m['bk_db_diff'] < tol
            m['db_d2_close'] = m['db_d2_diff'] < tol
            
            all_detail[defect].append(m[['upc','defect','lane','bin','bk_val','db_val','d2_val',
                                         'db_bf','d2_breach','bk_db_diff','db_d2_diff','bk_db_close','db_d2_close',
                                         'unit','threshold']])
            
            # Summary row
            n = len(m.dropna(subset=['db_val']))
            bk_db_pct = m['bk_db_close'].sum() / n * 100 if n else 0
            db_d2_pct = m['db_d2_close'].sum() / n * 100 if n else 0
            bf_match = ((m['db_bf'] == m['d2_breach']).sum() / n * 100) if n else 0
            all_summary.append({
                'upc': upc, 'defect': defect, 'lane': lane, 'level': '1000m',
                'total_bins': n,
                'bk_db_close_pct': round(bk_db_pct, 1),
                'db_d2_close_pct': round(db_d2_pct, 1),
                'breach_match_pct': round(bf_match, 1),
                'bk_db_avg_diff': round(m['bk_db_diff'].mean(), 4) if n else None,
                'db_d2_avg_diff': round(m['db_d2_diff'].mean(), 4) if n else None,
                'threshold': thr, 'unit': unit,
                'status': '✅ PASS' if bk_db_pct >= 80 and bf_match >= 90 else ('🟡 CHECK' if bf_match >= 80 else '❌ FAIL'),
            })
            
            # Breach summary
            if n:
                d2_b = int(m['d2_breach'].sum())
                db_b = int(m['db_bf'].sum())
                all_breach.append({'upc':upc,'defect':defect,'lane':lane,'d2_breach_count':d2_b,
                                   'db_breach_count':db_b,'total_bins':n,'threshold':thr,'unit':unit,
                                   'match': '✅' if d2_b == db_b else f'Δ{abs(d2_b-db_b)}'})
    
    print(f"  ✅ {upc} complete")

# ── Write Excel ──
print(f"\n{'='*80}")
print(f"  Writing reconciliation to: {OUTPUT_PATH}")

summary_df = pd.DataFrame(all_summary)
row_df = pd.DataFrame(all_row_counts)
breach_df = pd.DataFrame(all_breach)

import shutil, tempfile
_tmp = tempfile.mktemp(suffix='.xlsx')
with pd.ExcelWriter(_tmp, engine='openpyxl') as writer:
    # Sheet 1: Summary
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Sheet 2: Row Counts
    row_df.to_excel(writer, sheet_name='Row_Counts', index=False)
    
    # Sheet 3: Breach Comparison
    breach_df.to_excel(writer, sheet_name='Breach_Comparison', index=False)
    
    # Sheets 4-7: Detail per defect
    for defect, dfs in all_detail.items():
        if dfs:
            detail_df = pd.concat(dfs, ignore_index=True)
            # Truncate sheet name to 31 chars (Excel limit)
            detail_df.to_excel(writer, sheet_name=f'{defect}_Detail'[:31], index=False)

shutil.copy2(_tmp, OUTPUT_PATH)
os.remove(_tmp)
print(f"  ✅ Saved: {OUTPUT_PATH}")
print(f"  Sheets: Summary, Row_Counts, Breach_Comparison, + 4 defect detail sheets")

# Display summary
print(f"\n{'='*80}")
print("RECONCILIATION SUMMARY")
print(f"{'='*80}")
display(summary_df)

print(f"\nBREACH COMPARISON:")
display(breach_df)


  Processing: N_07024_01001_GJ
  D2: /Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output/Source=NSV/Zone=B/Phase=2/Cycle=1/UPC=N_07024_01001_GJ/year=2026/month=09/day=12/Deliverable=Deliverable - 2
  BK: /Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output/Source=NSV_Backup/Zone=B/Phase=2/Cycle=1/UPC=N_07024_01001_GJ/year=2026/month=09/day=12/Pavement=Unknown/Variant=Default
  ✅ N_07024_01001_GJ complete

  Processing: N_02015_03002_GJ
  D2: /Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output/Source=NSV/Zone=B/Phase=1/Cycle=1/UPC=N_02015_03002_GJ/year=2026/month=09/day=04/Deliverable=Deliverable - 2
  BK: /Volumes/nhai_dhms/bronze_layer/extraction_data/final_bronze_output/Source=NSV_Backup/Zone=B/Phase=1/Cycle=1/UPC=N_02015_03002_GJ/year=2026/month=09/day=04/Project=100m Interval/Pavement=Unknown/Variant=Default
  ✅ N_02015_03002_GJ complete

  Writing reconciliation to: /Volumes/nhai_dhms/silver_layer/validation_reports/DLM_Reconciliatio

upc,defect,lane,level,total_bins,bk_db_close_pct,db_d2_close_pct,breach_match_pct,bk_db_avg_diff,db_d2_avg_diff,threshold,unit,status
N_07024_01001_GJ,Roughness,L1,1000m,77,100.0,98.7,100.0,0.0,14.4474,2750.0,mm/km,✅ PASS
N_07024_01001_GJ,Roughness,L2,1000m,77,98.7,98.7,98.7,9.6224,26.2117,2750.0,mm/km,✅ PASS
N_07024_01001_GJ,Roughness,R1,1000m,77,100.0,100.0,100.0,0.0,13.9649,2750.0,mm/km,✅ PASS
N_07024_01001_GJ,Roughness,R2,1000m,77,100.0,100.0,98.7,0.0,14.1119,2750.0,mm/km,✅ PASS
N_07024_01001_GJ,Rutting,L1,1000m,77,100.0,100.0,100.0,0.0,0.003,10.0,mm,✅ PASS
N_07024_01001_GJ,Rutting,L2,1000m,77,98.7,100.0,100.0,0.0219,0.0032,10.0,mm,✅ PASS
N_07024_01001_GJ,Rutting,R1,1000m,77,100.0,100.0,100.0,0.0,0.0026,10.0,mm,✅ PASS
N_07024_01001_GJ,Rutting,R2,1000m,77,100.0,100.0,100.0,0.0,0.0026,10.0,mm,✅ PASS
N_07024_01001_GJ,Cracking,L1,1000m,77,76.6,100.0,14.3,0.3869,0.0019,5.0,%,❌ FAIL
N_07024_01001_GJ,Cracking,L2,1000m,77,61.0,100.0,14.3,0.5772,5.0E-4,5.0,%,❌ FAIL



BREACH COMPARISON:


upc,defect,lane,d2_breach_count,db_breach_count,total_bins,threshold,unit,match
N_07024_01001_GJ,Roughness,L1,12,12,77,2750.0,mm/km,✅
N_07024_01001_GJ,Roughness,L2,16,15,77,2750.0,mm/km,Δ1
N_07024_01001_GJ,Roughness,R1,13,13,77,2750.0,mm/km,✅
N_07024_01001_GJ,Roughness,R2,13,14,77,2750.0,mm/km,Δ1
N_07024_01001_GJ,Rutting,L1,0,0,77,10.0,mm,✅
N_07024_01001_GJ,Rutting,L2,0,0,77,10.0,mm,✅
N_07024_01001_GJ,Rutting,R1,0,0,77,10.0,mm,✅
N_07024_01001_GJ,Rutting,R2,0,0,77,10.0,mm,✅
N_07024_01001_GJ,Cracking,L1,2,68,77,5.0,%,Δ66
N_07024_01001_GJ,Cracking,L2,5,71,77,5.0,%,Δ66
